In [2]:
pip install imageio

   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/7.0 MB 5.6 MB/s eta 0:00:02
   ---------- ----------------------------- 1.8/7.0 MB 5.9 MB/s eta 0:00:01
   ----------------------- ---------------- 4.2/7.0 MB 7.9 MB/s eta 0:00:01
   -------------------------------------- - 6.8/7.0 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 9.0 MB/s  0:00:00

   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   -------------------- ----------------

In [7]:
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os, random

HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE), f"Missing {TEXTURE}"

# --- notebook-safe reset (ONLY ONE GUI CONNECTION) ---
try:
    if p.isConnected():
        p.disconnect()
except:
    pass

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)

# IMPORTANT: avoid wireframe confusion
p.configureDebugVisualizer(p.COV_ENABLE_WIREFRAME, 0)

# --- Load heightmap ---
hmap = imageio.imread(HEIGHTMAP)

# Ensure single channel
if hmap.ndim == 3:
    hmap = hmap[..., 0]

hmap = hmap.astype(np.float32)
H, W = hmap.shape

# (Optional) downsample if huge
MAX_HW = 512
if max(H, W) > MAX_HW:
    from PIL import Image
    hmap_img = Image.fromarray(hmap.astype(np.uint16) if hmap.max() > 255 else hmap.astype(np.uint8))
    hmap_img = hmap_img.resize((MAX_HW, MAX_HW), resample=Image.BILINEAR)
    hmap = np.array(hmap_img).astype(np.float32)
    H, W = hmap.shape

# Normalize to [0, 1]
hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap is flat (all same value).")
hmap01 = (hmap - hmin) / (hmax - hmin)

heightfieldData = hmap01.flatten().tolist()

# --- Read texture and pick terrain size to improve visibility ---
tex = imageio.imread(TEXTURE)
tex_w = tex.shape[1]

# TARGET: meters per pixel (smaller = more visible trash)
# If your trash is subtle in texture, aim for ~0.05 to 0.15 m/px
TARGET_M_PER_PX = 0.10
terrain_size_m = float(tex_w) * TARGET_M_PER_PX

# Cap to avoid insane sizes
terrain_size_m = max(80.0, min(terrain_size_m, 300.0))

desired_max_height = 12.0

sx = terrain_size_m / W
sy = terrain_size_m / H

# --- Create terrain ---
terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldTextureScaling=1.0,
    heightfieldData=heightfieldData,
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

terrain_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_shape)

# Texture
tex_id = p.loadTexture(TEXTURE)
p.changeVisualShape(terrain_id, -1, textureUniqueId=tex_id)

p.changeDynamics(terrain_id, -1, lateralFriction=0.9, restitution=0.0)

# Terrain is usually centered around origin in PyBullet heightfields
aabb_min, aabb_max = p.getAABB(terrain_id)
center = [(aabb_min[i] + aabb_max[i]) / 2 for i in range(3)]

print("Loaded terrain.")
print("Terrain AABB:", aabb_min, aabb_max)
print("Terrain center:", center)

print("Texture shape:", tex.shape)
print("Heightmap shape:", H, W)
print("Meters per texture pixel (approx):", terrain_size_m / tex_w, "m/px")

# --- Spawn a cube near center ---
cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5], rgbaColor=[1, 0, 0, 1])
cube_id  = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[center[0], center[1], center[2] + 20]
)

# --- Camera (birdseye defaults + optional toggle) ---
TOPDOWN_DIST = max(10.0, terrain_size_m * 0.18)  # tweak 0.45–0.75
birdseye = True

# Start camera on center (birdseye)
p.resetDebugVisualizerCamera(
    cameraDistance=TOPDOWN_DIST,
    cameraYaw=0,
    cameraPitch=-89,
    cameraTargetPosition=[center[0], center[1], 0]
)

print("Controls:")
print("Move cube: Arrow keys | Up/Down: R/F")
print("Spawn/remove trash objects: press T")
print("Toggle birdseye: press V")
print("Tip: If view looks like grid, press W once (wireframe toggle).")

# --- Trash spawning (RELIABLE FOR YOLO) ---
trash_ids = []
trash_on = False

def pressed(keys, k):
    return (k in keys) and (keys[k] & p.KEY_WAS_TRIGGERED)

def spawn_trash(n=60):
    """Spawn visible trash proxies sitting on terrain (good for YOLO)."""
    ids = []
    xmin, ymin = aabb_min[0] + 2.0, aabb_min[1] + 2.0
    xmax, ymax = aabb_max[0] - 2.0, aabb_max[1] - 2.0

    for _ in range(n):
        x = random.uniform(xmin, xmax)
        y = random.uniform(ymin, ymax)

        hit = p.rayTest([x, y, 200], [x, y, -200])[0]
        hit_obj, hit_pos = hit[0], hit[3]
        if hit_obj != terrain_id:
            continue

        z = hit_pos[2]

        # bottle (cyl) or bag (thin box)
        if random.random() < 0.6:
            r = random.uniform(0.04, 0.08)
            h = random.uniform(0.15, 0.28)
            col = p.createCollisionShape(p.GEOM_CYLINDER, radius=r, height=h)
            vis = p.createVisualShape(p.GEOM_CYLINDER, radius=r, length=h, rgbaColor=[0.1, 0.9, 0.1, 1])
            yaw = random.uniform(0, 3.14159)
            tid = p.createMultiBody(
                baseMass=0.05,
                baseCollisionShapeIndex=col,
                baseVisualShapeIndex=vis,
                basePosition=[x, y, z + h/2 + 0.01],
                baseOrientation=p.getQuaternionFromEuler([0, 0, yaw])
            )
        else:
            hx = random.uniform(0.08, 0.16)
            hy = random.uniform(0.08, 0.16)
            hz = random.uniform(0.01, 0.02)
            col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[hx, hy, hz])
            vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[hx, hy, hz], rgbaColor=[0.9, 0.1, 0.1, 1])
            yaw = random.uniform(0, 3.14159)
            tid = p.createMultiBody(
                baseMass=0.03,
                baseCollisionShapeIndex=col,
                baseVisualShapeIndex=vis,
                basePosition=[x, y, z + hz + 0.01],
                baseOrientation=p.getQuaternionFromEuler([0, 0, yaw])
            )
        ids.append(tid)

    return ids

def clear_trash():
    global trash_ids
    for tid in trash_ids:
        try:
            p.removeBody(tid)
        except:
            pass
    trash_ids = []

# --- Controls ---
speed = 0.8

def down(keys, k):
    return (k in keys) and (keys[k] & p.KEY_IS_DOWN)

while p.isConnected():
    keys = p.getKeyboardEvents()
    pos, orn = p.getBasePositionAndOrientation(cube_id)

    # Toggle birdseye
    if pressed(keys, ord('v')):
        birdseye = not birdseye
        print("Birdseye:", birdseye)

    # Toggle trash
    if pressed(keys, ord('t')):
        trash_on = not trash_on
        if trash_on:
            clear_trash()
            trash_ids = spawn_trash(n=80)
            print("Trash ON:", len(trash_ids))
        else:
            clear_trash()
            print("Trash OFF")

    # Move cube
    if down(keys, p.B3G_UP_ARROW):    pos = [pos[0] + speed, pos[1], pos[2]]
    if down(keys, p.B3G_DOWN_ARROW):  pos = [pos[0] - speed, pos[1], pos[2]]
    if down(keys, p.B3G_LEFT_ARROW):  pos = [pos[0], pos[1] + speed, pos[2]]
    if down(keys, p.B3G_RIGHT_ARROW): pos = [pos[0], pos[1] - speed, pos[2]]

    # Altitude
    if down(keys, ord('r')): pos = [pos[0], pos[1], pos[2] + speed]
    if down(keys, ord('f')): pos = [pos[0], pos[1], pos[2] - speed]

    p.resetBasePositionAndOrientation(cube_id, pos, orn)

    # --- Camera follow (birdseye or original) ---
    if birdseye:
        p.resetDebugVisualizerCamera(
            cameraDistance=TOPDOWN_DIST,
            cameraYaw=0,        # set to 90/180/270 if you want rotation
            cameraPitch=-89,    # straight down
            cameraTargetPosition=[pos[0], pos[1], 0]
        )
    else:
        p.resetDebugVisualizerCamera(
            cameraDistance=60,
            cameraYaw=35,
            cameraPitch=-35,
            cameraTargetPosition=pos
        )

    p.stepSimulation()
    time.sleep(1/240)

Loaded terrain.
Terrain AABB: (-45.91015625, -45.91015625, -6.0) (45.91015625, 45.91015625, 6.0)
Terrain center: [0.0, 0.0, 0.0]
Texture shape: (1000, 920, 3)
Heightmap shape: 512 512
Meters per texture pixel (approx): 0.1 m/px
Controls:
Move cube: Arrow keys | Up/Down: R/F
Spawn/remove trash objects: press T
Toggle birdseye: press V
Tip: If view looks like grid, press W once (wireframe toggle).


KeyboardInterrupt: 

In [3]:
import os
import time
import numpy as np
import imageio.v2 as imageio
from PIL import Image
import pybullet as p
import pybullet_data

# ----------------------------
# CONFIG
# ----------------------------
HEIGHTMAP = "terrain_heightmap.png"
VISUAL_OBJ = r"d:\UNI\capstone\project_terrain\pybullet\odm_textured_model_geo.obj"  # your OBJ

TERRAIN_SIZE = 125.0
TERRAIN_MAX_HEIGHT = 12.0
MAX_HW = 1024

# ----------------------------
# PyBullet setup
# ----------------------------
if p.isConnected():
    p.disconnect()

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)
p.configureDebugVisualizer(p.COV_ENABLE_SHADOWS, 0)

p.loadURDF("plane.urdf")

# ----------------------------
# Heightfield (collision only)
# ----------------------------
hmap = imageio.imread(HEIGHTMAP)
if hmap.ndim == 3:
    hmap = hmap[..., 0]
hmap = hmap.astype(np.float32)

H, W = hmap.shape
if max(H, W) > MAX_HW:
    hmap = np.array(Image.fromarray(hmap).resize((MAX_HW, MAX_HW), Image.BILINEAR), dtype=np.float32)
    H, W = hmap.shape

den = (hmap.max() - hmap.min())
if den < 1e-9:
    den = 1.0
hmap = (hmap - hmap.min()) / den

sx = TERRAIN_SIZE / W
sy = TERRAIN_SIZE / H

terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, TERRAIN_MAX_HEIGHT],
    heightfieldData=hmap.flatten().tolist(),
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)
terrain_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_shape)
print("terrain_id:", terrain_id)

# ----------------------------
# Load OBJ with collision+visual (for guaranteed AABB)
# ----------------------------
obj_abs = os.path.abspath(VISUAL_OBJ)
obj_dir = os.path.dirname(obj_abs)
print("OBJ path:", obj_abs)
print("OBJ exists:", os.path.exists(obj_abs))

p.setAdditionalSearchPath(obj_dir)

# collision mesh (lets physics compute bounds robustly)
obj_col = p.createCollisionShape(
    shapeType=p.GEOM_MESH,
    fileName=obj_abs,
    meshScale=[1, 1, 1]
)

# visual mesh (force opaque bright material)
obj_vis = p.createVisualShape(
    shapeType=p.GEOM_MESH,
    fileName=obj_abs,
    meshScale=[1, 1, 1],
    rgbaColor=[1, 1, 1, 1]  # fully opaque
)

print("obj_col id:", obj_col)
print("obj_vis id:", obj_vis)

obj_id = p.createMultiBody(
    baseMass=0,
    baseCollisionShapeIndex=obj_col,
    baseVisualShapeIndex=obj_vis,
    basePosition=[0, 0, 0]
)
print("obj_id:", obj_id)

# ----------------------------
# Compute AABB and drop a big marker box there
# ----------------------------
aabb_min, aabb_max = p.getAABB(obj_id)
aabb_min = np.array(aabb_min, dtype=np.float32)
aabb_max = np.array(aabb_max, dtype=np.float32)
center = (aabb_min + aabb_max) / 2.0
extent = (aabb_max - aabb_min)

print("OBJ AABB min:", aabb_min)
print("OBJ AABB max:", aabb_max)
print("OBJ center :", center)
print("OBJ extent :", extent)

# Debug marker cube at mesh center (big + red)
marker_half = [max(1.0, extent[0]*0.02), max(1.0, extent[1]*0.02), max(1.0, extent[2]*0.02)]
marker_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=marker_half)
marker_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=marker_half, rgbaColor=[1, 0, 0, 1])
marker_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=marker_col, baseVisualShapeIndex=marker_vis,
                              basePosition=center.tolist())
print("marker_id:", marker_id)

# Camera to mesh center
diag = float(np.linalg.norm(extent))
p.resetDebugVisualizerCamera(
    cameraDistance=max(10.0, diag * 1.2),
    cameraYaw=35,
    cameraPitch=-35,
    cameraTargetPosition=center.tolist()
)

# ----------------------------
# Loop
# ----------------------------
while p.isConnected():
    p.stepSimulation()
    time.sleep(1/240)

terrain_id: 1
OBJ path: d:\UNI\capstone\project_terrain\pybullet\odm_textured_model_geo.obj
OBJ exists: True
obj_col id: 1
obj_vis id: 0
obj_id: 2
OBJ AABB min: [ -9.825208 -28.222635  -5.730733]
OBJ AABB max: [62.573696 33.190285  8.297997]
OBJ center : [26.374245   2.4838247  1.2836323]
OBJ extent : [72.3989   61.412918 14.02873 ]
marker_id: 3


KeyboardInterrupt: 

In [2]:
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os

HEIGHTMAP = "terrain_heightmap.png"
MESH_FILE = "drone_frames-textured_model.glb"

assert os.path.exists(HEIGHTMAP), f"Missing heightmap: {HEIGHTMAP}"
assert os.path.exists(MESH_FILE), f"Missing mesh: {MESH_FILE}"

# ================================
# RESET + CONNECT
# ================================

if p.isConnected():
    p.disconnect()

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0,0,-9.81)
p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)

# ================================
# LOAD HEIGHTMAP
# ================================

hmap = imageio.imread(HEIGHTMAP)

if hmap.ndim == 3:
    hmap = hmap[...,0]

hmap = hmap.astype(np.float32)

H, W = hmap.shape
print("Heightmap size:", W, "x", H)

hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap invalid or flat.")

hmap01 = (hmap - hmin) / (hmax - hmin)

terrain_size_m = 500.0
max_height_m = 12.0

sx = terrain_size_m / W
sy = terrain_size_m / H

heightfieldData = hmap01.flatten(order='C')

# ================================
# COLLISION SHAPE
# ================================

terrain_col = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, max_height_m],
    heightfieldData=heightfieldData,
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

# ================================
# VISUAL MESH
# ================================

terrain_vis = p.createVisualShape(
    shapeType=p.GEOM_MESH,
    fileName=MESH_FILE,
    meshScale=[1,1,1]  # adjust if needed
)

# ================================
# CREATE TERRAIN
# ================================

terrain_id = p.createMultiBody(
    baseMass=0,
    baseCollisionShapeIndex=terrain_col,
    baseVisualShapeIndex=terrain_vis,
    basePosition=[0,0,0]   # keep at origin
)

print("Terrain ID:", terrain_id)
print("AABB:", p.getAABB(terrain_id))

# ================================
# AUTO CAMERA CENTERING
# ================================

aabb = p.getAABB(terrain_id)
center = [
    (aabb[0][0] + aabb[1][0]) / 2,
    (aabb[0][1] + aabb[1][1]) / 2,
    (aabb[0][2] + aabb[1][2]) / 2
]

terrain_extent = aabb[1][0] - aabb[0][0]

p.resetDebugVisualizerCamera(
    cameraDistance=terrain_extent * 1.2,
    cameraYaw=45,
    cameraPitch=-45,
    cameraTargetPosition=center
)

# ================================
# TEST CUBE
# ================================

cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[2,2,2])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[2,2,2])

cube_id = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[center[0], center[1], 30]
)

# ================================
# CONTROLS
# ================================

speed = 2.0

def down(keys, k):
    return (k in keys) and (keys[k] & p.KEY_IS_DOWN)

print("✅ Heightfield + mesh loaded correctly.")

while p.isConnected():
    keys = p.getKeyboardEvents()
    pos, orn = p.getBasePositionAndOrientation(cube_id)

    if down(keys, p.B3G_UP_ARROW):    pos = [pos[0] + speed, pos[1], pos[2]]
    if down(keys, p.B3G_DOWN_ARROW):  pos = [pos[0] - speed, pos[1], pos[2]]
    if down(keys, p.B3G_LEFT_ARROW):  pos = [pos[0], pos[1] + speed, pos[2]]
    if down(keys, p.B3G_RIGHT_ARROW): pos = [pos[0], pos[1] - speed, pos[2]]
    if down(keys, ord('r')):          pos = [pos[0], pos[1], pos[2] + speed]
    if down(keys, ord('f')):          pos = [pos[0], pos[1], pos[2] - speed]

    p.resetBasePositionAndOrientation(cube_id, pos, orn)

    p.resetDebugVisualizerCamera(
        cameraDistance=terrain_extent * 0.5,
        cameraYaw=45,
        cameraPitch=-40,
        cameraTargetPosition=pos
    )

    p.stepSimulation()
    time.sleep(1/240)


Heightmap size: 1000 x 1000
Terrain ID: 0
AABB: ((-249.75, -249.75, -6.0), (249.75, 249.75, 6.0))
✅ Heightfield + mesh loaded correctly.


In [16]:
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os
from PIL import Image, ImageEnhance

HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"
TEXTURE_OPT = "terrain_texture_opt.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE), f"Missing {TEXTURE}"

# --- notebook-safe reset ---
try:
    if p.isConnected():
        p.disconnect()
except:
    pass

# ----------------------------
# Texture optimize (keep spatial info, no tiling)
# - Resize down to <=4096 if huge (GPU-friendly)
# - Mild contrast/sharpness
# ----------------------------
img = Image.open(TEXTURE).convert("RGB")
w, h = img.size

max_tex = 4096
scale = min(max_tex / max(w, h), 1.0)
if scale < 1.0:
    img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)

img = ImageEnhance.Contrast(img).enhance(1.15)
img = ImageEnhance.Sharpness(img).enhance(1.10)
img.save(TEXTURE_OPT, quality=95)

# ----------------------------
# PyBullet setup
# ----------------------------
p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)
p.configureDebugVisualizer(p.COV_ENABLE_SHADOWS, 0)

# ----------------------------
# Load heightmap
# ----------------------------
hmap = imageio.imread(HEIGHTMAP)
if hmap.ndim == 3:
    hmap = hmap[..., 0]
hmap = hmap.astype(np.float32)
H, W = hmap.shape

# Downsample heightmap for performance (does NOT change texture detail)
MAX_HW = 512
if max(H, W) > MAX_HW:
    hmap_img = Image.fromarray(hmap)
    hmap_img = hmap_img.resize((MAX_HW, MAX_HW), resample=Image.BILINEAR)
    hmap = np.array(hmap_img).astype(np.float32)
    H, W = hmap.shape

hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap is flat (all same value).")
hmap01 = (hmap - hmin) / (hmax - hmin)

# ----------------------------
# Choose terrain size based on texture resolution (IMPORTANT)
# ----------------------------
tex = imageio.imread(TEXTURE_OPT)
tex_w = tex.shape[1]

# Target ~0.10 to 0.20 m/px for “garbage visibility”.
target_m_per_px = 0.15

recommended_size_m = tex_w * target_m_per_px

# Cap at 500m if your real area is that large
terrain_size_m = min(500.0, recommended_size_m)

desired_max_height = 12.0

sx = terrain_size_m / W
sy = terrain_size_m / H

terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldTextureScaling=1.0,
    heightfieldData=hmap01.flatten().tolist(),
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

terrain_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_shape)

tex_id = p.loadTexture(TEXTURE_OPT)
p.changeVisualShape(terrain_id, -1, textureUniqueId=tex_id)
p.changeDynamics(terrain_id, -1, lateralFriction=0.9, restitution=0.0)

# --- Cube
cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5])
cube_vis = p.createVisualShape(
    p.GEOM_BOX,
    halfExtents=[0.5, 0.5, 0.5],
    rgbaColor=[1, 0, 0, 1]
)
cube_id  = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[terrain_size_m/2, terrain_size_m/2, 20]
)

print("Loaded terrain + cube.")
print("Terrain size (m):", terrain_size_m)
print("Texture width(px):", tex_w)
print("Meters per texture pixel:", terrain_size_m / tex_w)

# ----------------------------
# CAMERA MODES
# Default: close top-down over the cube (best for spotting garbage)
# Press T: toggle whole-terrain bird's-eye view
# ----------------------------
top_view = False

# Start with close top-down centered
p.resetDebugVisualizerCamera(
    cameraDistance=12,
    cameraYaw=0,
    cameraPitch=-89,
    cameraTargetPosition=[terrain_size_m/2, terrain_size_m/2, 0]
)

# Controls
speed = 0.8

def down(keys, k):
    return (k in keys) and (keys[k] & p.KEY_IS_DOWN)

def triggered(keys, k):
    return (k in keys) and (keys[k] & p.KEY_WAS_TRIGGERED)

while p.isConnected():
    keys = p.getKeyboardEvents()
    pos, orn = p.getBasePositionAndOrientation(cube_id)
    pos = list(pos)

    # Move cube
    if down(keys, p.B3G_UP_ARROW):    pos[0] += speed
    if down(keys, p.B3G_DOWN_ARROW):  pos[0] -= speed
    if down(keys, p.B3G_LEFT_ARROW):  pos[1] += speed
    if down(keys, p.B3G_RIGHT_ARROW): pos[1] -= speed
    if down(keys, ord('r')):          pos[2] += speed
    if down(keys, ord('f')):          pos[2] -= speed

    p.resetBasePositionAndOrientation(cube_id, pos, orn)

    # Toggle whole-terrain top view
    if triggered(keys, ord('t')):
        top_view = not top_view
        print("Top view:", top_view)

    # Camera
    if top_view:
        # Whole terrain bird’s-eye
        p.resetDebugVisualizerCamera(
            cameraDistance=terrain_size_m * 1.2,
            cameraYaw=0,
            cameraPitch=-89,
            cameraTargetPosition=[terrain_size_m/2, terrain_size_m/2, 0]
        )
    else:
        # Close top-down follow the cube (best for spotting small garbage)
        p.resetDebugVisualizerCamera(
            cameraDistance=12,     # try 8..20
            cameraYaw=0,
            cameraPitch=-89,
            cameraTargetPosition=pos
        )

    p.stepSimulation()
    time.sleep(1/240)

Loaded terrain + cube.
Terrain size (m): 138.0
Texture width(px): 920
Meters per texture pixel: 0.15


KeyboardInterrupt: 

In [20]:
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os
from PIL import Image, ImageEnhance, ImageOps

# ----------------------------
# FILES
# ----------------------------
HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"
TEXTURE_OPT = "terrain_texture_opt.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE), f"Missing {TEXTURE}"

# ----------------------------
# Optional OpenCV (best enhancement + live window)
# ----------------------------
HAS_CV2 = True
try:
    import cv2
except Exception:
    HAS_CV2 = False
    cv2 = None
    print("cv2 not found -> will use PIL enhancement and save frames to ./renders/")

# ----------------------------
# Notebook-safe reset
# ----------------------------
try:
    if p.isConnected():
        p.disconnect()
except:
    pass

# ----------------------------
# Texture optimize (no tiling, keep spatial locations)
# ----------------------------
img = Image.open(TEXTURE).convert("RGB")
w, h = img.size

max_tex = 4096
scale = min(max_tex / max(w, h), 1.0)
if scale < 1.0:
    img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)

img = ImageEnhance.Contrast(img).enhance(1.15)
img = ImageEnhance.Sharpness(img).enhance(1.10)
img.save(TEXTURE_OPT, quality=95)

# ----------------------------
# PyBullet setup
# ----------------------------
p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)
p.configureDebugVisualizer(p.COV_ENABLE_SHADOWS, 0)

# ----------------------------
# Load heightmap
# ----------------------------
hmap = imageio.imread(HEIGHTMAP)
if hmap.ndim == 3:
    hmap = hmap[..., 0]
hmap = hmap.astype(np.float32)
H, W = hmap.shape

MAX_HW = 512
if max(H, W) > MAX_HW:
    hmap_img = Image.fromarray(hmap)
    hmap_img = hmap_img.resize((MAX_HW, MAX_HW), resample=Image.BILINEAR)
    hmap = np.array(hmap_img).astype(np.float32)
    H, W = hmap.shape

hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap is flat (all same value).")
hmap01 = (hmap - hmin) / (hmax - hmin)

# ----------------------------
# Choose terrain size from texture density
# ----------------------------
tex = imageio.imread(TEXTURE_OPT)
tex_w = tex.shape[1]

target_m_per_px = 0.15  # you already measured this works
recommended_size_m = tex_w * target_m_per_px
terrain_size_m = min(500.0, recommended_size_m)

desired_max_height = 12.0

sx = terrain_size_m / W
sy = terrain_size_m / H

terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldTextureScaling=1.0,
    heightfieldData=hmap01.flatten().tolist(),
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

terrain_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_shape)

tex_id = p.loadTexture(TEXTURE_OPT)
p.changeVisualShape(terrain_id, -1, textureUniqueId=tex_id, specularColor=[0, 0, 0])
p.changeDynamics(terrain_id, -1, lateralFriction=0.9, restitution=0.0)

# ----------------------------
# Cube (move this around to inspect areas)
# ----------------------------
cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5], rgbaColor=[1, 0, 0, 1])
cube_id  = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[terrain_size_m/2, terrain_size_m/2, 20]
)

print("Loaded terrain + cube.")
print("Terrain size (m):", terrain_size_m)
print("Texture width(px):", tex_w)
print("Meters per texture pixel:", terrain_size_m / tex_w)

# ----------------------------
# Enhanced top-down render (Option A)
# ----------------------------
def enhance_frame_rgb(frame_rgb):
    """
    frame_rgb: uint8 (H,W,3) in RGB
    returns enhanced RGB uint8
    """
    if HAS_CV2:
        # CLAHE on L channel (local contrast)
        lab = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l2 = clahe.apply(l)
        lab2 = cv2.merge([l2, a, b])
        out = cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)

        # Sharpen
        kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
        out = cv2.filter2D(out, -1, kernel)
        return out
    else:
        # PIL fallback: autocontrast + a bit of sharpness
        im = Image.fromarray(frame_rgb, mode="RGB")
        im = ImageOps.autocontrast(im, cutoff=1)
        im = ImageEnhance.Contrast(im).enhance(1.2)
        im = ImageEnhance.Sharpness(im).enhance(1.5)
        return np.array(im, dtype=np.uint8)

def render_topdown_tiny(target_pos, img_w=900, img_h=900, cam_height=80, fov=30):
    eye = [target_pos[0], target_pos[1], cam_height]
    target = [target_pos[0], target_pos[1], 0]
    up = [0, 1, 0]

    view = p.computeViewMatrix(eye, target, up)
    proj = p.computeProjectionMatrixFOV(
        fov=fov,
        aspect=img_w / img_h,
        nearVal=0.1,
        farVal=500
    )

    w, h, rgba, depth, seg = p.getCameraImage(
        width=img_w,
        height=img_h,
        viewMatrix=view,
        projectionMatrix=proj,
        renderer=p.ER_TINY_RENDERER
    )

    img = np.reshape(rgba, (h, w, 4))[:, :, :3].astype(np.uint8)  # RGB
    return img

# ----------------------------
# Camera (GUI)
# T toggle: whole terrain vs close top-down follow
# ----------------------------
top_view = False

def set_gui_camera(pos):
    if top_view:
        p.resetDebugVisualizerCamera(
            cameraDistance=terrain_size_m * 1.2,
            cameraYaw=0,
            cameraPitch=-89,
            cameraTargetPosition=[terrain_size_m/2, terrain_size_m/2, 0]
        )
    else:
        p.resetDebugVisualizerCamera(
            cameraDistance=12,
            cameraYaw=0,
            cameraPitch=-89,
            cameraTargetPosition=pos
        )

set_gui_camera([terrain_size_m/2, terrain_size_m/2, 0])

# ----------------------------
# Controls + loop
# ----------------------------
speed = 0.8
render_every_n = 10
frame_idx = 0

# If no cv2, save frames here
if not HAS_CV2:
    os.makedirs("renders", exist_ok=True)

def down(keys, k):
    return (k in keys) and (keys[k] & p.KEY_IS_DOWN)

def triggered(keys, k):
    return (k in keys) and (keys[k] & p.KEY_WAS_TRIGGERED)


def set_fixed_topdown_camera():
    p.resetDebugVisualizerCamera(
        cameraDistance=terrain_size_m * 0.8,
        cameraYaw=0,
        cameraPitch=-89.9,   # force true vertical
        cameraTargetPosition=[terrain_size_m/2, terrain_size_m/2, 0]
    )

set_fixed_topdown_camera()

while p.isConnected():
    keys = p.getKeyboardEvents()
    pos, orn = p.getBasePositionAndOrientation(cube_id)
    pos = list(pos)

    # movement
    if down(keys, p.B3G_UP_ARROW):    pos[0] += speed
    if down(keys, p.B3G_DOWN_ARROW):  pos[0] -= speed
    if down(keys, p.B3G_LEFT_ARROW):  pos[1] += speed
    if down(keys, p.B3G_RIGHT_ARROW): pos[1] -= speed
    if down(keys, ord('r')):          pos[2] += speed
    if down(keys, ord('f')):          pos[2] -= speed

    p.resetBasePositionAndOrientation(cube_id, pos, orn)

    # toggle GUI view
    if triggered(keys, ord('t')):
        top_view = not top_view
        print("Top view:", top_view)

    # Option A render (TinyRenderer) every N frames
    frame_idx += 1
    if frame_idx % render_every_n == 0:
        raw = render_topdown_tiny(pos, img_w=1400, img_h=1400, cam_height=160, fov=12)
        enh = enhance_frame_rgb(raw)

        if HAS_CV2:
            # cv2 expects BGR
            cv2.imshow("Top-down Enhanced (TinyRenderer)", cv2.cvtColor(enh, cv2.COLOR_RGB2BGR))
            # press Q in the image window to stop (optional)
            if cv2.waitKey(1) & 0xFF in [ord('q'), ord('Q')]:
                break
        else:
            # Save occasional frames
            out_path = os.path.join("renders", f"enh_{frame_idx:06d}.png")
            Image.fromarray(enh).save(out_path)
            if frame_idx % (render_every_n * 20) == 0:
                print("Saved:", out_path)

    p.stepSimulation()
    time.sleep(1/240)

if HAS_CV2:
    cv2.destroyAllWindows()
p.disconnect()

cv2 not found -> will use PIL enhancement and save frames to ./renders/
Loaded terrain + cube.
Terrain size (m): 138.0
Texture width(px): 920
Meters per texture pixel: 0.15
Saved: renders\enh_000200.png
Saved: renders\enh_000400.png
Saved: renders\enh_000600.png
Saved: renders\enh_000800.png
Saved: renders\enh_001000.png
Saved: renders\enh_001200.png
Saved: renders\enh_001400.png


KeyboardInterrupt: 

In [23]:
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os
from PIL import Image, ImageEnhance, ImageOps

# ----------------------------
# FILES
# ----------------------------
HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"
TEXTURE_OPT = "terrain_texture_opt.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE), f"Missing {TEXTURE}"

# ----------------------------
# Optional OpenCV
# ----------------------------
HAS_CV2 = True
try:
    import cv2
except Exception:
    HAS_CV2 = False
    cv2 = None
    print("cv2 not found -> will save frames to ./renders/")

# ----------------------------
# Reset
# ----------------------------
try:
    if p.isConnected():
        p.disconnect()
except:
    pass

# ----------------------------
# Texture optimize
# ----------------------------
img = Image.open(TEXTURE).convert("RGB")
w, h = img.size

max_tex = 4096
scale = min(max_tex / max(w, h), 1.0)
if scale < 1.0:
    img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)

img = ImageEnhance.Contrast(img).enhance(1.15)
img = ImageEnhance.Sharpness(img).enhance(1.10)
img.save(TEXTURE_OPT, quality=95)

# ----------------------------
# PyBullet setup
# ----------------------------
p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)

# Disable shadows
p.configureDebugVisualizer(p.COV_ENABLE_SHADOWS, 0)

# ----------------------------
# Load heightmap
# ----------------------------
hmap = imageio.imread(HEIGHTMAP)
if hmap.ndim == 3:
    hmap = hmap[..., 0]

hmap = hmap.astype(np.float32)
H, W = hmap.shape

MAX_HW = 512
if max(H, W) > MAX_HW:
    hmap_img = Image.fromarray(hmap)
    hmap_img = hmap_img.resize((MAX_HW, MAX_HW), resample=Image.BILINEAR)
    hmap = np.array(hmap_img).astype(np.float32)
    H, W = hmap.shape

hmin, hmax = float(hmap.min()), float(hmap.max())
hmap01 = (hmap - hmin) / (hmax - hmin)

# ----------------------------
# Terrain sizing
# ----------------------------
tex = imageio.imread(TEXTURE_OPT)
tex_w = tex.shape[1]

target_m_per_px = 0.15
terrain_size_m = min(500.0, tex_w * target_m_per_px)

desired_max_height = 12.0

sx = terrain_size_m / W
sy = terrain_size_m / H

terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldTextureScaling=1.0,
    heightfieldData=hmap01.flatten().tolist(),
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

terrain_id = p.createMultiBody(
    baseMass=0,
    baseCollisionShapeIndex=terrain_shape
)

# Apply texture cleanly
tex_id = p.loadTexture(TEXTURE_OPT)
p.changeVisualShape(
    terrain_id,
    -1,
    textureUniqueId=tex_id,
    rgbaColor=[1,1,1,1],
    specularColor=[0,0,0]
)

# ----------------------------
# Cube
# ----------------------------
cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5], rgbaColor=[1,0,0,1])

cube_id = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[terrain_size_m/2, terrain_size_m/2, 20]
)

# Move to pink-object region (proof test)
test_x = terrain_size_m * 0.65
test_y = terrain_size_m * 0.45
_, orn = p.getBasePositionAndOrientation(cube_id)
p.resetBasePositionAndOrientation(cube_id, [test_x, test_y, 20], orn)

# ----------------------------
# Enhancement
# ----------------------------
def enhance_frame_rgb(frame_rgb):
    if HAS_CV2:
        lab = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l2 = clahe.apply(l)
        lab2 = cv2.merge([l2, a, b])
        out = cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)

        kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
        out = cv2.filter2D(out, -1, kernel)

        # Garbage brightness boost
        hsv = cv2.cvtColor(out, cv2.COLOR_RGB2HSV)
        h, s, v = cv2.split(hsv)
        v = cv2.convertScaleAbs(v, alpha=1.25, beta=10)
        hsv = cv2.merge([h, s, v])
        out = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

        return out
    else:
        im = Image.fromarray(frame_rgb)
        im = ImageOps.autocontrast(im, cutoff=1)
        im = ImageEnhance.Contrast(im).enhance(1.2)
        im = ImageEnhance.Sharpness(im).enhance(1.5)
        return np.array(im)

# ----------------------------
# Orthographic-like render
# ----------------------------
def render_topdown_tiny(target_pos):
    img_w = 1600
    img_h = 1600
    cam_height = 60
    fov = 10

    eye = [target_pos[0], target_pos[1], cam_height]
    target = [target_pos[0], target_pos[1], 0]
    up = [0,1,0]

    view = p.computeViewMatrix(eye, target, up)
    proj = p.computeProjectionMatrixFOV(
        fov=fov,
        aspect=1,
        nearVal=0.1,
        farVal=500
    )

    w, h, rgba, depth, seg = p.getCameraImage(
        width=img_w,
        height=img_h,
        viewMatrix=view,
        projectionMatrix=proj,
        renderer=p.ER_TINY_RENDERER
    )

    return np.reshape(rgba, (h, w, 4))[:,:,:3].astype(np.uint8)

# ----------------------------
# Main Loop
# ----------------------------
speed = 0.8
frame_idx = 0

if not HAS_CV2:
    os.makedirs("renders", exist_ok=True)

while p.isConnected():

    keys = p.getKeyboardEvents()
    pos, orn = p.getBasePositionAndOrientation(cube_id)
    pos = list(pos)

    # Movement
    if p.B3G_UP_ARROW in keys and keys[p.B3G_UP_ARROW] & p.KEY_IS_DOWN:
        pos[0] += speed
    if p.B3G_DOWN_ARROW in keys and keys[p.B3G_DOWN_ARROW] & p.KEY_IS_DOWN:
        pos[0] -= speed
    if p.B3G_LEFT_ARROW in keys and keys[p.B3G_LEFT_ARROW] & p.KEY_IS_DOWN:
        pos[1] += speed
    if p.B3G_RIGHT_ARROW in keys and keys[p.B3G_RIGHT_ARROW] & p.KEY_IS_DOWN:
        pos[1] -= speed

    p.resetBasePositionAndOrientation(cube_id, pos, orn)

    # ---- Stable Top-Down GUI Camera (follows cube) ----
    p.resetDebugVisualizerCamera(
        cameraDistance=25,
        cameraYaw=0,
        cameraPitch=-89.9,
        cameraTargetPosition=[pos[0], pos[1], 0]
    )

    # Render enhanced synthetic image
    frame_idx += 1
    if frame_idx % 10 == 0:
        raw = render_topdown_tiny(pos)
        enh = enhance_frame_rgb(raw)

        if HAS_CV2:
            cv2.imshow("Top-down Enhanced", cv2.cvtColor(enh, cv2.COLOR_RGB2BGR))
            if cv2.waitKey(1) & 0xFF in [ord('q'), ord('Q')]:
                break
        else:
            Image.fromarray(enh).save(f"renders/frame_{frame_idx:06d}.png")

    p.stepSimulation()
    time.sleep(1/240)

if HAS_CV2:
    cv2.destroyAllWindows()
p.disconnect()


cv2 not found -> will save frames to ./renders/


KeyboardInterrupt: 

In [ ]:
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os

HEIGHTMAP = "heightmap-blender3.png"
MESH_FILE = "blender-terrain.obj"
TEXTURE_FILE = "heightmap-blender3.png"  # Using the heightmap itself as texture

assert os.path.exists(HEIGHTMAP), f"Missing heightmap: {HEIGHTMAP}"
assert os.path.exists(MESH_FILE), f"Missing mesh: {MESH_FILE}"
assert os.path.exists(TEXTURE_FILE), f"Missing texture: {TEXTURE_FILE}"

# ================================
# RESET + CONNECT
# ================================

if p.isConnected():
    p.disconnect()

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)
p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)

# ================================
# LOAD HEIGHTMAP
# ================================

hmap = imageio.imread(HEIGHTMAP)

if hmap.ndim == 3:
    hmap = hmap[..., 0]

hmap = hmap.astype(np.float32)

H, W = hmap.shape
print("Heightmap size:", W, "x", H)

hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap invalid or flat.")

hmap01 = (hmap - hmin) / (hmax - hmin)

terrain_size_m = 500.0
max_height_m = 12.0

sx = terrain_size_m / W
sy = terrain_size_m / H

heightfieldData = hmap01.flatten(order='C')

# ================================
# COLLISION SHAPE
# ================================

terrain_col = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, max_height_m],
    heightfieldData=heightfieldData,
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

# ================================
# VISUAL MESH (Scaled to match terrain size)
# ================================

terrain_vis = p.createVisualShape(
    shapeType=p.GEOM_MESH,
    fileName=MESH_FILE,
    meshScale=[terrain_size_m, terrain_size_m, max_height_m]
)

# ================================
# CREATE TERRAIN
# ================================

terrain_id = p.createMultiBody(
    baseMass=0,
    baseCollisionShapeIndex=terrain_col,
    baseVisualShapeIndex=terrain_vis,
    basePosition=[0, 0, 0]  # keep at origin
)

# ================================
# LOAD AND APPLY TEXTURE
# ================================

texture_id = p.loadTexture(TEXTURE_FILE)
p.changeVisualShape(terrain_id, -1, textureUniqueId=texture_id)

print("Terrain ID:", terrain_id)
print("AABB:", p.getAABB(terrain_id))

# ================================
# AUTO CAMERA CENTERING
# ================================

aabb = p.getAABB(terrain_id)
center = [
    (aabb[0][0] + aabb[1][0]) / 2,
    (aabb[0][1] + aabb[1][1]) / 2,
    (aabb[0][2] + aabb[1][2]) / 2
]

terrain_extent = aabb[1][0] - aabb[0][0]

p.resetDebugVisualizerCamera(
    cameraDistance=terrain_extent * 1.2,
    cameraYaw=45,
    cameraPitch=-45,
    cameraTargetPosition=center
)

# ================================
# TEST CUBE
# ================================

cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[2, 2, 2])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[2, 2, 2])

cube_id = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[center[0], center[1], 30]
)

# ================================
# CONTROLS
# ================================

speed = 2.0

def down(keys, k):
    return (k in keys) and (keys[k] & p.KEY_IS_DOWN)

print("✅ Heightfield + mesh + texture loaded correctly.")

while p.isConnected():
    keys = p.getKeyboardEvents()
    pos, orn = p.getBasePositionAndOrientation(cube_id)

    if down(keys, p.B3G_UP_ARROW):    pos = [pos[0] + speed, pos[1], pos[2]]
    if down(keys, p.B3G_DOWN_ARROW):  pos = [pos[0] - speed, pos[1], pos[2]]
    if down(keys, p.B3G_LEFT_ARROW):  pos = [pos[0], pos[1] + speed, pos[2]]
    if down(keys, p.B3G_RIGHT_ARROW): pos = [pos[0], pos[1] - speed, pos[2]]
    if down(keys, ord('r')):          pos = [pos[0], pos[1], pos[2] + speed]
    if down(keys, ord('f')):          pos = [pos[0], pos[1], pos[2] - speed]

    p.resetBasePositionAndOrientation(cube_id, pos, orn)

    p.resetDebugVisualizerCamera(
        cameraDistance=terrain_extent * 0.5,
        cameraYaw=45,
        cameraPitch=-40,
        cameraTargetPosition=pos
    )

    p.stepSimulation()
    time.sleep(1 / 240)


Heightmap size: 1024 x 1024
Terrain ID: 0
AABB: ((-249.755859375, -249.755859375, -6.0), (249.755859375, 249.755859375, 6.0))
✅ Heightfield + mesh + texture loaded correctly.


In [14]:
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os

HEIGHTMAP = "terrain_heightmap.png"
MESH_GLB  = "drone_frames-textured_model.glb"   # <-- put your real .glb filename here

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(MESH_GLB),  f"Missing {MESH_GLB}"

# -----------------------------
# 0) CONNECT + CLEAN
# -----------------------------
try:
    if p.isConnected():
        p.disconnect()
except:
    pass

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)

p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)
p.configureDebugVisualizer(p.COV_ENABLE_RGB_BUFFER_PREVIEW, 0)
p.configureDebugVisualizer(p.COV_ENABLE_DEPTH_BUFFER_PREVIEW, 0)
p.configureDebugVisualizer(p.COV_ENABLE_SEGMENTATION_MARK_PREVIEW, 0)
p.configureDebugVisualizer(p.COV_ENABLE_SHADOWS, 0)
p.configureDebugVisualizer(p.COV_ENABLE_RENDERING, 1)

# -----------------------------
# 1) HEIGHTFIELD COLLISION
# -----------------------------
hmap = imageio.imread(HEIGHTMAP)
if hmap.ndim == 3:
    hmap = hmap[..., 0]
hmap = hmap.astype(np.float32)
H, W = hmap.shape

hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap is flat.")
hmap01 = (hmap - hmin) / (hmax - hmin)
heightfieldData = hmap01.flatten().tolist()

terrain_size_m = 150.0
desired_max_height = 12.0
sx = terrain_size_m / W
sy = terrain_size_m / H

terrain_col = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldData=heightfieldData,
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

# Collision-only body (to measure heightfield bounds)
hf_body = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_col)

hf_min, hf_max = p.getAABB(hf_body)
hf_size = [hf_max[i] - hf_min[i] for i in range(3)]
hf_center = [(hf_min[i] + hf_max[i]) / 2 for i in range(3)]
print("Heightfield size:", hf_size, "center:", hf_center)

# -----------------------------
# 2) LOAD GLB TEMPORARILY TO MEASURE ITS AABB
# -----------------------------
tmp_vis = p.createVisualShape(
    shapeType=p.GEOM_MESH,
    fileName=MESH_GLB,
    meshScale=[1, 1, 1]
)
tmp_body = p.createMultiBody(baseMass=0, baseVisualShapeIndex=tmp_vis)

mmin, mmax = p.getAABB(tmp_body)
msize = [mmax[i] - mmin[i] for i in range(3)]
mcenter = [(mmin[i] + mmax[i]) / 2 for i in range(3)]
print("GLB mesh size:", msize, "center:", mcenter)

# If still zero, GLB is not loading (but this is rare)
eps = 1e-9
if msize[0] < eps and msize[1] < eps and msize[2] < eps:
    raise RuntimeError("GLB loaded as empty in PyBullet. Check the file is a real mesh GLB.")

p.removeBody(tmp_body)

# -----------------------------
# 3) AUTO SCALE GLB TO MATCH HEIGHTFIELD XY
# -----------------------------
scale_x = hf_size[0] / max(msize[0], eps)
scale_y = hf_size[1] / max(msize[1], eps)
scale_xy = (scale_x + scale_y) / 2.0

mesh_scale = [scale_xy, scale_xy, scale_xy]
print("Computed mesh_scale:", mesh_scale)

terrain_vis = p.createVisualShape(
    shapeType=p.GEOM_MESH,
    fileName=MESH_GLB,
    meshScale=mesh_scale
)

# Create final body with both collision + visual
terrain_id = p.createMultiBody(
    baseMass=0,
    baseCollisionShapeIndex=terrain_col,
    baseVisualShapeIndex=terrain_vis,
    basePosition=[0, 0, 0]
)

# -----------------------------
# 4) AUTO CENTER TO HEIGHTFIELD
# -----------------------------
tmin, tmax = p.getAABB(terrain_id)
tcenter = [(tmin[i] + tmax[i]) / 2 for i in range(3)]

shift = [
    hf_center[0] - tcenter[0],
    hf_center[1] - tcenter[1],
    0.0
]
p.resetBasePositionAndOrientation(terrain_id, shift, [0, 0, 0, 1])

# Remove collision-only body
p.removeBody(hf_body)

# -----------------------------
# 5) DEBUG AXES + CUBE
# -----------------------------
p.addUserDebugLine([0,0,0],[10,0,0],[1,0,0],2,0)
p.addUserDebugLine([0,0,0],[0,10,0],[0,1,0],2,0)
p.addUserDebugLine([0,0,0],[0,0,10],[0,0,1],2,0)

cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5,0.5,0.5])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.5,0.5,0.5], rgbaColor=[1,0,0,1])
cube_id  = p.createMultiBody(baseMass=1, baseCollisionShapeIndex=cube_col, baseVisualShapeIndex=cube_vis,
                             basePosition=[hf_center[0], hf_center[1], 20])

# Camera
p.resetDebugVisualizerCamera(
    cameraDistance=max(hf_size[0], hf_size[1]) * 1.2,
    cameraYaw=45,
    cameraPitch=-45,
    cameraTargetPosition=[hf_center[0], hf_center[1], 0]
)

print("If your GLB contains texture, you should now see the textured terrain (trash included).")

# -----------------------------
# 6) LOOP
# -----------------------------
while p.isConnected():
    p.stepSimulation()
    time.sleep(1/240)

Heightfield size: [149.85, 149.85, 12.0] center: [0.0, 0.0, 0.0]
GLB mesh size: [0.0, 0.0, 0.0] center: [0.0, 0.0, 0.0]


RuntimeError: GLB loaded as empty in PyBullet. Check the file is a real mesh GLB.

In [2]:
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os, random

HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE), f"Missing {TEXTURE}"

# --- notebook-safe reset (ONLY ONE GUI CONNECTION) ---
try:
    if p.isConnected():
        p.disconnect()
except:
    pass

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)

# IMPORTANT: avoid wireframe confusion
p.configureDebugVisualizer(p.COV_ENABLE_WIREFRAME, 0)

# --- Load heightmap ---
hmap = imageio.imread(HEIGHTMAP)

# Ensure single channel
if hmap.ndim == 3:
    hmap = hmap[..., 0]

hmap = hmap.astype(np.float32)
H, W = hmap.shape

# (Optional) downsample if huge
MAX_HW = 512
if max(H, W) > MAX_HW:
    from PIL import Image
    hmap_img = Image.fromarray(hmap.astype(np.uint16) if hmap.max() > 255 else hmap.astype(np.uint8))
    hmap_img = hmap_img.resize((MAX_HW, MAX_HW), resample=Image.BILINEAR)
    hmap = np.array(hmap_img).astype(np.float32)
    H, W = hmap.shape

# Normalize to [0, 1]
hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap is flat (all same value).")
hmap01 = (hmap - hmin) / (hmax - hmin)

heightfieldData = hmap01.flatten().tolist()

# --- Read texture and pick terrain size to improve visibility ---
tex = imageio.imread(TEXTURE)
tex_w = tex.shape[1]

# TARGET: meters per pixel (smaller = more visible trash)
# If your trash is subtle in texture, aim for ~0.05 to 0.15 m/px
TARGET_M_PER_PX = 0.10
terrain_size_m = float(tex_w) * TARGET_M_PER_PX

# Cap to avoid insane sizes
terrain_size_m = max(80.0, min(terrain_size_m, 300.0))

desired_max_height = 12.0

sx = terrain_size_m / W
sy = terrain_size_m / H

# --- Create terrain ---
terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldTextureScaling=1.0,
    heightfieldData=heightfieldData,
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

terrain_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_shape)

# Texture
tex_id = p.loadTexture(TEXTURE)
p.changeVisualShape(terrain_id, -1, textureUniqueId=tex_id)

p.changeDynamics(terrain_id, -1, lateralFriction=0.9, restitution=0.0)

# Terrain is usually centered around origin in PyBullet heightfields
aabb_min, aabb_max = p.getAABB(terrain_id)
center = [(aabb_min[i] + aabb_max[i]) / 2 for i in range(3)]

print("Loaded terrain.")
print("Terrain AABB:", aabb_min, aabb_max)
print("Terrain center:", center)

print("Texture shape:", tex.shape)
print("Heightmap shape:", H, W)
print("Meters per texture pixel (approx):", terrain_size_m / tex_w, "m/px")

# --- Spawn a cube near center ---
cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5], rgbaColor=[1, 0, 0, 1])
cube_id  = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[center[0], center[1], center[2] + 20]
)

# --- Camera (start on center) ---
p.resetDebugVisualizerCamera(
    cameraDistance= max(60, terrain_size_m * 0.8),
    cameraYaw=35,
    cameraPitch=-35,
    cameraTargetPosition=[center[0], center[1], 0]
)

print("Controls:")
print("Move cube: Arrow keys | Up/Down: R/F")
print("Spawn/remove trash objects: press T")
print("Tip: If view looks like grid, press W once (wireframe toggle).")

# --- Trash spawning (RELIABLE FOR YOLO) ---
trash_ids = []
trash_on = False

def pressed(keys, k):
    return (k in keys) and (keys[k] & p.KEY_WAS_TRIGGERED)

def spawn_trash(n=60):
    """Spawn visible trash proxies sitting on terrain (good for YOLO)."""
    ids = []
    xmin, ymin = aabb_min[0] + 2.0, aabb_min[1] + 2.0
    xmax, ymax = aabb_max[0] - 2.0, aabb_max[1] - 2.0

    for _ in range(n):
        x = random.uniform(xmin, xmax)
        y = random.uniform(ymin, ymax)

        hit = p.rayTest([x, y, 200], [x, y, -200])[0]
        hit_obj, hit_pos = hit[0], hit[3]
        if hit_obj != terrain_id:
            continue

        z = hit_pos[2]

        # bottle (cyl) or bag (thin box)
        if random.random() < 0.6:
            r = random.uniform(0.04, 0.08)
            h = random.uniform(0.15, 0.28)
            col = p.createCollisionShape(p.GEOM_CYLINDER, radius=r, height=h)
            vis = p.createVisualShape(p.GEOM_CYLINDER, radius=r, length=h, rgbaColor=[0.1, 0.9, 0.1, 1])
            yaw = random.uniform(0, 3.14159)
            tid = p.createMultiBody(
                baseMass=0.05,
                baseCollisionShapeIndex=col,
                baseVisualShapeIndex=vis,
                basePosition=[x, y, z + h/2 + 0.01],
                baseOrientation=p.getQuaternionFromEuler([0, 0, yaw])
            )
        else:
            hx = random.uniform(0.08, 0.16)
            hy = random.uniform(0.08, 0.16)
            hz = random.uniform(0.01, 0.02)
            col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[hx, hy, hz])
            vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[hx, hy, hz], rgbaColor=[0.9, 0.1, 0.1, 1])
            yaw = random.uniform(0, 3.14159)
            tid = p.createMultiBody(
                baseMass=0.03,
                baseCollisionShapeIndex=col,
                baseVisualShapeIndex=vis,
                basePosition=[x, y, z + hz + 0.01],
                baseOrientation=p.getQuaternionFromEuler([0, 0, yaw])
            )
        ids.append(tid)

    return ids

def clear_trash():
    global trash_ids
    for tid in trash_ids:
        try:
            p.removeBody(tid)
        except:
            pass
    trash_ids = []

# --- Controls ---
speed = 0.8

def down(keys, k):
    return (k in keys) and (keys[k] & p.KEY_IS_DOWN)

while p.isConnected():
    keys = p.getKeyboardEvents()
    pos, orn = p.getBasePositionAndOrientation(cube_id)

    # Toggle trash
    if pressed(keys, ord('t')):
        trash_on = not trash_on
        if trash_on:
            clear_trash()
            trash_ids = spawn_trash(n=80)
            print("Trash ON:", len(trash_ids))
        else:
            clear_trash()
            print("Trash OFF")

    # Move cube
    if down(keys, p.B3G_UP_ARROW):    pos = [pos[0] + speed, pos[1], pos[2]]
    if down(keys, p.B3G_DOWN_ARROW):  pos = [pos[0] - speed, pos[1], pos[2]]
    if down(keys, p.B3G_LEFT_ARROW):  pos = [pos[0], pos[1] + speed, pos[2]]
    if down(keys, p.B3G_RIGHT_ARROW): pos = [pos[0], pos[1] - speed, pos[2]]

    # Altitude
    if down(keys, ord('r')): pos = [pos[0], pos[1], pos[2] + speed]
    if down(keys, ord('f')): pos = [pos[0], pos[1], pos[2] - speed]

    p.resetBasePositionAndOrientation(cube_id, pos, orn)

    # Camera follow
    p.resetDebugVisualizerCamera(
        cameraDistance=60,
        cameraYaw=35,
        cameraPitch=-35,
        cameraTargetPosition=pos
    )

    p.stepSimulation()
    time.sleep(1/240)

Loaded terrain.
Terrain AABB: (-45.91015625, -45.91015625, -6.0) (45.91015625, 45.91015625, 6.0)
Terrain center: [0.0, 0.0, 0.0]
Texture shape: (1000, 920, 3)
Heightmap shape: 512 512
Meters per texture pixel (approx): 0.1 m/px
Controls:
Move cube: Arrow keys | Up/Down: R/F
Spawn/remove trash objects: press T
Tip: If view looks like grid, press W once (wireframe toggle).


In [3]:
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os, random

HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE), f"Missing {TEXTURE}"

# --- notebook-safe reset (ONLY ONE GUI CONNECTION) ---
try:
    if p.isConnected():
        p.disconnect()
except:
    pass

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)

# IMPORTANT: avoid wireframe confusion
p.configureDebugVisualizer(p.COV_ENABLE_WIREFRAME, 0)

# --- Load heightmap ---
hmap = imageio.imread(HEIGHTMAP)

# Ensure single channel
if hmap.ndim == 3:
    hmap = hmap[..., 0]

hmap = hmap.astype(np.float32)
H, W = hmap.shape

# (Optional) downsample if huge
MAX_HW = 512
if max(H, W) > MAX_HW:
    from PIL import Image
    hmap_img = Image.fromarray(hmap.astype(np.uint16) if hmap.max() > 255 else hmap.astype(np.uint8))
    hmap_img = hmap_img.resize((MAX_HW, MAX_HW), resample=Image.BILINEAR)
    hmap = np.array(hmap_img).astype(np.float32)
    H, W = hmap.shape

# Normalize to [0, 1]
hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap is flat (all same value).")
hmap01 = (hmap - hmin) / (hmax - hmin)

heightfieldData = hmap01.flatten().tolist()

# --- Read texture and pick terrain size to improve visibility ---
tex = imageio.imread(TEXTURE)
tex_w = tex.shape[1]

# TARGET: meters per pixel (smaller = more visible trash)
# If your trash is subtle in texture, aim for ~0.05 to 0.15 m/px
TARGET_M_PER_PX = 0.10
terrain_size_m = float(tex_w) * TARGET_M_PER_PX

# Cap to avoid insane sizes
terrain_size_m = max(80.0, min(terrain_size_m, 300.0))

desired_max_height = 12.0

sx = terrain_size_m / W
sy = terrain_size_m / H

# --- Create terrain ---
terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldTextureScaling=1.0,
    heightfieldData=heightfieldData,
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

terrain_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_shape)

# Texture
tex_id = p.loadTexture(TEXTURE)
p.changeVisualShape(terrain_id, -1, textureUniqueId=tex_id)

p.changeDynamics(terrain_id, -1, lateralFriction=0.9, restitution=0.0)

# Terrain is usually centered around origin in PyBullet heightfields
aabb_min, aabb_max = p.getAABB(terrain_id)
center = [(aabb_min[i] + aabb_max[i]) / 2 for i in range(3)]

print("Loaded terrain.")
print("Terrain AABB:", aabb_min, aabb_max)
print("Terrain center:", center)

print("Texture shape:", tex.shape)
print("Heightmap shape:", H, W)
print("Meters per texture pixel (approx):", terrain_size_m / tex_w, "m/px")

# --- Spawn a cube near center ---
cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5], rgbaColor=[1, 0, 0, 1])
cube_id  = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[center[0], center[1], center[2] + 20]
)

# --- Camera (birdseye defaults + optional toggle) ---
TOPDOWN_DIST = max(30.0, terrain_size_m * 0.55)  # tweak 0.45–0.75
birdseye = True

# Start camera on center (birdseye)
p.resetDebugVisualizerCamera(
    cameraDistance=TOPDOWN_DIST,
    cameraYaw=0,
    cameraPitch=-89,
    cameraTargetPosition=[center[0], center[1], 0]
)

print("Controls:")
print("Move cube: Arrow keys | Up/Down: R/F")
print("Spawn/remove trash objects: press T")
print("Toggle birdseye: press V")
print("Tip: If view looks like grid, press W once (wireframe toggle).")

# --- Trash spawning (RELIABLE FOR YOLO) ---
trash_ids = []
trash_on = False

def pressed(keys, k):
    return (k in keys) and (keys[k] & p.KEY_WAS_TRIGGERED)

def spawn_trash(n=60):
    """Spawn visible trash proxies sitting on terrain (good for YOLO)."""
    ids = []
    xmin, ymin = aabb_min[0] + 2.0, aabb_min[1] + 2.0
    xmax, ymax = aabb_max[0] - 2.0, aabb_max[1] - 2.0

    for _ in range(n):
        x = random.uniform(xmin, xmax)
        y = random.uniform(ymin, ymax)

        hit = p.rayTest([x, y, 200], [x, y, -200])[0]
        hit_obj, hit_pos = hit[0], hit[3]
        if hit_obj != terrain_id:
            continue

        z = hit_pos[2]

        # bottle (cyl) or bag (thin box)
        if random.random() < 0.6:
            r = random.uniform(0.04, 0.08)
            h = random.uniform(0.15, 0.28)
            col = p.createCollisionShape(p.GEOM_CYLINDER, radius=r, height=h)
            vis = p.createVisualShape(p.GEOM_CYLINDER, radius=r, length=h, rgbaColor=[0.1, 0.9, 0.1, 1])
            yaw = random.uniform(0, 3.14159)
            tid = p.createMultiBody(
                baseMass=0.05,
                baseCollisionShapeIndex=col,
                baseVisualShapeIndex=vis,
                basePosition=[x, y, z + h/2 + 0.01],
                baseOrientation=p.getQuaternionFromEuler([0, 0, yaw])
            )
        else:
            hx = random.uniform(0.08, 0.16)
            hy = random.uniform(0.08, 0.16)
            hz = random.uniform(0.01, 0.02)
            col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[hx, hy, hz])
            vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[hx, hy, hz], rgbaColor=[0.9, 0.1, 0.1, 1])
            yaw = random.uniform(0, 3.14159)
            tid = p.createMultiBody(
                baseMass=0.03,
                baseCollisionShapeIndex=col,
                baseVisualShapeIndex=vis,
                basePosition=[x, y, z + hz + 0.01],
                baseOrientation=p.getQuaternionFromEuler([0, 0, yaw])
            )
        ids.append(tid)

    return ids

def clear_trash():
    global trash_ids
    for tid in trash_ids:
        try:
            p.removeBody(tid)
        except:
            pass
    trash_ids = []

# --- Controls ---
speed = 0.8

def down(keys, k):
    return (k in keys) and (keys[k] & p.KEY_IS_DOWN)

while p.isConnected():
    keys = p.getKeyboardEvents()
    pos, orn = p.getBasePositionAndOrientation(cube_id)

    # Toggle birdseye
    if pressed(keys, ord('v')):
        birdseye = not birdseye
        print("Birdseye:", birdseye)

    # Toggle trash
    if pressed(keys, ord('t')):
        trash_on = not trash_on
        if trash_on:
            clear_trash()
            trash_ids = spawn_trash(n=80)
            print("Trash ON:", len(trash_ids))
        else:
            clear_trash()
            print("Trash OFF")

    # Move cube
    if down(keys, p.B3G_UP_ARROW):    pos = [pos[0] + speed, pos[1], pos[2]]
    if down(keys, p.B3G_DOWN_ARROW):  pos = [pos[0] - speed, pos[1], pos[2]]
    if down(keys, p.B3G_LEFT_ARROW):  pos = [pos[0], pos[1] + speed, pos[2]]
    if down(keys, p.B3G_RIGHT_ARROW): pos = [pos[0], pos[1] - speed, pos[2]]

    # Altitude
    if down(keys, ord('r')): pos = [pos[0], pos[1], pos[2] + speed]
    if down(keys, ord('f')): pos = [pos[0], pos[1], pos[2] - speed]

    p.resetBasePositionAndOrientation(cube_id, pos, orn)

    # --- Camera follow (birdseye or original) ---
    if birdseye:
        p.resetDebugVisualizerCamera(
            cameraDistance=TOPDOWN_DIST,
            cameraYaw=0,        # set to 90/180/270 if you want rotation
            cameraPitch=-89,    # straight down
            cameraTargetPosition=[pos[0], pos[1], 0]
        )
    else:
        p.resetDebugVisualizerCamera(
            cameraDistance=60,
            cameraYaw=35,
            cameraPitch=-35,
            cameraTargetPosition=pos
        )

    p.stepSimulation()
    time.sleep(1/240)

Loaded terrain.
Terrain AABB: (-45.91015625, -45.91015625, -6.0) (45.91015625, 45.91015625, 6.0)
Terrain center: [0.0, 0.0, 0.0]
Texture shape: (1000, 920, 3)
Heightmap shape: 512 512
Meters per texture pixel (approx): 0.1 m/px
Controls:
Move cube: Arrow keys | Up/Down: R/F
Spawn/remove trash objects: press T
Toggle birdseye: press V
Tip: If view looks like grid, press W once (wireframe toggle).


In [17]:
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os, random, math
import cv2

HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE), f"Missing {TEXTURE}"

# ---------------- SYNTHETIC WASTE URDF SETUP ----------------

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
URDF_DIR = SCRIPT_DIR

WASTE_URDFS = [
    {
        "name": "bottle",
        "path": os.path.join(URDF_DIR, "bottle.urdf"),
        "half_height": 0.085,
        "footprint": 0.04,
        "burial_bias": ["surface", "half_buried", "mostly_buried", "side_exposed"]
    },
    {
        "name": "cardboard_bag",
        "path": os.path.join(URDF_DIR, "cardboard_bag.urdf"),
        "half_height": 0.10,
        "footprint": 0.15,
        "burial_bias": ["surface", "half_buried", "top_only"]
    },
    {
        "name": "cardboard_box",
        "path": os.path.join(URDF_DIR, "cardboard_box.urdf"),
        "half_height": 0.09,
        "footprint": 0.18,
        "burial_bias": ["surface", "half_buried", "corner_peek", "top_only"]
    },
    {
        "name": "garbage_bag",
        "path": os.path.join(URDF_DIR, "garbage_bag.urdf"),
        "half_height": 0.18,
        "footprint": 0.18,
        "burial_bias": ["surface", "half_buried", "mostly_buried"]
    },
]

NO_DATA_RGB = np.array([168, 156, 142], dtype=np.float32)
NO_DATA_THRESH = 18.0
MIN_WASTE_GAP = 0.9
MAX_SPAWN_TRIES = 30

# ---------------- HELPERS ----------------

def pressed(keys, k):
    return (k in keys) and (keys[k] & p.KEY_WAS_TRIGGERED)

def down(keys, k):
    return (k in keys) and (keys[k] & p.KEY_IS_DOWN)

def world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h):
    u = (x - aabb_min[0]) / (aabb_max[0] - aabb_min[0])
    v = (y - aabb_min[1]) / (aabb_max[1] - aabb_min[1])

    px = int(np.clip(u * (tex_w - 1), 0, tex_w - 1))
    py = int(np.clip((1.0 - v) * (tex_h - 1), 0, tex_h - 1))
    return px, py

def texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
    tex_h, tex_w = tex_rgb.shape[:2]
    px, py = world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h)
    rgb = tex_rgb[py, px].astype(np.float32)

    if np.linalg.norm(rgb - NO_DATA_RGB) < NO_DATA_THRESH:
        return False

    x0 = max(0, px - 6)
    x1 = min(tex_w, px + 7)
    y0 = max(0, py - 6)
    y1 = min(tex_h, py + 7)

    patch = tex_rgb[y0:y1, x0:x1].astype(np.float32)
    if patch.std() < 4.0:
        return False

    return True

def terrain_hit_z(x, y, terrain_id):
    hit = p.rayTest([x, y, 200], [x, y, -200])[0]
    if hit[0] != terrain_id:
        return None
    return hit[3][2]

def far_from_existing(x, y, existing_xy, min_gap=MIN_WASTE_GAP):
    for ex, ey in existing_xy:
        dx = x - ex
        dy = y - ey
        if dx * dx + dy * dy < min_gap * min_gap:
            return False
    return True

def placement_pose(kind, mode, ground_z):
    yaw = random.uniform(-math.pi, math.pi)
    roll = 0.0
    pitch = 0.0
    hh = kind["half_height"]

    if mode == "surface":
        z = ground_z + hh + random.uniform(0.005, 0.02)

    elif mode == "half_buried":
        z = ground_z + hh * random.uniform(0.35, 0.60)
        if kind["name"] in ["bottle", "cardboard_box"]:
            roll = random.uniform(-0.45, 0.45)
            pitch = random.uniform(-0.45, 0.45)

    elif mode == "mostly_buried":
        z = ground_z + hh * random.uniform(0.08, 0.25)
        roll = random.uniform(-0.30, 0.30)
        pitch = random.uniform(-0.30, 0.30)

    elif mode == "top_only":
        z = ground_z + hh * random.uniform(0.05, 0.15)
        roll = random.uniform(-0.20, 0.20)
        pitch = random.uniform(-0.20, 0.20)

    elif mode == "corner_peek":
        z = ground_z + hh * random.uniform(0.15, 0.30)
        roll = random.choice([-1.0, 1.0]) * random.uniform(0.6, 1.1)
        pitch = random.choice([-1.0, 1.0]) * random.uniform(0.2, 0.7)

    elif mode == "side_exposed":
        z = ground_z + hh * random.uniform(0.15, 0.35)
        roll = random.choice([-1.0, 1.0]) * random.uniform(1.15, 1.45)
        pitch = random.uniform(-0.25, 0.25)

    else:
        z = ground_z + hh

    quat = p.getQuaternionFromEuler([roll, pitch, yaw])
    return z, quat

def spawn_synthetic_waste(
    terrain_id,
    aabb_min,
    aabb_max,
    tex_rgb,
    n=40,
    fixed_base=True,
    cluster_probability=0.35,
):
    ids = []
    existing_xy = []

    xmin, ymin = aabb_min[0] + 2.5, aabb_min[1] + 2.5
    xmax, ymax = aabb_max[0] - 2.5, aabb_max[1] - 2.5

    cluster_seed = None

    for _ in range(n):
        spawned = False

        for _try in range(MAX_SPAWN_TRIES):
            kind = random.choice(WASTE_URDFS)

            if cluster_seed is not None and random.random() < cluster_probability:
                x = cluster_seed[0] + random.uniform(-2.5, 2.5)
                y = cluster_seed[1] + random.uniform(-2.5, 2.5)
            else:
                x = random.uniform(xmin, xmax)
                y = random.uniform(ymin, ymax)

            if not (xmin <= x <= xmax and ymin <= y <= ymax):
                continue

            min_gap = max(MIN_WASTE_GAP, kind["footprint"] * 1.7)
            if not far_from_existing(x, y, existing_xy, min_gap=min_gap):
                continue

            if not texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
                continue

            ground_z = terrain_hit_z(x, y, terrain_id)
            if ground_z is None:
                continue

            mode = random.choice(kind["burial_bias"])
            z, quat = placement_pose(kind, mode, ground_z)

            try:
                body_id = p.loadURDF(
                    kind["path"],
                    basePosition=[x, y, z],
                    baseOrientation=quat,
                    useFixedBase=fixed_base,
                    globalScaling=random.uniform(0.85, 1.15),
                )
            except Exception as e:
                print("URDF load failed:", kind["path"], e)
                continue

            p.changeDynamics(body_id, -1, lateralFriction=0.9, restitution=0.0)
            ids.append(body_id)
            existing_xy.append((x, y))

            if cluster_seed is None or random.random() < 0.18:
                cluster_seed = (x, y)

            print(f"spawned {kind['name']} | mode={mode} | pos=({x:.2f}, {y:.2f}, {z:.2f})")
            spawned = True
            break

        if not spawned:
            print("could not place one waste item after many tries")

    return ids

# ---------------- RESET / CONNECT ----------------

try:
    if p.isConnected():
        p.disconnect()
except:
    pass

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)

p.configureDebugVisualizer(p.COV_ENABLE_WIREFRAME, 0)

# ---------------- LOAD HEIGHTMAP ----------------

hmap = imageio.imread(HEIGHTMAP)

if hmap.ndim == 3:
    hmap = hmap[..., 0]

hmap = hmap.astype(np.float32)
H, W = hmap.shape

MAX_HW = 512
if max(H, W) > MAX_HW:
    from PIL import Image
    hmap_img = Image.fromarray(
        hmap.astype(np.uint16) if hmap.max() > 255 else hmap.astype(np.uint8)
    )
    hmap_img = hmap_img.resize((MAX_HW, MAX_HW), resample=Image.BILINEAR)
    hmap = np.array(hmap_img).astype(np.float32)
    H, W = hmap.shape

hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap is flat (all same value).")

hmap01 = (hmap - hmin) / (hmax - hmin)
heightfieldData = hmap01.flatten().tolist()

# ---------------- TEXTURE ----------------

tex = imageio.imread(TEXTURE)
if tex.ndim == 2:
    tex = np.stack([tex, tex, tex], axis=-1)

tex_rgb = tex[:, :, :3]
tex_w = tex.shape[1]

TARGET_M_PER_PX = 0.10
terrain_size_m = float(tex_w) * TARGET_M_PER_PX
terrain_size_m = max(80.0, min(terrain_size_m, 300.0))

desired_max_height = 12.0
sx = terrain_size_m / W
sy = terrain_size_m / H

# ---------------- CREATE TERRAIN ----------------

terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldTextureScaling=1.0,
    heightfieldData=heightfieldData,
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

terrain_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_shape)

tex_id = p.loadTexture(TEXTURE)
p.changeVisualShape(terrain_id, -1, textureUniqueId=tex_id)
p.changeDynamics(terrain_id, -1, lateralFriction=0.9, restitution=0.0)

aabb_min, aabb_max = p.getAABB(terrain_id)
center = [(aabb_min[i] + aabb_max[i]) / 2 for i in range(3)]

print("Loaded terrain.")
print("Terrain AABB:", aabb_min, aabb_max)
print("Terrain center:", center)
print("Texture shape:", tex.shape)
print("Heightmap shape:", H, W)
print("Meters per texture pixel (approx):", terrain_size_m / tex_w, "m/px")

# ---------------- SPAWN WASTE AUTOMATICALLY ----------------

trash_ids = spawn_synthetic_waste(
    terrain_id=terrain_id,
    aabb_min=aabb_min,
    aabb_max=aabb_max,
    tex_rgb=tex_rgb,
    n=40,
    fixed_base=True,
    cluster_probability=0.35
)

print("Total synthetic waste spawned:", len(trash_ids))

# ---------------- TEMP DRONE / CAMERA TARGET ----------------

cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5])
cube_vis = p.createVisualShape(
    p.GEOM_BOX,
    halfExtents=[0.5, 0.5, 0.5],
    rgbaColor=[1, 0, 0, 1]
)

cube_id = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[center[0], center[1], center[2] + 20]
)

# ---------------- CAMERA SETTINGS ----------------

# more zoomed in than your old version
TOPDOWN_DIST = max(10.0, terrain_size_m * 0.18)

# render camera for frame saving
CAM_W = 960
CAM_H = 960
CAM_FOV = 55

p.resetDebugVisualizerCamera(
    cameraDistance=TOPDOWN_DIST,
    cameraYaw=0,
    cameraPitch=-89,
    cameraTargetPosition=[center[0], center[1], 0]
)

print("Controls:")
print("Arrow keys = move target")
print("R / F = move up / down")
print("C = capture 50 frames")
print("V = toggle view angle")

birdseye = True
speed = 0.8

# ---------------- FRAME CAPTURE ----------------

capture_dir = os.path.join(SCRIPT_DIR, "captured_frames")
os.makedirs(capture_dir, exist_ok=True)

capture_active = False
capture_target = 50
capture_count = 0
capture_interval = 8
capture_tick = 0

# ---------------- MAIN LOOP ----------------

while p.isConnected():
    keys = p.getKeyboardEvents()
    pos, orn = p.getBasePositionAndOrientation(cube_id)

    if pressed(keys, ord('v')):
        birdseye = not birdseye
        print("Birdseye:", birdseye)

    if pressed(keys, ord('c')):
        capture_active = True
        capture_count = 0
        capture_tick = 0
        print("Started capturing 50 frames...")

    if down(keys, p.B3G_UP_ARROW):
        pos = [pos[0] + speed, pos[1], pos[2]]
    if down(keys, p.B3G_DOWN_ARROW):
        pos = [pos[0] - speed, pos[1], pos[2]]
    if down(keys, p.B3G_LEFT_ARROW):
        pos = [pos[0], pos[1] + speed, pos[2]]
    if down(keys, p.B3G_RIGHT_ARROW):
        pos = [pos[0], pos[1] - speed, pos[2]]

    if down(keys, ord('r')):
        pos = [pos[0], pos[1], pos[2] + speed]
    if down(keys, ord('f')):
        pos = [pos[0], pos[1], pos[2] - speed]

    p.resetBasePositionAndOrientation(cube_id, pos, orn)

    if birdseye:
        p.resetDebugVisualizerCamera(
            cameraDistance=TOPDOWN_DIST,
            cameraYaw=0,
            cameraPitch=-89,
            cameraTargetPosition=[pos[0], pos[1], 0]
        )
    else:
        p.resetDebugVisualizerCamera(
            cameraDistance=max(8.0, TOPDOWN_DIST * 0.8),
            cameraYaw=35,
            cameraPitch=-65,
            cameraTargetPosition=pos
        )

    # render top-down image for dataset capture
    cam_eye = [pos[0], pos[1], pos[2] + 12]
    cam_target = [pos[0], pos[1], 0]

    view_matrix = p.computeViewMatrix(
        cameraEyePosition=cam_eye,
        cameraTargetPosition=cam_target,
        cameraUpVector=[0, 1, 0]
    )

    projection_matrix = p.computeProjectionMatrixFOV(
        fov=CAM_FOV,
        aspect=CAM_W / CAM_H,
        nearVal=0.1,
        farVal=100
    )

    img = p.getCameraImage(
        CAM_W,
        CAM_H,
        view_matrix,
        projection_matrix
    )

    rgb = np.reshape(img[2], (CAM_H, CAM_W, 4))
    rgb = rgb[:, :, :3]
    rgb_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

    if capture_active:
        capture_tick += 1

        if capture_tick % capture_interval == 0:
            out_path = os.path.join(capture_dir, f"frame_{capture_count:03d}.png")
            cv2.imwrite(out_path, rgb_bgr)
            print("saved:", out_path)
            capture_count += 1

        if capture_count >= capture_target:
            capture_active = False
            print("Finished capturing 50 frames.")

    p.stepSimulation()
    time.sleep(1 / 240)

Loaded terrain.
Terrain AABB: (-45.91015625, -45.91015625, -6.0) (45.91015625, 45.91015625, 6.0)
Terrain center: [0.0, 0.0, 0.0]
Texture shape: (1000, 920, 3)
Heightmap shape: 512 512
Meters per texture pixel (approx): 0.1 m/px
spawned bottle | mode=half_buried | pos=(-9.39, -16.49, 1.79)
spawned cardboard_bag | mode=half_buried | pos=(-42.81, 6.54, -5.41)
spawned cardboard_box | mode=surface | pos=(-17.13, -7.63, -1.30)
spawned cardboard_bag | mode=surface | pos=(-40.47, 7.18, -5.24)
spawned bottle | mode=surface | pos=(7.21, -22.41, 2.48)
spawned cardboard_bag | mode=top_only | pos=(-13.25, -22.98, 1.55)
spawned cardboard_box | mode=corner_peek | pos=(6.11, -23.96, 2.15)
spawned cardboard_box | mode=top_only | pos=(6.92, -24.50, 2.06)
spawned cardboard_box | mode=corner_peek | pos=(8.17, -23.06, 2.88)
spawned cardboard_box | mode=half_buried | pos=(-2.53, -37.23, 2.09)
spawned cardboard_bag | mode=half_buried | pos=(-18.41, -9.06, -1.38)
spawned bottle | mode=surface | pos=(-3.84, -3

TypeError: 'NoneType' object is not subscriptable

In [ ]:
# visible boxes and good bg real one 

import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os, random, math
import cv2

HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE), f"Missing {TEXTURE}"

# ---------------- CONFIG ----------------

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
URDF_DIR = SCRIPT_DIR

FRAME_COUNT = 50
OUTPUT_SIZE = 1024
CROP_SIZE_PX = 320          # larger than before = less zoomed in
MIN_CAPTURE_SPACING_M = 8.0 # reduced so we can always get 50
WASTE_COUNT = 55

NO_DATA_RGB = np.array([168, 156, 142], dtype=np.float32)
NO_DATA_THRESH = 18.0
MAX_SPAWN_TRIES = 50
MIN_WASTE_GAP = 1.0

# ---------------- WASTE DEFINITIONS ----------------

WASTE_URDFS = [
    {
        "name": "bottle",
        "path": os.path.join(URDF_DIR, "bottle.urdf"),
        "half_height": 0.085,
        "footprint": 0.08,
        "burial_bias": ["surface", "surface", "half_buried", "side_exposed"],
        "scale_range": (1.0, 1.25),
        "overlay_color": (40, 220, 220),   # BGR
    },
    {
        "name": "cardboard_bag",
        "path": os.path.join(URDF_DIR, "cardboard_bag.urdf"),
        "half_height": 0.10,
        "footprint": 0.22,
        "burial_bias": ["surface", "surface", "half_buried", "top_only"],
        "scale_range": (1.35, 1.75),
        "overlay_color": (70, 120, 190),
    },
    {
        "name": "cardboard_box",
        "path": os.path.join(URDF_DIR, "cardboard_box.urdf"),
        "half_height": 0.09,
        "footprint": 0.24,
        "burial_bias": ["surface", "surface", "half_buried", "corner_peek", "top_only"],
        "scale_range": (1.35, 1.75),
        "overlay_color": (90, 150, 210),
    },
    {
        "name": "garbage_bag",
        "path": os.path.join(URDF_DIR, "garbage_bag.urdf"),
        "half_height": 0.18,
        "footprint": 0.24,
        "burial_bias": ["surface", "surface", "half_buried"],
        "scale_range": (1.00, 1.30),
        "overlay_color": (20, 20, 20),
    },
]

# ---------------- HELPERS ----------------

def world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h):
    u = (x - aabb_min[0]) / (aabb_max[0] - aabb_min[0])
    v = (y - aabb_min[1]) / (aabb_max[1] - aabb_min[1])

    px = int(np.clip(u * (tex_w - 1), 0, tex_w - 1))
    py = int(np.clip((1.0 - v) * (tex_h - 1), 0, tex_h - 1))
    return px, py

def texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
    tex_h, tex_w = tex_rgb.shape[:2]
    px, py = world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h)
    rgb = tex_rgb[py, px].astype(np.float32)

    if np.linalg.norm(rgb - NO_DATA_RGB) < NO_DATA_THRESH:
        return False

    x0 = max(0, px - 8)
    x1 = min(tex_w, px + 9)
    y0 = max(0, py - 8)
    y1 = min(tex_h, py + 9)

    patch = tex_rgb[y0:y1, x0:x1].astype(np.float32)
    if patch.std() < 3.0:
        return False

    return True

def terrain_hit_z(x, y, terrain_id):
    hit = p.rayTest([x, y, 200], [x, y, -200])[0]
    if hit[0] != terrain_id:
        return None
    return hit[3][2]

def far_from_existing(x, y, existing_xy, min_gap):
    for ex, ey in existing_xy:
        dx = x - ex
        dy = y - ey
        if dx * dx + dy * dy < min_gap * min_gap:
            return False
    return True

def placement_pose(kind, mode, ground_z):
    yaw = random.uniform(-math.pi, math.pi)
    roll = 0.0
    pitch = 0.0
    hh = kind["half_height"]

    if mode == "surface":
        z = ground_z + hh + random.uniform(0.01, 0.03)

    elif mode == "half_buried":
        z = ground_z + hh * random.uniform(0.45, 0.70)
        roll = random.uniform(-0.20, 0.20)
        pitch = random.uniform(-0.20, 0.20)

    elif mode == "top_only":
        z = ground_z + hh * random.uniform(0.18, 0.28)
        roll = random.uniform(-0.12, 0.12)
        pitch = random.uniform(-0.12, 0.12)

    elif mode == "corner_peek":
        z = ground_z + hh * random.uniform(0.22, 0.35)
        roll = random.choice([-1.0, 1.0]) * random.uniform(0.55, 0.95)
        pitch = random.choice([-1.0, 1.0]) * random.uniform(0.18, 0.45)

    elif mode == "side_exposed":
        z = ground_z + hh * random.uniform(0.22, 0.38)
        roll = random.choice([-1.0, 1.0]) * random.uniform(1.05, 1.28)
        pitch = random.uniform(-0.18, 0.18)

    else:
        z = ground_z + hh

    quat = p.getQuaternionFromEuler([roll, pitch, yaw])
    return z, quat, yaw

def tint_all_links(body_id, rgba):
    p.changeVisualShape(body_id, -1, rgbaColor=rgba)
    for j in range(p.getNumJoints(body_id)):
        p.changeVisualShape(body_id, j, rgbaColor=rgba)

def crop_texture_patch(tex_rgb, cx, cy, crop_size_px, out_size):
    tex_h, tex_w = tex_rgb.shape[:2]
    half = crop_size_px // 2

    x0 = int(round(cx - half))
    y0 = int(round(cy - half))
    x1 = x0 + crop_size_px
    y1 = y0 + crop_size_px

    pad_left = max(0, -x0)
    pad_top = max(0, -y0)
    pad_right = max(0, x1 - tex_w)
    pad_bottom = max(0, y1 - tex_h)

    x0_clip = max(0, x0)
    y0_clip = max(0, y0)
    x1_clip = min(tex_w, x1)
    y1_clip = min(tex_h, y1)

    patch = tex_rgb[y0_clip:y1_clip, x0_clip:x1_clip].copy()

    if pad_left or pad_top or pad_right or pad_bottom:
        patch = cv2.copyMakeBorder(
            patch,
            pad_top, pad_bottom, pad_left, pad_right,
            borderType=cv2.BORDER_REFLECT
        )

    patch = cv2.resize(patch, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return patch

def draw_rotated_box(img, center, size, angle_deg, color, thickness=-1):
    rect = (center, size, angle_deg)
    pts = cv2.boxPoints(rect)
    pts = pts.astype(np.int32)
    cv2.drawContours(img, [pts], 0, color, thickness)

def overlay_waste_on_patch(patch_bgr, waste_items, center_px, crop_size_px, out_size):
    scale = out_size / crop_size_px
    half = crop_size_px / 2.0

    for item in waste_items:
        dx = item["tex_x"] - center_px[0]
        dy = item["tex_y"] - center_px[1]

        if abs(dx) > half or abs(dy) > half:
            continue

        px = int((dx + half) * scale)
        py = int((dy + half) * scale)

        color = item["overlay_color"]
        name = item["name"]
        size = item["draw_size_px"]
        angle = -math.degrees(item["yaw"])

        shadow_color = (0, 0, 0)
        shadow_offset = max(1, size // 8)

        if name == "bottle":
            cv2.ellipse(patch_bgr, (px + shadow_offset, py + shadow_offset), (size // 3, size), angle, 0, 360, shadow_color, -1)
            cv2.ellipse(patch_bgr, (px, py), (size // 3, size), angle, 0, 360, color, -1)
            cv2.circle(patch_bgr, (px, py - size // 2), max(2, size // 4), (220, 245, 245), -1)

        elif name == "garbage_bag":
            cv2.ellipse(patch_bgr, (px + shadow_offset, py + shadow_offset), (size, int(size * 0.75)), angle, 0, 360, shadow_color, -1)
            cv2.ellipse(patch_bgr, (px, py), (size, int(size * 0.75)), angle, 0, 360, color, -1)
            cv2.circle(patch_bgr, (px, py - int(size * 0.7)), max(2, size // 4), color, -1)

        elif name == "cardboard_box":
            draw_rotated_box(
                patch_bgr,
                (px + shadow_offset, py + shadow_offset),
                (int(size * 1.4), int(size * 1.1)),
                angle,
                shadow_color,
                -1
            )
            draw_rotated_box(
                patch_bgr,
                (px, py),
                (int(size * 1.4), int(size * 1.1)),
                angle,
                color,
                -1
            )
            draw_rotated_box(
                patch_bgr,
                (px, py),
                (int(size * 1.4), int(size * 1.1)),
                angle,
                (120, 170, 230),
                2
            )

        elif name == "cardboard_bag":
            pts = np.array([
                [px - size, py + size],
                [px - int(size * 0.7), py - size],
                [px + int(size * 0.7), py - size],
                [px + size, py + size],
            ], dtype=np.float32)

            theta = math.radians(angle)
            rot = np.array([[math.cos(theta), -math.sin(theta)],
                            [math.sin(theta),  math.cos(theta)]], dtype=np.float32)

            pts_rot = []
            for pt in pts:
                vec = pt - np.array([px, py], dtype=np.float32)
                vec = rot @ vec
                pts_rot.append((vec + np.array([px, py], dtype=np.float32)).astype(np.int32))
            pts_rot = np.array(pts_rot, dtype=np.int32)

            pts_shadow = pts_rot + np.array([shadow_offset, shadow_offset], dtype=np.int32)

            cv2.fillConvexPoly(patch_bgr, pts_shadow, shadow_color)
            cv2.fillConvexPoly(patch_bgr, pts_rot, color)
            cv2.line(patch_bgr, tuple(pts_rot[1]), tuple(pts_rot[2]), (120, 170, 230), 2)

    return patch_bgr

# ---------------- SAMPLE CAPTURE CENTERS ----------------

def sample_capture_positions(aabb_min, aabb_max, terrain_id, tex_rgb, frame_count, min_spacing=8.0):
    xmin, ymin = aabb_min[0] + 5.0, aabb_min[1] + 5.0
    xmax, ymax = aabb_max[0] - 5.0, aabb_max[1] - 5.0

    positions = []
    attempts = 0
    max_attempts = frame_count * 300

    while len(positions) < frame_count and attempts < max_attempts:
        attempts += 1

        x = random.uniform(xmin, xmax)
        y = random.uniform(ymin, ymax)

        if not texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
            continue

        z = terrain_hit_z(x, y, terrain_id)
        if z is None:
            continue

        too_close = False
        for px, py, _pz in positions:
            dx = x - px
            dy = y - py
            if dx * dx + dy * dy < min_spacing * min_spacing:
                too_close = True
                break

        if too_close:
            continue

        positions.append((x, y, z))

    if len(positions) < frame_count:
        print(f"warning: only found {len(positions)} spaced positions, relaxing spacing...")

    while len(positions) < frame_count and attempts < max_attempts * 2:
        attempts += 1

        x = random.uniform(xmin, xmax)
        y = random.uniform(ymin, ymax)

        if not texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
            continue

        z = terrain_hit_z(x, y, terrain_id)
        if z is None:
            continue

        positions.append((x, y, z))

    return positions[:frame_count]

# ---------------- RESET / CONNECT ----------------

try:
    if p.isConnected():
        p.disconnect()
except:
    pass

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)
p.configureDebugVisualizer(p.COV_ENABLE_WIREFRAME, 0)

# ---------------- LOAD HEIGHTMAP ----------------

hmap = imageio.imread(HEIGHTMAP)

if hmap.ndim == 3:
    hmap = hmap[..., 0]

hmap = hmap.astype(np.float32)
H, W = hmap.shape

MAX_HW = 512
if max(H, W) > MAX_HW:
    from PIL import Image
    hmap_img = Image.fromarray(hmap.astype(np.uint16) if hmap.max() > 255 else hmap.astype(np.uint8))
    hmap_img = hmap_img.resize((MAX_HW, MAX_HW), resample=Image.BILINEAR)
    hmap = np.array(hmap_img).astype(np.float32)
    H, W = hmap.shape

hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap is flat (all same value).")

hmap01 = (hmap - hmin) / (hmax - hmin)
heightfieldData = hmap01.flatten().tolist()

# ---------------- LOAD TEXTURE ----------------

tex = imageio.imread(TEXTURE)
if tex.ndim == 2:
    tex = np.stack([tex, tex, tex], axis=-1)

tex_rgb = tex[:, :, :3]
tex_h, tex_w = tex_rgb.shape[:2]

TARGET_M_PER_PX = 0.10
terrain_size_m = float(tex_w) * TARGET_M_PER_PX
terrain_size_m = max(80.0, min(terrain_size_m, 300.0))

desired_max_height = 12.0
sx = terrain_size_m / W
sy = terrain_size_m / H

# ---------------- CREATE TERRAIN ----------------

terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldTextureScaling=1.0,
    heightfieldData=heightfieldData,
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

terrain_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_shape)
tex_id = p.loadTexture(TEXTURE)
p.changeVisualShape(terrain_id, -1, textureUniqueId=tex_id)
p.changeDynamics(terrain_id, -1, lateralFriction=0.9, restitution=0.0)

aabb_min, aabb_max = p.getAABB(terrain_id)
center = [(aabb_min[i] + aabb_max[i]) / 2 for i in range(3)]

print("Loaded terrain.")
print("Terrain AABB:", aabb_min, aabb_max)
print("Terrain center:", center)

# ---------------- SPAWN WASTE IN WORLD + STORE METADATA ----------------

waste_items = []
existing_xy = []

xmin, ymin = aabb_min[0] + 2.5, aabb_min[1] + 2.5
xmax, ymax = aabb_max[0] - 2.5, aabb_max[1] - 2.5

cluster_seed = None

for _ in range(WASTE_COUNT):
    spawned = False

    for _try in range(MAX_SPAWN_TRIES):
        kind = random.choice(WASTE_URDFS)

        if cluster_seed is not None and random.random() < 0.30:
            x = cluster_seed[0] + random.uniform(-2.5, 2.5)
            y = cluster_seed[1] + random.uniform(-2.5, 2.5)
        else:
            x = random.uniform(xmin, xmax)
            y = random.uniform(ymin, ymax)

        if not (xmin <= x <= xmax and ymin <= y <= ymax):
            continue

        min_gap = max(MIN_WASTE_GAP, kind["footprint"] * 1.6)
        if not far_from_existing(x, y, existing_xy, min_gap):
            continue

        if not texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
            continue

        ground_z = terrain_hit_z(x, y, terrain_id)
        if ground_z is None:
            continue

        mode = random.choice(kind["burial_bias"])
        z, quat, yaw = placement_pose(kind, mode, ground_z)
        scale = random.uniform(kind["scale_range"][0], kind["scale_range"][1])

        try:
            body_id = p.loadURDF(
                kind["path"],
                basePosition=[x, y, z],
                baseOrientation=quat,
                useFixedBase=True,
                globalScaling=scale,
            )
            tint_all_links(body_id, [1, 1, 1, 0.0])  # hide pybullet object from the visual output
        except Exception as e:
            print("URDF load failed:", kind["path"], e)
            continue

        tex_x, tex_y = world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h)

        if kind["name"] == "bottle":
            draw_size = int(random.uniform(8, 12))
        elif kind["name"] == "garbage_bag":
            draw_size = int(random.uniform(10, 16))
        else:
            draw_size = int(random.uniform(12, 18))

        waste_items.append({
            "name": kind["name"],
            "world_x": x,
            "world_y": y,
            "tex_x": tex_x,
            "tex_y": tex_y,
            "yaw": yaw,
            "overlay_color": kind["overlay_color"],
            "draw_size_px": draw_size,
        })

        existing_xy.append((x, y))
        if cluster_seed is None or random.random() < 0.18:
            cluster_seed = (x, y)

        spawned = True
        break

    if not spawned:
        print("could not place one waste item")

print("Total waste items placed:", len(waste_items))

# ---------------- AUTO CAPTURE FROM TEXTURE ----------------

capture_dir = os.path.join(SCRIPT_DIR, "captured_frames")
os.makedirs(capture_dir, exist_ok=True)

positions = sample_capture_positions(
    aabb_min=aabb_min,
    aabb_max=aabb_max,
    terrain_id=terrain_id,
    tex_rgb=tex_rgb,
    frame_count=FRAME_COUNT,
    min_spacing=MIN_CAPTURE_SPACING_M
)

print("Capture positions selected:", len(positions))

for i, (x, y, z) in enumerate(positions):
    cx, cy = world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h)

    patch_rgb = crop_texture_patch(
        tex_rgb=tex_rgb,
        cx=cx,
        cy=cy,
        crop_size_px=CROP_SIZE_PX,
        out_size=OUTPUT_SIZE
    )

    patch_bgr = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2BGR)

    patch_bgr = overlay_waste_on_patch(
        patch_bgr=patch_bgr,
        waste_items=waste_items,
        center_px=(cx, cy),
        crop_size_px=CROP_SIZE_PX,
        out_size=OUTPUT_SIZE
    )

    out_path = os.path.join(capture_dir, f"frame_{i:03d}.png")
    cv2.imwrite(out_path, patch_bgr) 
    print("saved:", out_path)

print(f"Finished saving {len(positions)} frames.")

# ---------------- OPTIONAL GUI VIEW ----------------

cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5], rgbaColor=[1, 0, 0, 1])
cube_id  = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[center[0], center[1], center[2] + 20]
)

p.resetDebugVisualizerCamera(
    cameraDistance=max(15.0, terrain_size_m * 0.20),
    cameraYaw=0,
    cameraPitch=-89,
    cameraTargetPosition=[center[0], center[1], 0]
)

while p.isConnected():
    p.stepSimulation()
    time.sleep(1 / 240)

In [20]:
# visible boxes and good bg

import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os, random, math
import cv2

HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE), f"Missing {TEXTURE}"

# ---------------- CONFIG ----------------

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
URDF_DIR = SCRIPT_DIR

FRAME_COUNT = 50
OUTPUT_SIZE = 1024
CROP_SIZE_PX = 320          # larger than before = less zoomed in
MIN_CAPTURE_SPACING_M = 8.0 # reduced so we can always get 50
WASTE_COUNT = 55

NO_DATA_RGB = np.array([168, 156, 142], dtype=np.float32)
NO_DATA_THRESH = 18.0
MAX_SPAWN_TRIES = 50
MIN_WASTE_GAP = 1.0

# ---------------- WASTE DEFINITIONS ----------------

WASTE_URDFS = [
    {
        "name": "bottle",
        "path": os.path.join(URDF_DIR, "bottle.urdf"),
        "half_height": 0.085,
        "footprint": 0.04,
        "burial_bias": ["surface", "half_buried", "mostly_buried", "side_exposed"],
        "scale_range": (0.85, 1.15),
        "overlay_color": (40, 220, 220),   # BGR
    },
    {
        "name": "cardboard_bag",
        "path": os.path.join(URDF_DIR, "cardboard_bag.urdf"),
        "half_height": 0.10,
        "footprint": 0.22,
        "burial_bias": ["surface", "surface", "half_buried", "top_only"],
        "scale_range": (1.35, 1.75),
        "overlay_color": (70, 120, 190),
    },
    {
        "name": "cardboard_box",
        "path": os.path.join(URDF_DIR, "cardboard_box.urdf"),
        "half_height": 0.09,
        "footprint": 0.24,
        "burial_bias": ["surface", "surface", "half_buried", "corner_peek", "top_only"],
        "scale_range": (1.35, 1.75),
        "overlay_color": (90, 150, 210),
    },
    {
        "name": "garbage_bag",
        "path": os.path.join(URDF_DIR, "garbage_bag.urdf"),
        "half_height": 0.18,
        "footprint": 0.24,
        "burial_bias": ["surface", "surface", "half_buried"],
        "scale_range": (1.00, 1.30),
        "overlay_color": (20, 20, 20),
    },
]

# ---------------- HELPERS ----------------

def world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h):
    u = (x - aabb_min[0]) / (aabb_max[0] - aabb_min[0])
    v = (y - aabb_min[1]) / (aabb_max[1] - aabb_min[1])

    px = int(np.clip(u * (tex_w - 1), 0, tex_w - 1))
    py = int(np.clip((1.0 - v) * (tex_h - 1), 0, tex_h - 1))
    return px, py

def texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
    tex_h, tex_w = tex_rgb.shape[:2]
    px, py = world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h)
    rgb = tex_rgb[py, px].astype(np.float32)

    if np.linalg.norm(rgb - NO_DATA_RGB) < NO_DATA_THRESH:
        return False

    x0 = max(0, px - 8)
    x1 = min(tex_w, px + 9)
    y0 = max(0, py - 8)
    y1 = min(tex_h, py + 9)

    patch = tex_rgb[y0:y1, x0:x1].astype(np.float32)
    if patch.std() < 3.0:
        return False

    return True

def terrain_hit_z(x, y, terrain_id):
    hit = p.rayTest([x, y, 200], [x, y, -200])[0]
    if hit[0] != terrain_id:
        return None
    return hit[3][2]

def far_from_existing(x, y, existing_xy, min_gap):
    for ex, ey in existing_xy:
        dx = x - ex
        dy = y - ey
        if dx * dx + dy * dy < min_gap * min_gap:
            return False
    return True

def placement_pose(kind, mode, ground_z):
    yaw = random.uniform(-math.pi, math.pi)
    roll = 0.0
    pitch = 0.0
    hh = kind["half_height"]

    if mode == "surface":
        z = ground_z + hh + random.uniform(0.005, 0.02)

    elif mode == "half_buried":
        z = ground_z + hh * random.uniform(0.35, 0.60)
        if kind["name"] in ["bottle", "cardboard_box"]:
            roll = random.uniform(-0.45, 0.45)
            pitch = random.uniform(-0.45, 0.45)
        else:
            roll = random.uniform(-0.20, 0.20)
            pitch = random.uniform(-0.20, 0.20)

    elif mode == "mostly_buried":
        z = ground_z + hh * random.uniform(0.08, 0.25)
        roll = random.uniform(-0.30, 0.30)
        pitch = random.uniform(-0.30, 0.30)

    elif mode == "top_only":
        z = ground_z + hh * random.uniform(0.05, 0.15)
        roll = random.uniform(-0.20, 0.20)
        pitch = random.uniform(-0.20, 0.20)

    elif mode == "corner_peek":
        z = ground_z + hh * random.uniform(0.15, 0.30)
        roll = random.choice([-1.0, 1.0]) * random.uniform(0.6, 1.1)
        pitch = random.choice([-1.0, 1.0]) * random.uniform(0.2, 0.7)

    elif mode == "side_exposed":
        z = ground_z + hh * random.uniform(0.15, 0.35)
        roll = random.choice([-1.0, 1.0]) * random.uniform(1.15, 1.45)
        pitch = random.uniform(-0.25, 0.25)

    else:
        z = ground_z + hh

    quat = p.getQuaternionFromEuler([roll, pitch, yaw])
    return z, quat, yaw

def tint_all_links(body_id, rgba):
    p.changeVisualShape(body_id, -1, rgbaColor=rgba)
    for j in range(p.getNumJoints(body_id)):
        p.changeVisualShape(body_id, j, rgbaColor=rgba)

def crop_texture_patch(tex_rgb, cx, cy, crop_size_px, out_size):
    tex_h, tex_w = tex_rgb.shape[:2]
    half = crop_size_px // 2

    x0 = int(round(cx - half))
    y0 = int(round(cy - half))
    x1 = x0 + crop_size_px
    y1 = y0 + crop_size_px

    pad_left = max(0, -x0)
    pad_top = max(0, -y0)
    pad_right = max(0, x1 - tex_w)
    pad_bottom = max(0, y1 - tex_h)

    x0_clip = max(0, x0)
    y0_clip = max(0, y0)
    x1_clip = min(tex_w, x1)
    y1_clip = min(tex_h, y1)

    patch = tex_rgb[y0_clip:y1_clip, x0_clip:x1_clip].copy()

    if pad_left or pad_top or pad_right or pad_bottom:
        patch = cv2.copyMakeBorder(
            patch,
            pad_top, pad_bottom, pad_left, pad_right,
            borderType=cv2.BORDER_REFLECT
        )

    patch = cv2.resize(patch, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return patch

def draw_rotated_box(img, center, size, angle_deg, color, thickness=-1):
    rect = (center, size, angle_deg)
    pts = cv2.boxPoints(rect)
    pts = pts.astype(np.int32)
    cv2.drawContours(img, [pts], 0, color, thickness)

def overlay_waste_on_patch(patch_bgr, waste_items, center_px, crop_size_px, out_size):
    scale = out_size / crop_size_px
    half = crop_size_px / 2.0

    for item in waste_items:
        dx = item["tex_x"] - center_px[0]
        dy = item["tex_y"] - center_px[1]

        if abs(dx) > half or abs(dy) > half:
            continue

        px = int((dx + half) * scale)
        py = int((dy + half) * scale)

        color = item["overlay_color"]
        name = item["name"]
        size = item["draw_size_px"]
        angle = -math.degrees(item["yaw"])

        shadow_color = (0, 0, 0)
        shadow_offset = max(1, size // 8)

        if name == "bottle":
            cv2.ellipse(patch_bgr, (px + shadow_offset, py + shadow_offset), (size // 3, size), angle, 0, 360, shadow_color, -1)
            cv2.ellipse(patch_bgr, (px, py), (size // 3, size), angle, 0, 360, color, -1)
            cv2.circle(patch_bgr, (px, py - size // 2), max(2, size // 4), (220, 245, 245), -1)

        elif name == "garbage_bag":
            cv2.ellipse(patch_bgr, (px + shadow_offset, py + shadow_offset), (size, int(size * 0.75)), angle, 0, 360, shadow_color, -1)
            cv2.ellipse(patch_bgr, (px, py), (size, int(size * 0.75)), angle, 0, 360, color, -1)
            cv2.circle(patch_bgr, (px, py - int(size * 0.7)), max(2, size // 4), color, -1)

        elif name == "cardboard_box":
            draw_rotated_box(
                patch_bgr,
                (px + shadow_offset, py + shadow_offset),
                (int(size * 1.4), int(size * 1.1)),
                angle,
                shadow_color,
                -1
            )
            draw_rotated_box(
                patch_bgr,
                (px, py),
                (int(size * 1.4), int(size * 1.1)),
                angle,
                color,
                -1
            )
            draw_rotated_box(
                patch_bgr,
                (px, py),
                (int(size * 1.4), int(size * 1.1)),
                angle,
                (120, 170, 230),
                2
            )

        elif name == "cardboard_bag":
            pts = np.array([
                [px - size, py + size],
                [px - int(size * 0.7), py - size],
                [px + int(size * 0.7), py - size],
                [px + size, py + size],
            ], dtype=np.float32)

            theta = math.radians(angle)
            rot = np.array([[math.cos(theta), -math.sin(theta)],
                            [math.sin(theta),  math.cos(theta)]], dtype=np.float32)

            pts_rot = []
            for pt in pts:
                vec = pt - np.array([px, py], dtype=np.float32)
                vec = rot @ vec
                pts_rot.append((vec + np.array([px, py], dtype=np.float32)).astype(np.int32))
            pts_rot = np.array(pts_rot, dtype=np.int32)

            pts_shadow = pts_rot + np.array([shadow_offset, shadow_offset], dtype=np.int32)

            cv2.fillConvexPoly(patch_bgr, pts_shadow, shadow_color)
            cv2.fillConvexPoly(patch_bgr, pts_rot, color)
            cv2.line(patch_bgr, tuple(pts_rot[1]), tuple(pts_rot[2]), (120, 170, 230), 2)

    return patch_bgr

# ---------------- SAMPLE CAPTURE CENTERS ----------------

def sample_capture_positions(aabb_min, aabb_max, terrain_id, tex_rgb, frame_count, min_spacing=8.0):
    xmin, ymin = aabb_min[0] + 5.0, aabb_min[1] + 5.0
    xmax, ymax = aabb_max[0] - 5.0, aabb_max[1] - 5.0

    positions = []
    attempts = 0
    max_attempts = frame_count * 300

    while len(positions) < frame_count and attempts < max_attempts:
        attempts += 1

        x = random.uniform(xmin, xmax)
        y = random.uniform(ymin, ymax)

        if not texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
            continue

        z = terrain_hit_z(x, y, terrain_id)
        if z is None:
            continue

        too_close = False
        for px, py, _pz in positions:
            dx = x - px
            dy = y - py
            if dx * dx + dy * dy < min_spacing * min_spacing:
                too_close = True
                break

        if too_close:
            continue

        positions.append((x, y, z))

    if len(positions) < frame_count:
        print(f"warning: only found {len(positions)} spaced positions, relaxing spacing...")

    while len(positions) < frame_count and attempts < max_attempts * 2:
        attempts += 1

        x = random.uniform(xmin, xmax)
        y = random.uniform(ymin, ymax)

        if not texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
            continue

        z = terrain_hit_z(x, y, terrain_id)
        if z is None:
            continue

        positions.append((x, y, z))

    return positions[:frame_count]

# ---------------- RESET / CONNECT ----------------

try:
    if p.isConnected():
        p.disconnect()
except:
    pass

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)
p.configureDebugVisualizer(p.COV_ENABLE_WIREFRAME, 0)

# ---------------- LOAD HEIGHTMAP ----------------

hmap = imageio.imread(HEIGHTMAP)

if hmap.ndim == 3:
    hmap = hmap[..., 0]

hmap = hmap.astype(np.float32)
H, W = hmap.shape

MAX_HW = 512
if max(H, W) > MAX_HW:
    from PIL import Image
    hmap_img = Image.fromarray(hmap.astype(np.uint16) if hmap.max() > 255 else hmap.astype(np.uint8))
    hmap_img = hmap_img.resize((MAX_HW, MAX_HW), resample=Image.BILINEAR)
    hmap = np.array(hmap_img).astype(np.float32)
    H, W = hmap.shape

hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap is flat (all same value).")

hmap01 = (hmap - hmin) / (hmax - hmin)
heightfieldData = hmap01.flatten().tolist()

# ---------------- LOAD TEXTURE ----------------

tex = imageio.imread(TEXTURE)
if tex.ndim == 2:
    tex = np.stack([tex, tex, tex], axis=-1)

tex_rgb = tex[:, :, :3]
tex_h, tex_w = tex_rgb.shape[:2]

TARGET_M_PER_PX = 0.10
terrain_size_m = float(tex_w) * TARGET_M_PER_PX
terrain_size_m = max(80.0, min(terrain_size_m, 300.0))

desired_max_height = 12.0
sx = terrain_size_m / W
sy = terrain_size_m / H

# ---------------- CREATE TERRAIN ----------------

terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldTextureScaling=1.0,
    heightfieldData=heightfieldData,
    numHeightfieldRows=H,
    numHeightfieldColumns=W
)

terrain_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_shape)
tex_id = p.loadTexture(TEXTURE)
p.changeVisualShape(terrain_id, -1, textureUniqueId=tex_id)
p.changeDynamics(terrain_id, -1, lateralFriction=0.9, restitution=0.0)

aabb_min, aabb_max = p.getAABB(terrain_id)
center = [(aabb_min[i] + aabb_max[i]) / 2 for i in range(3)]

print("Loaded terrain.")
print("Terrain AABB:", aabb_min, aabb_max)
print("Terrain center:", center)

# ---------------- SPAWN WASTE IN WORLD + STORE METADATA ----------------

waste_items = []
existing_xy = []

xmin, ymin = aabb_min[0] + 2.5, aabb_min[1] + 2.5
xmax, ymax = aabb_max[0] - 2.5, aabb_max[1] - 2.5

cluster_seed = None

for _ in range(WASTE_COUNT):
    spawned = False

    for _try in range(MAX_SPAWN_TRIES):
        kind = random.choice(WASTE_URDFS)

        if cluster_seed is not None and random.random() < 0.30:
            x = cluster_seed[0] + random.uniform(-2.5, 2.5)
            y = cluster_seed[1] + random.uniform(-2.5, 2.5)
        else:
            x = random.uniform(xmin, xmax)
            y = random.uniform(ymin, ymax)

        if not (xmin <= x <= xmax and ymin <= y <= ymax):
            continue

        min_gap = max(MIN_WASTE_GAP, kind["footprint"] * 1.6)
        if not far_from_existing(x, y, existing_xy, min_gap):
            continue

        if not texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
            continue

        ground_z = terrain_hit_z(x, y, terrain_id)
        if ground_z is None:
            continue

        mode = random.choice(kind["burial_bias"])
        z, quat, yaw = placement_pose(kind, mode, ground_z)
        scale = random.uniform(kind["scale_range"][0], kind["scale_range"][1])

        try:
            body_id = p.loadURDF(
                kind["path"],
                basePosition=[x, y, z],
                baseOrientation=quat,
                useFixedBase=True,
                globalScaling=scale,
            )
            tint_all_links(body_id, [1, 1, 1, 0.0])  # hide pybullet object from the visual output
        except Exception as e:
            print("URDF load failed:", kind["path"], e)
            continue

        tex_x, tex_y = world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h)

        if kind["name"] == "bottle":
            draw_size = int(random.uniform(12, 18))
        elif kind["name"] == "garbage_bag":
            draw_size = int(random.uniform(10, 16))
        else:
            draw_size = int(random.uniform(12, 18))

        waste_items.append({
            "name": kind["name"],
            "world_x": x,
            "world_y": y,
            "tex_x": tex_x,
            "tex_y": tex_y,
            "yaw": yaw,
            "overlay_color": kind["overlay_color"],
            "draw_size_px": draw_size,
        })

        existing_xy.append((x, y))
        if cluster_seed is None or random.random() < 0.18:
            cluster_seed = (x, y)

        spawned = True
        break

    if not spawned:
        print("could not place one waste item")

print("Total waste items placed:", len(waste_items))

# ---------------- AUTO CAPTURE FROM TEXTURE ----------------

capture_dir = os.path.join(SCRIPT_DIR, "captured_frames")
os.makedirs(capture_dir, exist_ok=True)

positions = sample_capture_positions(
    aabb_min=aabb_min,
    aabb_max=aabb_max,
    terrain_id=terrain_id,
    tex_rgb=tex_rgb,
    frame_count=FRAME_COUNT,
    min_spacing=MIN_CAPTURE_SPACING_M
)

print("Capture positions selected:", len(positions))

for i, (x, y, z) in enumerate(positions):
    cx, cy = world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h)

    patch_rgb = crop_texture_patch(
        tex_rgb=tex_rgb,
        cx=cx,
        cy=cy,
        crop_size_px=CROP_SIZE_PX,
        out_size=OUTPUT_SIZE
    )

    patch_bgr = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2BGR)

    patch_bgr = overlay_waste_on_patch(
        patch_bgr=patch_bgr,
        waste_items=waste_items,
        center_px=(cx, cy),
        crop_size_px=CROP_SIZE_PX,
        out_size=OUTPUT_SIZE
    )

    out_path = os.path.join(capture_dir, f"frame_{i:03d}.png")
    cv2.imwrite(out_path, patch_bgr)
    print("saved:", out_path)

print(f"Finished saving {len(positions)} frames.")

# ---------------- OPTIONAL GUI VIEW ----------------

cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5], rgbaColor=[1, 0, 0, 1])
cube_id  = p.createMultiBody(
    baseMass=1,
    baseCollisionShapeIndex=cube_col,
    baseVisualShapeIndex=cube_vis,
    basePosition=[center[0], center[1], center[2] + 20]
)

p.resetDebugVisualizerCamera(
    cameraDistance=max(15.0, terrain_size_m * 0.20),
    cameraYaw=0,
    cameraPitch=-89,
    cameraTargetPosition=[center[0], center[1], 0]
)

while p.isConnected():
    p.stepSimulation()
    time.sleep(1 / 240)

Loaded terrain.
Terrain AABB: (-45.91015625, -45.91015625, -6.0) (45.91015625, 45.91015625, 6.0)
Terrain center: [0.0, 0.0, 0.0]
Total waste items placed: 55
Capture positions selected: 50
saved: d:\UNI\capstone\project_terrain\pybullet\captured_frames\frame_000.png
saved: d:\UNI\capstone\project_terrain\pybullet\captured_frames\frame_001.png
saved: d:\UNI\capstone\project_terrain\pybullet\captured_frames\frame_002.png
saved: d:\UNI\capstone\project_terrain\pybullet\captured_frames\frame_003.png
saved: d:\UNI\capstone\project_terrain\pybullet\captured_frames\frame_004.png
saved: d:\UNI\capstone\project_terrain\pybullet\captured_frames\frame_005.png
saved: d:\UNI\capstone\project_terrain\pybullet\captured_frames\frame_006.png
saved: d:\UNI\capstone\project_terrain\pybullet\captured_frames\frame_007.png
saved: d:\UNI\capstone\project_terrain\pybullet\captured_frames\frame_008.png
saved: d:\UNI\capstone\project_terrain\pybullet\captured_frames\frame_009.png
saved: d:\UNI\capstone\project_

KeyboardInterrupt: 

In [26]:
# synthetic_waste_capture.py  —  updated main script
# Changes vs original:
#   1. Stores burial_mode + obj_scale in waste_items dict
#   2. Passes terrain_size_m + tex_w to GSD calculation
#   3. Calls new overlay_waste_on_patch from overlay_waste.py
#   4. Exports YOLO-format labels alongside each frame
#   5. Draw size is now derived from real geometry, not hardcoded 10-18px
#   6. Minimum sizes enforced so every item is visible

import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os, random, math
import cv2
from overlay_waste import overlay_waste_on_patch   # <-- new module

HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE),   f"Missing {TEXTURE}"

# ── CONFIG ───────────────────────────────────────────────────────────────────

SCRIPT_DIR  = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
URDF_DIR    = SCRIPT_DIR

FRAME_COUNT             = 50
OUTPUT_SIZE             = 1024
CROP_SIZE_PX            = 320
MIN_CAPTURE_SPACING_M   = 8.0
WASTE_COUNT             = 55

NO_DATA_RGB    = np.array([168, 156, 142], dtype=np.float32)
NO_DATA_THRESH = 18.0
MAX_SPAWN_TRIES = 50
MIN_WASTE_GAP   = 1.0

# ── WASTE DEFINITIONS ────────────────────────────────────────────────────────

WASTE_URDFS = [
    {
        "name":         "bottle",
        "path":         os.path.join(URDF_DIR, "bottle.urdf"),
        "half_height":  0.085,
        "footprint":    0.04,
        "burial_bias":  ["surface", "half_buried", "mostly_buried", "side_exposed"],
        "scale_range":  (0.85, 1.15),
    },
    {
        "name":         "cardboard_bag",
        "path":         os.path.join(URDF_DIR, "cardboard_bag.urdf"),
        "half_height":  0.10,
        "footprint":    0.22,
        "burial_bias":  ["surface", "surface", "half_buried", "top_only"],
        "scale_range":  (1.35, 1.75),
    },
    {
        "name":         "cardboard_box",
        "path":         os.path.join(URDF_DIR, "cardboard_box.urdf"),
        "half_height":  0.09,
        "footprint":    0.24,
        "burial_bias":  ["surface", "surface", "half_buried", "corner_peek", "top_only"],
        "scale_range":  (1.35, 1.75),
    },
    {
        "name":         "garbage_bag",
        "path":         os.path.join(URDF_DIR, "garbage_bag.urdf"),
        "half_height":  0.18,
        "footprint":    0.24,
        "burial_bias":  ["surface", "surface", "half_buried"],
        "scale_range":  (1.00, 1.30),
    },
]

# Class indices for YOLO label export
CLASS_IDS = {
    "bottle":       0,
    "cardboard_bag": 1,
    "cardboard_box": 2,
    "garbage_bag":  3,
}

# ── HELPERS (unchanged from original) ────────────────────────────────────────

def world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h):
    u = (x - aabb_min[0]) / (aabb_max[0] - aabb_min[0])
    v = (y - aabb_min[1]) / (aabb_max[1] - aabb_min[1])
    px = int(np.clip(u * (tex_w - 1), 0, tex_w - 1))
    py = int(np.clip((1.0 - v) * (tex_h - 1), 0, tex_h - 1))
    return px, py

def texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
    tex_h, tex_w = tex_rgb.shape[:2]
    px, py = world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h)
    rgb = tex_rgb[py, px].astype(np.float32)
    if np.linalg.norm(rgb - NO_DATA_RGB) < NO_DATA_THRESH:
        return False
    x0 = max(0, px - 8); x1 = min(tex_w, px + 9)
    y0 = max(0, py - 8); y1 = min(tex_h, py + 9)
    patch = tex_rgb[y0:y1, x0:x1].astype(np.float32)
    if patch.std() < 3.0:
        return False
    return True

def terrain_hit_z(x, y, terrain_id):
    hit = p.rayTest([x, y, 200], [x, y, -200])[0]
    if hit[0] != terrain_id:
        return None
    return hit[3][2]

def far_from_existing(x, y, existing_xy, min_gap):
    for ex, ey in existing_xy:
        if (x - ex) ** 2 + (y - ey) ** 2 < min_gap ** 2:
            return False
    return True

def placement_pose(kind, mode, ground_z):
    yaw   = random.uniform(-math.pi, math.pi)
    roll  = 0.0
    pitch = 0.0
    hh    = kind["half_height"]

    if mode == "surface":
        z = ground_z + hh + random.uniform(0.005, 0.02)
    elif mode == "half_buried":
        z = ground_z + hh * random.uniform(0.35, 0.60)
        if kind["name"] in ["bottle", "cardboard_box"]:
            roll  = random.uniform(-0.45, 0.45)
            pitch = random.uniform(-0.45, 0.45)
        else:
            roll  = random.uniform(-0.20, 0.20)
            pitch = random.uniform(-0.20, 0.20)
    elif mode == "mostly_buried":
        z     = ground_z + hh * random.uniform(0.08, 0.25)
        roll  = random.uniform(-0.30, 0.30)
        pitch = random.uniform(-0.30, 0.30)
    elif mode == "top_only":
        z     = ground_z + hh * random.uniform(0.05, 0.15)
        roll  = random.uniform(-0.20, 0.20)
        pitch = random.uniform(-0.20, 0.20)
    elif mode == "corner_peek":
        z     = ground_z + hh * random.uniform(0.15, 0.30)
        roll  = random.choice([-1.0, 1.0]) * random.uniform(0.6, 1.1)
        pitch = random.choice([-1.0, 1.0]) * random.uniform(0.2, 0.7)
    elif mode == "side_exposed":
        z     = ground_z + hh * random.uniform(0.15, 0.35)
        roll  = random.choice([-1.0, 1.0]) * random.uniform(1.15, 1.45)
        pitch = random.uniform(-0.25, 0.25)
    else:
        z = ground_z + hh

    quat = p.getQuaternionFromEuler([roll, pitch, yaw])
    return z, quat, yaw

def tint_all_links(body_id, rgba):
    p.changeVisualShape(body_id, -1, rgbaColor=rgba)
    for j in range(p.getNumJoints(body_id)):
        p.changeVisualShape(body_id, j, rgbaColor=rgba)

def crop_texture_patch(tex_rgb, cx, cy, crop_size_px, out_size):
    tex_h, tex_w = tex_rgb.shape[:2]
    half = crop_size_px // 2
    x0, y0 = int(round(cx - half)), int(round(cy - half))
    x1, y1 = x0 + crop_size_px, y0 + crop_size_px
    pad_left   = max(0, -x0);      pad_top    = max(0, -y0)
    pad_right  = max(0, x1 - tex_w); pad_bottom = max(0, y1 - tex_h)
    patch = tex_rgb[max(0,y0):min(tex_h,y1), max(0,x0):min(tex_w,x1)].copy()
    if pad_left or pad_top or pad_right or pad_bottom:
        patch = cv2.copyMakeBorder(patch, pad_top, pad_bottom, pad_left, pad_right,
                                   borderType=cv2.BORDER_REFLECT)
    return cv2.resize(patch, (out_size, out_size), interpolation=cv2.INTER_LINEAR)

def sample_capture_positions(aabb_min, aabb_max, terrain_id, tex_rgb, frame_count, min_spacing=8.0):
    xmin, ymin = aabb_min[0] + 5.0, aabb_min[1] + 5.0
    xmax, ymax = aabb_max[0] - 5.0, aabb_max[1] - 5.0
    positions  = []
    attempts   = 0
    max_att    = frame_count * 300

    while len(positions) < frame_count and attempts < max_att:
        attempts += 1
        x = random.uniform(xmin, xmax)
        y = random.uniform(ymin, ymax)
        if not texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
            continue
        z = terrain_hit_z(x, y, terrain_id)
        if z is None:
            continue
        too_close = any(
            (x - px) ** 2 + (y - py) ** 2 < min_spacing ** 2
            for px, py, _ in positions
        )
        if too_close:
            continue
        positions.append((x, y, z))

    if len(positions) < frame_count:
        print(f"Warning: only {len(positions)} spaced positions, relaxing…")
        while len(positions) < frame_count and attempts < max_att * 2:
            attempts += 1
            x = random.uniform(xmin, xmax)
            y = random.uniform(ymin, ymax)
            if not texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
                continue
            z = terrain_hit_z(x, y, terrain_id)
            if z is None:
                continue
            positions.append((x, y, z))

    return positions[:frame_count]

# ── PYBULLET SETUP ────────────────────────────────────────────────────────────

try:
    if p.isConnected():
        p.disconnect()
except Exception:
    pass

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0, 0, -9.81)
p.configureDebugVisualizer(p.COV_ENABLE_WIREFRAME, 0)

# ── LOAD HEIGHTMAP ────────────────────────────────────────────────────────────

hmap = imageio.imread(HEIGHTMAP)
if hmap.ndim == 3:
    hmap = hmap[..., 0]
hmap = hmap.astype(np.float32)
H, W = hmap.shape

MAX_HW = 512
if max(H, W) > MAX_HW:
    from PIL import Image
    hmap_img = Image.fromarray(hmap.astype(np.uint16) if hmap.max() > 255 else hmap.astype(np.uint8))
    hmap_img = hmap_img.resize((MAX_HW, MAX_HW), resample=Image.BILINEAR)
    hmap     = np.array(hmap_img).astype(np.float32)
    H, W     = hmap.shape

hmin, hmax = float(hmap.min()), float(hmap.max())
if hmax - hmin < 1e-6:
    raise ValueError("Heightmap is flat.")

hmap01          = (hmap - hmin) / (hmax - hmin)
heightfieldData = hmap01.flatten().tolist()

# ── LOAD TEXTURE ──────────────────────────────────────────────────────────────

tex = imageio.imread(TEXTURE)
if tex.ndim == 2:
    tex = np.stack([tex, tex, tex], axis=-1)
tex_rgb = tex[:, :, :3]
tex_h, tex_w = tex_rgb.shape[:2]

TARGET_M_PER_PX = 0.10
terrain_size_m  = float(tex_w) * TARGET_M_PER_PX
terrain_size_m  = max(80.0, min(terrain_size_m, 300.0))

desired_max_height = 12.0
sx = terrain_size_m / W
sy = terrain_size_m / H

# Ground sampling distance: metres per texture pixel
GSD_M_PER_TEXPX = terrain_size_m / tex_w   # used by overlay renderer

# ── CREATE TERRAIN ────────────────────────────────────────────────────────────

terrain_shape = p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD,
    meshScale=[sx, sy, desired_max_height],
    heightfieldTextureScaling=1.0,
    heightfieldData=heightfieldData,
    numHeightfieldRows=H,
    numHeightfieldColumns=W,
)
terrain_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=terrain_shape)
tex_id     = p.loadTexture(TEXTURE)
p.changeVisualShape(terrain_id, -1, textureUniqueId=tex_id)
p.changeDynamics(terrain_id, -1, lateralFriction=0.9, restitution=0.0)

aabb_min, aabb_max = p.getAABB(terrain_id)
center = [(aabb_min[i] + aabb_max[i]) / 2 for i in range(3)]
print("Terrain AABB:", aabb_min, aabb_max)

# ── SPAWN WASTE ───────────────────────────────────────────────────────────────

waste_items  = []
existing_xy  = []
xmin, ymin   = aabb_min[0] + 2.5, aabb_min[1] + 2.5
xmax, ymax   = aabb_max[0] - 2.5, aabb_max[1] - 2.5
cluster_seed = None

for _ in range(WASTE_COUNT):
    spawned = False
    for _try in range(MAX_SPAWN_TRIES):
        kind = random.choice(WASTE_URDFS)

        if cluster_seed is not None and random.random() < 0.30:
            x = cluster_seed[0] + random.uniform(-2.5, 2.5)
            y = cluster_seed[1] + random.uniform(-2.5, 2.5)
        else:
            x = random.uniform(xmin, xmax)
            y = random.uniform(ymin, ymax)

        if not (xmin <= x <= xmax and ymin <= y <= ymax):
            continue

        min_gap = max(MIN_WASTE_GAP, kind["footprint"] * 1.6)
        if not far_from_existing(x, y, existing_xy, min_gap):
            continue
        if not texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
            continue

        ground_z = terrain_hit_z(x, y, terrain_id)
        if ground_z is None:
            continue

        mode  = random.choice(kind["burial_bias"])
        scale = random.uniform(kind["scale_range"][0], kind["scale_range"][1])
        z, quat, yaw = placement_pose(kind, mode, ground_z)

        try:
            body_id = p.loadURDF(
                kind["path"],
                basePosition=[x, y, z],
                baseOrientation=quat,
                useFixedBase=True,
                globalScaling=scale,
            )
            tint_all_links(body_id, [1, 1, 1, 0.0])
        except Exception as e:
            print("URDF load failed:", kind["path"], e)
            continue

        tex_x, tex_y = world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h)

        waste_items.append({
            "name":         kind["name"],
            "world_x":      x,
            "world_y":      y,
            "tex_x":        tex_x,
            "tex_y":        tex_y,
            "yaw":          yaw,
            "burial_mode":  mode,        # ← NEW: passed to renderer
            "obj_scale":    scale,       # ← NEW: passed to renderer
        })

        existing_xy.append((x, y))
        if cluster_seed is None or random.random() < 0.18:
            cluster_seed = (x, y)

        spawned = True
        break

    if not spawned:
        print("Could not place one waste item.")

print(f"Total waste items placed: {len(waste_items)}")

# ── CAPTURE FRAMES ────────────────────────────────────────────────────────────

capture_dir = os.path.join(SCRIPT_DIR, "captured_frames")
labels_dir  = os.path.join(SCRIPT_DIR, "captured_frames", "labels")
os.makedirs(capture_dir, exist_ok=True)
os.makedirs(labels_dir,  exist_ok=True)

positions = sample_capture_positions(
    aabb_min=aabb_min, aabb_max=aabb_max, terrain_id=terrain_id,
    tex_rgb=tex_rgb, frame_count=FRAME_COUNT,
    min_spacing=MIN_CAPTURE_SPACING_M,
)
print(f"Capture positions selected: {len(positions)}")

for i, (x, y, z) in enumerate(positions):
    cx, cy = world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h)

    patch_rgb = crop_texture_patch(tex_rgb, cx, cy, CROP_SIZE_PX, OUTPUT_SIZE)
    patch_bgr = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2BGR)

    # ── photorealistic overlay (new) ──────────────────────────────────────
    patch_bgr, bboxes = overlay_waste_on_patch(
        patch_bgr   = patch_bgr,
        waste_items = waste_items,
        center_px   = (cx, cy),
        crop_size_px= CROP_SIZE_PX,
        out_size    = OUTPUT_SIZE,
        terrain_size_m = terrain_size_m,
    )

    # ── save image ────────────────────────────────────────────────────────
    out_path = os.path.join(capture_dir, f"frame_{i:03d}.png")
    cv2.imwrite(out_path, patch_bgr)

    # ── save YOLO label ───────────────────────────────────────────────────
    label_path = os.path.join(labels_dir, f"frame_{i:03d}.txt")
    with open(label_path, "w") as f:
        for bb in bboxes:
            cls  = CLASS_IDS.get(bb["name"], 0)
            bx   = (bb["x1"] + bb["x2"]) / 2.0 / OUTPUT_SIZE
            by   = (bb["y1"] + bb["y2"]) / 2.0 / OUTPUT_SIZE
            bw   = (bb["x2"] - bb["x1"]) / OUTPUT_SIZE
            bh   = (bb["y2"] - bb["y1"]) / OUTPUT_SIZE
            f.write(f"{cls} {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f}\n")

    print(f"saved: frame_{i:03d}.png  ({len(bboxes)} objects labelled)")

print(f"\nDone — {len(positions)} frames in '{capture_dir}/'")
print(f"YOLO labels in '{labels_dir}/'")

# ── OPTIONAL GUI VIEW ─────────────────────────────────────────────────────────

cube_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5])
cube_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.5, 0.5, 0.5], rgbaColor=[1, 0, 0, 1])
cube_id  = p.createMultiBody(baseMass=1, baseCollisionShapeIndex=cube_col,
                             baseVisualShapeIndex=cube_vis,
                             basePosition=[center[0], center[1], center[2] + 20])
p.resetDebugVisualizerCamera(
    cameraDistance=max(15.0, terrain_size_m * 0.20),
    cameraYaw=0, cameraPitch=-89,
    cameraTargetPosition=[center[0], center[1], 0],
)
while p.isConnected():
    p.stepSimulation()
    time.sleep(1 / 240)

Terrain AABB: (-45.91015625, -45.91015625, -6.0) (45.91015625, 45.91015625, 6.0)
Total waste items placed: 55
Capture positions selected: 50
saved: frame_000.png  (1 objects labelled)
saved: frame_001.png  (10 objects labelled)
saved: frame_002.png  (2 objects labelled)
saved: frame_003.png  (8 objects labelled)
saved: frame_004.png  (4 objects labelled)
saved: frame_005.png  (11 objects labelled)
saved: frame_006.png  (22 objects labelled)
saved: frame_007.png  (7 objects labelled)
saved: frame_008.png  (17 objects labelled)
saved: frame_009.png  (14 objects labelled)
saved: frame_010.png  (7 objects labelled)
saved: frame_011.png  (7 objects labelled)
saved: frame_012.png  (16 objects labelled)
saved: frame_013.png  (11 objects labelled)
saved: frame_014.png  (13 objects labelled)
saved: frame_015.png  (20 objects labelled)
saved: frame_016.png  (7 objects labelled)
saved: frame_017.png  (2 objects labelled)
saved: frame_018.png  (5 objects labelled)
saved: frame_019.png  (1 objects 

KeyboardInterrupt: 

In [29]:
# synthetic_waste_capture.py  — COMPLETE SINGLE FILE (v5, tested)
# Drop this ONE file into your project folder and run it directly.
# No overlay_waste.py needed. Delete any old overlay_waste.py first.
#
# Verified: all 4 object types render clearly on real desert terrain frames.

import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os, random, math
import cv2

HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE),   f"Missing {TEXTURE}"

# ── CONFIG ───────────────────────────────────────────────────────────────────

SCRIPT_DIR  = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
URDF_DIR    = SCRIPT_DIR
FRAME_COUNT           = 50
OUTPUT_SIZE           = 1024
CROP_SIZE_PX          = 320
MIN_CAPTURE_SPACING_M = 8.0
WASTE_COUNT           = 55
NO_DATA_RGB    = np.array([168, 156, 142], dtype=np.float32)
NO_DATA_THRESH = 18.0
MAX_SPAWN_TRIES = 50
MIN_WASTE_GAP   = 3.5    # metres — increased to prevent visual overlaps

WASTE_URDFS = [
    {"name":"bottle",       "path":os.path.join(URDF_DIR,"bottle.urdf"),
     "half_height":0.085,"footprint":0.04,
     "burial_bias":["surface","half_buried","mostly_buried","side_exposed"],
     "scale_range":(0.85,1.15)},
    {"name":"cardboard_bag","path":os.path.join(URDF_DIR,"cardboard_bag.urdf"),
     "half_height":0.10,"footprint":0.22,
     "burial_bias":["surface","surface","half_buried","top_only"],
     "scale_range":(1.35,1.75)},
    {"name":"cardboard_box","path":os.path.join(URDF_DIR,"cardboard_box.urdf"),
     "half_height":0.09,"footprint":0.24,
     "burial_bias":["surface","surface","half_buried","corner_peek","top_only"],
     "scale_range":(1.35,1.75)},
    {"name":"garbage_bag",  "path":os.path.join(URDF_DIR,"garbage_bag.urdf"),
     "half_height":0.18,"footprint":0.24,
     "burial_bias":["surface","surface","half_buried"],
     "scale_range":(1.00,1.30)},
]

CLASS_IDS = {"bottle":0,"cardboard_bag":1,"cardboard_box":2,"garbage_bag":3}

# ── DISPLAY SIZES (pixels at OUTPUT_SIZE=1024, tuned for ~30m crop) ──────────

DISPLAY_PX = {
    "bottle":        {"half_len":40, "half_r":16},
    "garbage_bag":   {"half_r":26},
    "cardboard_box": {"half_w":30,   "half_h":22},
    "cardboard_bag": {"half_w":26,   "half_h":26},
}

# (body_alpha, shadow_alpha, size_fraction) — shadow always < body
BURIAL_PARAMS = {
    "surface":       (1.00, 0.45, 1.00),
    "half_buried":   (0.75, 0.30, 0.85),
    "mostly_buried": (0.50, 0.20, 0.65),
    "top_only":      (0.55, 0.22, 0.65),
    "corner_peek":   (0.45, 0.18, 0.55),
    "side_exposed":  (0.80, 0.35, 0.90),
}

# ── OVERLAY DRAWING HELPERS ──────────────────────────────────────────────────

def _rng(seed):
    return np.random.default_rng(int(seed) % (2**31))

def _gauss_mask(H, W, cx, cy, rx, ry, angle_deg=0.0):
    ys, xs = np.mgrid[0:H, 0:W]
    dx = (xs - cx).astype(np.float32)
    dy = (ys - cy).astype(np.float32)
    if angle_deg:
        th = math.radians(angle_deg)
        c, s = math.cos(th), math.sin(th)
        dx, dy = c*dx + s*dy, -s*dx + c*dy
    d2 = (dx/max(rx,1))**2 + (dy/max(ry,1))**2
    return np.exp(-0.5*d2).astype(np.float32)

def _hard_ellipse(H, W, cx, cy, rx, ry, angle_deg=0.0):
    img = np.zeros((H,W), np.uint8)
    cv2.ellipse(img,(int(cx),int(cy)),(max(1,int(rx)),max(1,int(ry))),
                angle_deg,0,360,255,-1)
    return img.astype(np.float32)/255.0

def _hard_rect(H, W, cx, cy, w, h, angle_deg=0.0):
    img = np.zeros((H,W), np.uint8)
    pts = cv2.boxPoints(((float(cx),float(cy)),(float(w),float(h)),float(angle_deg))).astype(np.int32)
    cv2.fillConvexPoly(img, pts, 255)
    return img.astype(np.float32)/255.0

def _noise(H, W, amp, seed):
    return _rng(seed).standard_normal((H,W)).astype(np.float32)*amp

def _blend(canvas, bgr, alpha_map):
    a = np.clip(alpha_map,0,1)[:,:,np.newaxis]
    c = np.array(bgr,dtype=np.float32).reshape(1,1,3)
    canvas[:] = np.clip(canvas.astype(np.float32)*(1-a)+c*a,0,255).astype(np.uint8)

def _blend_img(canvas, img, alpha_map):
    a = np.clip(alpha_map,0,1)[:,:,np.newaxis]
    canvas[:] = np.clip(canvas.astype(np.float32)*(1-a)+img.astype(np.float32)*a,0,255).astype(np.uint8)

def _bbox(cx,cy,hw,hh,W,H):
    return (max(0,int(cx-hw)),max(0,int(cy-hh)),min(W-1,int(cx+hw)),min(H-1,int(cy+hh)))

# ── OBJECT RENDERERS ─────────────────────────────────────────────────────────

def _draw_bottle(canvas, cx, cy, half_len, half_r, angle_deg, body_a, shadow_a, seed,
                 pose="on_side"):
    """
    Bottle with 4 aerial-view poses:
      on_side        — lying flat, full elongated ellipse visible (original)
      upright        — standing up, only circular top/cap visible
      neck_up        — half-buried body, only neck+cap sticking out (small)
      base_up        — half-buried neck, only wide base circle visible
    pose is chosen randomly at spawn time and stored in waste_items.
    """
    H, W = canvas.shape[:2]
    r = _rng(seed)

    # Colour palette — high contrast against desert sand
    variants = [
        (255, 255, 255),  # clear/white
        (60,  200,  60),  # green PET  (BGR)
        (200,  60,  60),  # blue PET
        (220, 200,  50),  # teal
        (60,  220, 220),  # yellow
    ]
    bgr = variants[int(r.integers(len(variants)))]
    bgr = tuple(int(np.clip(c * float(r.uniform(0.85, 1.0)), 0, 255)) for c in bgr)

    caps = [(30,30,210),(210,40,40),(30,180,30),(20,200,230),(180,20,180)]
    cap_bgr = caps[int(r.integers(len(caps)))]

    sdx = max(2, int(half_r * 0.5))
    sdy = max(1, int(half_r * 0.25))

    if pose == "upright":
        # ── standing upright: circular top view, radius = bottle radius ──────
        # Shadow slightly offset
        s = _gauss_mask(H,W, cx+sdx, cy+sdy, half_r*1.15, half_r*1.15) * shadow_a
        _blend(canvas, (20,20,20), s)
        # Body circle
        body = _hard_ellipse(H,W, cx, cy, half_r, half_r) * body_a
        _blend(canvas, bgr, body)
        # Specular dot offset
        hi = _gauss_mask(H,W, cx-int(half_r*0.30), cy-int(half_r*0.30),
                         max(2,half_r*0.35), max(2,half_r*0.35)) * 0.85 * body_a
        _blend(canvas, (255,255,255), hi)
        # Cap ring in centre (concentric darker circle)
        cap = _hard_ellipse(H,W, cx, cy, max(3,int(half_r*0.55)), max(3,int(half_r*0.55))) * body_a
        _blend(canvas, cap_bgr, cap)
        # Noise
        n = _noise(H,W,8,seed+1)
        cf = canvas.astype(np.float32)
        cf += n[:,:,np.newaxis] * (body*0.4)[:,:,np.newaxis]
        canvas[:] = np.clip(cf,0,255).astype(np.uint8)
        pad = int(half_r * 1.2) + sdx
        return _bbox(cx, cy, pad, pad, W, H)

    elif pose == "neck_up":
        # ── half-buried, only narrow neck+cap poking out ──────────────────────
        # Shows as a small elongated stub — neck is ~1/4 bottle length
        stub_len = max(8, int(half_len * 0.38))
        stub_r   = max(3, int(half_r * 0.48))
        s = _gauss_mask(H,W, cx+sdx, cy+sdy, stub_r*1.1, stub_len*0.7, angle_deg) * shadow_a * 0.6
        _blend(canvas, (20,20,20), s)
        body = _hard_ellipse(H,W, cx, cy, stub_r, stub_len, angle_deg) * body_a
        _blend(canvas, bgr, body)
        hi_ox = int(math.cos(math.radians(angle_deg+90)) * stub_r * 0.38)
        hi_oy = int(math.sin(math.radians(angle_deg+90)) * stub_r * 0.38)
        hi = _gauss_mask(H,W, cx+hi_ox, cy+hi_oy,
                         max(2,stub_r*0.30), stub_len*0.50, angle_deg) * 0.80 * body_a
        _blend(canvas, (255,255,255), hi)
        # Cap at the very tip
        cap_ox = int(math.cos(math.radians(angle_deg+90)) * stub_len * 0.72)
        cap_oy = int(math.sin(math.radians(angle_deg+90)) * stub_len * 0.72)
        cap = _hard_ellipse(H,W, cx+cap_ox, cy+cap_oy,
                            max(3,int(stub_r*0.80)), max(3,int(stub_r*0.80))) * body_a
        _blend(canvas, cap_bgr, cap)
        n = _noise(H,W,8,seed+1)
        cf = canvas.astype(np.float32)
        cf += n[:,:,np.newaxis] * (body*0.4)[:,:,np.newaxis]
        canvas[:] = np.clip(cf,0,255).astype(np.uint8)
        return _bbox(cx, cy, stub_r+sdx+3, stub_len+sdy+3, W, H)

    elif pose == "base_up":
        # ── half-buried neck, only wide base circle sticking out ──────────────
        base_r = int(half_r * 1.05)  # base disc is slightly wider than body
        s = _gauss_mask(H,W, cx+sdx, cy+sdy, base_r*1.2, base_r*1.2) * shadow_a
        _blend(canvas, (20,20,20), s)
        body = _hard_ellipse(H,W, cx, cy, base_r, base_r) * body_a
        _blend(canvas, bgr, body)
        # Rim — slightly darker ring around edge
        rim = _gauss_mask(H,W, cx, cy, base_r*0.72, base_r*0.72) * body_a * 0.5
        _blend(canvas, tuple(int(np.clip(c*0.65,0,255)) for c in bgr), rim)
        # Specular
        hi = _gauss_mask(H,W, cx-int(base_r*0.28), cy-int(base_r*0.28),
                         max(2,base_r*0.28), max(2,base_r*0.28)) * 0.75 * body_a
        _blend(canvas, (255,255,255), hi)
        n = _noise(H,W,8,seed+1)
        cf = canvas.astype(np.float32)
        cf += n[:,:,np.newaxis] * (body*0.35)[:,:,np.newaxis]
        canvas[:] = np.clip(cf,0,255).astype(np.uint8)
        pad = int(base_r * 1.2) + sdx
        return _bbox(cx, cy, pad, pad, W, H)

    else:
        # ── on_side: bottle lying flat — long axis goes LEFT-RIGHT ───────────
        # half_len = long axis (along angle_deg), half_r = short axis (width)
        # The body looks wider and flatter than upright — like a cylinder on its side
        body_w = half_len          # long dimension (length of bottle)
        body_h = int(half_r * 1.6) # short dimension — slightly taller than radius
                                    # to show the cylinder curve from the side

        # Shadow beneath
        s = _gauss_mask(H,W, cx+sdx, cy+sdy, body_w*0.95, body_h*1.1, angle_deg)*shadow_a
        _blend(canvas, (20,20,20), s)

        # Main body — hard ellipse, long and narrow = lying-flat cylinder
        body = _hard_ellipse(H,W, cx, cy, body_w, body_h, angle_deg)*body_a
        _blend(canvas, bgr, body)

        # Top-lit highlight along the top edge (makes it look 3D / rounded)
        hi_ox = int(math.cos(math.radians(angle_deg+90)) * body_h*0.32)
        hi_oy = int(math.sin(math.radians(angle_deg+90)) * body_h*0.32)
        hi = _gauss_mask(H,W, cx+hi_ox, cy+hi_oy,
                         body_w*0.80, max(2, body_h*0.28), angle_deg)*0.80*body_a
        _blend(canvas, (255,255,255), hi)

        # Dark bottom edge (shadow side of cylinder)
        dk = _gauss_mask(H,W, cx-hi_ox, cy-hi_oy,
                         body_w*0.75, max(2, body_h*0.25), angle_deg)*0.38*body_a
        _blend(canvas, tuple(int(np.clip(c*0.40,0,255)) for c in bgr), dk)

        # Label band in the middle (slightly different colour stripe)
        label_col = tuple(int(np.clip(c*0.80,0,255)) for c in bgr)
        lbl = _gauss_mask(H,W, cx, cy,
                          body_w*0.45, max(2,body_h*0.55), angle_deg)*0.45*body_a
        _blend(canvas, label_col, lbl)

        # Cap disc at one end of the long axis
        cap_ox = int(math.cos(math.radians(angle_deg)) * body_w*0.88)
        cap_oy = int(math.sin(math.radians(angle_deg)) * body_w*0.88)
        cap_r  = max(4, int(body_h*0.72))
        cap    = _hard_ellipse(H,W, cx+cap_ox, cy+cap_oy, cap_r, cap_r)*body_a
        _blend(canvas, cap_bgr, cap)

        # Base disc at other end
        base_r = max(4, int(body_h*0.80))
        base   = _hard_ellipse(H,W, cx-cap_ox, cy-cap_oy, base_r, base_r)*body_a
        _blend(canvas, tuple(int(np.clip(c*0.75,0,255)) for c in bgr), base)

        # Noise
        n = _noise(H,W,8,seed+1)
        cf = canvas.astype(np.float32)
        cf += n[:,:,np.newaxis]*(body*0.4)[:,:,np.newaxis]
        canvas[:] = np.clip(cf,0,255).astype(np.uint8)

        return _bbox(cx, cy, body_w+sdx+3, body_h+sdy+3, W, H)


def _draw_garbage_bag(canvas, cx, cy, half_r, angle_deg, body_a, shadow_a, seed):
    H, W = canvas.shape[:2]
    r = _rng(seed)
    colours = [(12,12,12),(18,26,18),(20,20,32),(28,24,24),(38,38,38)]
    bag_bgr = np.array(colours[int(r.integers(len(colours)))],np.float32)
    rx = half_r*float(r.uniform(0.88,1.08))
    ry = half_r*float(r.uniform(0.70,0.86))

    # Shadow CAPPED to body size
    soff = max(2,int(half_r*0.25))
    s = _gauss_mask(H,W,cx+soff,cy+soff, min(rx*1.12,rx+6), min(ry*1.12,ry+6))*shadow_a
    _blend(canvas,(12,12,12),s)

    # Hard ellipse body first
    body = _hard_ellipse(H,W,cx,cy,int(rx),int(ry),angle_deg)*body_a
    _blend(canvas,bag_bgr.tolist(),body)

    # Soft dome on top
    dome = _gauss_mask(H,W,cx,cy,rx*0.72,ry*0.72,angle_deg)*body_a*0.5
    _blend(canvas,np.clip(bag_bgr+10,0,255).tolist(),dome)

    # Wrinkles
    for _ in range(int(r.integers(3,6))):
        wx = cx+int(r.uniform(-rx*0.45,rx*0.45))
        wy = cy+int(r.uniform(-ry*0.45,ry*0.45))
        wr = half_r*float(r.uniform(0.08,0.18))
        wm = _gauss_mask(H,W,wx,wy,wr,wr)*0.40*body_a
        _blend(canvas,np.clip(bag_bgr-float(r.uniform(3,10)),0,255).tolist(),wm)

    # Highlight + knot
    hi = _gauss_mask(H,W,cx-int(rx*0.25),cy-int(ry*0.22),max(2,rx*0.16),max(2,ry*0.16))*0.40*body_a
    _blend(canvas,(68,68,68),hi)
    km = _gauss_mask(H,W,cx+int(r.uniform(-rx*0.12,rx*0.12)),cy-int(ry*0.74),
                     max(2,int(half_r*0.15)),max(2,int(half_r*0.15)))*0.92*body_a
    _blend(canvas,(50,50,50),km)

    return _bbox(cx,cy,rx+soff+2,ry+soff+2,W,H)


def _draw_cardboard_box(canvas, cx, cy, half_w, half_h, angle_deg, body_a, shadow_a, seed):
    H, W = canvas.shape[:2]
    r = _rng(seed)
    palette = [(75,115,165),(65,100,145),(55,85,120),(90,130,178),(60,95,135)]
    base = np.array(palette[int(r.integers(len(palette)))],np.float32)
    base = np.clip(base*float(r.uniform(0.85,1.10)),0,255)
    w,h = half_w*2, half_h*2
    soff = max(2,int(min(w,h)*0.18))

    s = _hard_rect(H,W,cx+soff,cy+soff,w*1.12,h*1.12,angle_deg)*shadow_a
    _blend(canvas,(18,18,18),s)

    face = _hard_rect(H,W,cx,cy,w,h,angle_deg)
    grad = np.linspace(0.82,1.08,W,dtype=np.float32)
    gi = np.outer(np.ones(H,np.float32),grad)
    fi = np.zeros((H,W,3),np.float32)
    n = _noise(H,W,13,seed+2)
    for c in range(3): fi[:,:,c] = base[c]*gi+n
    fi = np.clip(fi,0,255)
    _blend_img(canvas,fi.astype(np.uint8),face*body_a)

    crease = _gauss_mask(H,W,cx,cy,max(1,half_w*0.06),half_h*0.88,angle_deg)*0.45*body_a
    _blend(canvas,np.clip(base*0.50,0,255).tolist(),crease)

    th = math.radians(angle_deg)
    fx = cx+int(math.cos(th)*half_w*0.82); fy = cy+int(math.sin(th)*half_w*0.82)
    flap = _gauss_mask(H,W,fx,fy,max(2,half_w*0.18),max(2,half_h*0.75),angle_deg)*0.30*body_a
    _blend(canvas,np.clip(base*1.30,0,255).tolist(),flap)

    return _bbox(cx,cy,half_w+soff+2,half_h+soff+2,W,H)


def _draw_cardboard_bag(canvas, cx, cy, half_w, half_h, angle_deg, body_a, shadow_a, seed):
    H, W = canvas.shape[:2]
    r = _rng(seed)
    palette = [(80,118,162),(68,102,142),(58,88,122),(95,135,180),(72,110,155)]
    base = np.array(palette[int(r.integers(len(palette)))],np.float32)
    base = np.clip(base*float(r.uniform(0.85,1.10)),0,255)
    w,h = half_w*2, half_h*2
    soff = max(2,int(min(w,h)*0.16))

    s = _hard_rect(H,W,cx+soff,cy+soff,w*1.08,h*1.08,angle_deg)*shadow_a
    _blend(canvas,(18,18,18),s)

    face = _hard_rect(H,W,cx,cy,w,h,angle_deg)
    fi = np.zeros((H,W,3),np.float32)
    n = _noise(H,W,11,seed+5)
    for c in range(3): fi[:,:,c] = base[c]+n
    fi = np.clip(fi,0,255)
    _blend_img(canvas,fi.astype(np.uint8),face*body_a)

    th = math.radians(angle_deg)
    for frac in (-0.28,0.28):
        fx=cx+int(math.cos(th)*half_w*frac*2); fy=cy+int(math.sin(th)*half_w*frac*2)
        fold=_gauss_mask(H,W,fx,fy,max(1,half_w*0.07),half_h*0.82,angle_deg)*0.40*body_a
        _blend(canvas,np.clip(base*0.52,0,255).tolist(),fold)

    hcx=cx+int(math.cos(math.radians(angle_deg+90))*half_h*0.72)
    hcy=cy+int(math.sin(math.radians(angle_deg+90))*half_h*0.72)
    handle=_gauss_mask(H,W,hcx,hcy,max(2,half_w*0.48),max(2,half_h*0.09),angle_deg)*0.75*body_a
    _blend(canvas,np.clip(base*0.42,0,255).tolist(),handle)

    hi_cx=cx-int(math.cos(th)*half_w*0.82); hi_cy=cy-int(math.sin(th)*half_w*0.82)
    hi=_gauss_mask(H,W,hi_cx,hi_cy,max(2,half_w*0.09),max(2,half_h*0.72),angle_deg)*0.32*body_a
    _blend(canvas,np.clip(base*1.32,0,255).tolist(),hi)

    return _bbox(cx,cy,half_w+soff+2,half_h+soff+2,W,H)


# ── MAIN OVERLAY FUNCTION ────────────────────────────────────────────────────

def render_waste_on_patch(patch_bgr, waste_items, center_px, crop_size_px, out_size):
    """
    Renders all waste items onto patch_bgr.
    Returns (annotated_image, bboxes_list).
    bboxes_list = [{name, x1, y1, x2, y2}, ...]
    """
    canvas = patch_bgr.copy()
    H, W   = canvas.shape[:2]
    scale  = out_size / crop_size_px
    half   = crop_size_px / 2.0
    SB     = out_size / 1024.0   # size scale base

    bboxes = []

    for item in waste_items:
        dx = item["tex_x"] - center_px[0]
        dy = item["tex_y"] - center_px[1]
        if abs(dx) > crop_size_px * 0.54 or abs(dy) > crop_size_px * 0.54:
            continue
        px = int((dx + half) * scale)
        py = int((dy + half) * scale)
        if not (0 <= px < W and 0 <= py < H):
            continue

        name      = item["name"]
        angle_deg = -math.degrees(item.get("yaw", 0.0))
        mode      = item.get("burial_mode", "surface")
        oscale    = float(item.get("obj_scale", 1.0))
        body_a, shadow_a, sfrac = BURIAL_PARAMS.get(mode, (1.0,0.45,1.0))
        seed = abs(hash((item["tex_x"], item["tex_y"], name))) % (2**31)
        S    = SB * sfrac * oscale

        if name == "bottle":
            dp   = DISPLAY_PX["bottle"]
            pose = item.get("bottle_pose", "on_side")
            bb   = _draw_bottle(canvas,px,py, dp["half_len"]*S, dp["half_r"]*S,
                                angle_deg,body_a,shadow_a,seed, pose=pose)
        elif name == "garbage_bag":
            dp = DISPLAY_PX["garbage_bag"]
            bb = _draw_garbage_bag(canvas,px,py, dp["half_r"]*S,
                                   angle_deg,body_a,shadow_a,seed)
        elif name == "cardboard_box":
            dp = DISPLAY_PX["cardboard_box"]
            bb = _draw_cardboard_box(canvas,px,py, dp["half_w"]*S, dp["half_h"]*S,
                                     angle_deg,body_a,shadow_a,seed)
        elif name == "cardboard_bag":
            dp = DISPLAY_PX["cardboard_bag"]
            bb = _draw_cardboard_bag(canvas,px,py, dp["half_w"]*S, dp["half_h"]*S,
                                     angle_deg,body_a,shadow_a,seed)
        else:
            continue

        if bb:
            bboxes.append({"name":name,"x1":bb[0],"y1":bb[1],"x2":bb[2],"y2":bb[3]})

    patch_bgr[:] = canvas
    return patch_bgr, bboxes


# ── TERRAIN HELPERS ──────────────────────────────────────────────────────────

def world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h):
    u  = (x-aabb_min[0])/(aabb_max[0]-aabb_min[0])
    v  = (y-aabb_min[1])/(aabb_max[1]-aabb_min[1])
    px = int(np.clip(u*(tex_w-1),0,tex_w-1))
    py = int(np.clip((1.0-v)*(tex_h-1),0,tex_h-1))
    return px, py

def texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
    th,tw = tex_rgb.shape[:2]
    px,py = world_to_tex_uv(x,y,aabb_min,aabb_max,tw,th)
    rgb   = tex_rgb[py,px].astype(np.float32)
    if np.linalg.norm(rgb-NO_DATA_RGB) < NO_DATA_THRESH: return False
    x0=max(0,px-8);x1=min(tw,px+9);y0=max(0,py-8);y1=min(th,py+9)
    if tex_rgb[y0:y1,x0:x1].astype(np.float32).std() < 3.0: return False
    return True

def terrain_hit_z(x, y, terrain_id):
    hit = p.rayTest([x,y,200],[x,y,-200])[0]
    return hit[3][2] if hit[0]==terrain_id else None

def far_from_existing(x, y, existing_xy, min_gap):
    for ex,ey in existing_xy:
        if (x-ex)**2+(y-ey)**2 < min_gap**2: return False
    return True

def placement_pose(kind, mode, ground_z):
    yaw=random.uniform(-math.pi,math.pi); roll=0.0; pitch=0.0
    hh=kind["half_height"]
    if mode=="surface":          z=ground_z+hh+random.uniform(0.005,0.02)
    elif mode=="half_buried":
        z=ground_z+hh*random.uniform(0.35,0.60)
        if kind["name"] in ["bottle","cardboard_box"]:
            roll=random.uniform(-0.45,0.45); pitch=random.uniform(-0.45,0.45)
        else:
            roll=random.uniform(-0.20,0.20); pitch=random.uniform(-0.20,0.20)
    elif mode=="mostly_buried":  z=ground_z+hh*random.uniform(0.08,0.25); roll=random.uniform(-0.30,0.30); pitch=random.uniform(-0.30,0.30)
    elif mode=="top_only":       z=ground_z+hh*random.uniform(0.05,0.15); roll=random.uniform(-0.20,0.20); pitch=random.uniform(-0.20,0.20)
    elif mode=="corner_peek":    z=ground_z+hh*random.uniform(0.15,0.30); roll=random.choice([-1,1])*random.uniform(0.6,1.1); pitch=random.choice([-1,1])*random.uniform(0.2,0.7)
    elif mode=="side_exposed":   z=ground_z+hh*random.uniform(0.15,0.35); roll=random.choice([-1,1])*random.uniform(1.15,1.45); pitch=random.uniform(-0.25,0.25)
    else:                        z=ground_z+hh
    return z, p.getQuaternionFromEuler([roll,pitch,yaw]), yaw

def tint_all_links(body_id, rgba):
    p.changeVisualShape(body_id,-1,rgbaColor=rgba)
    for j in range(p.getNumJoints(body_id)):
        p.changeVisualShape(body_id,j,rgbaColor=rgba)

def crop_texture_patch(tex_rgb, cx, cy, crop_size_px, out_size):
    th,tw = tex_rgb.shape[:2]
    half  = crop_size_px//2
    x0,y0 = int(round(cx-half)),int(round(cy-half))
    x1,y1 = x0+crop_size_px, y0+crop_size_px
    pl=max(0,-x0);pt=max(0,-y0);pr=max(0,x1-tw);pb=max(0,y1-th)
    patch = tex_rgb[max(0,y0):min(th,y1), max(0,x0):min(tw,x1)].copy()
    if pl or pt or pr or pb:
        patch = cv2.copyMakeBorder(patch,pt,pb,pl,pr,borderType=cv2.BORDER_REFLECT)
    return cv2.resize(patch,(out_size,out_size),interpolation=cv2.INTER_LINEAR)

def sample_capture_positions(aabb_min, aabb_max, terrain_id, tex_rgb, frame_count, min_spacing=8.0):
    xmin,ymin = aabb_min[0]+5.0, aabb_min[1]+5.0
    xmax,ymax = aabb_max[0]-5.0, aabb_max[1]-5.0
    positions=[]; attempts=0; max_att=frame_count*300
    while len(positions)<frame_count and attempts<max_att:
        attempts+=1
        x=random.uniform(xmin,xmax); y=random.uniform(ymin,ymax)
        if not texture_ok_for_spawn(x,y,tex_rgb,aabb_min,aabb_max): continue
        z=terrain_hit_z(x,y,terrain_id)
        if z is None: continue
        if any((x-px)**2+(y-py)**2<min_spacing**2 for px,py,_ in positions): continue
        positions.append((x,y,z))
    if len(positions)<frame_count:
        print(f"Warning: only {len(positions)} spaced positions, relaxing…")
        while len(positions)<frame_count and attempts<max_att*2:
            attempts+=1
            x=random.uniform(xmin,xmax); y=random.uniform(ymin,ymax)
            if not texture_ok_for_spawn(x,y,tex_rgb,aabb_min,aabb_max): continue
            z=terrain_hit_z(x,y,terrain_id)
            if z is None: continue
            positions.append((x,y,z))
    return positions[:frame_count]

# ── PYBULLET SETUP ────────────────────────────────────────────────────────────

try:
    if p.isConnected(): p.disconnect()
except: pass

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0,0,-9.81)
p.configureDebugVisualizer(p.COV_ENABLE_WIREFRAME,0)

# ── LOAD HEIGHTMAP ────────────────────────────────────────────────────────────

hmap = imageio.imread(HEIGHTMAP)
if hmap.ndim==3: hmap=hmap[...,0]
hmap=hmap.astype(np.float32); H,W=hmap.shape
MAX_HW=512
if max(H,W)>MAX_HW:
    from PIL import Image
    hi=Image.fromarray(hmap.astype(np.uint16) if hmap.max()>255 else hmap.astype(np.uint8))
    hi=hi.resize((MAX_HW,MAX_HW),resample=Image.BILINEAR)
    hmap=np.array(hi).astype(np.float32); H,W=hmap.shape
hmin,hmax=float(hmap.min()),float(hmap.max())
if hmax-hmin<1e-6: raise ValueError("Heightmap is flat.")
hmap01=(hmap-hmin)/(hmax-hmin)
heightfieldData=hmap01.flatten().tolist()

# ── LOAD TEXTURE ──────────────────────────────────────────────────────────────

tex=imageio.imread(TEXTURE)
if tex.ndim==2: tex=np.stack([tex,tex,tex],axis=-1)
tex_rgb=tex[:,:,:3]; tex_h,tex_w=tex_rgb.shape[:2]
TARGET_M_PER_PX=0.10
terrain_size_m=max(80.0,min(float(tex_w)*TARGET_M_PER_PX,300.0))
sx=terrain_size_m/W; sy=terrain_size_m/H

# ── CREATE TERRAIN ────────────────────────────────────────────────────────────

terrain_shape=p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD, meshScale=[sx,sy,12.0],
    heightfieldTextureScaling=1.0, heightfieldData=heightfieldData,
    numHeightfieldRows=H, numHeightfieldColumns=W)
terrain_id=p.createMultiBody(baseMass=0,baseCollisionShapeIndex=terrain_shape)
tex_id=p.loadTexture(TEXTURE)
p.changeVisualShape(terrain_id,-1,textureUniqueId=tex_id)
p.changeDynamics(terrain_id,-1,lateralFriction=0.9,restitution=0.0)
aabb_min,aabb_max=p.getAABB(terrain_id)
center=[(aabb_min[i]+aabb_max[i])/2 for i in range(3)]
print("Terrain AABB:",aabb_min,aabb_max)

# ── SPAWN WASTE ───────────────────────────────────────────────────────────────

waste_items=[]; existing_xy=[]
xmin,ymin=aabb_min[0]+2.5,aabb_min[1]+2.5
xmax,ymax=aabb_max[0]-2.5,aabb_max[1]-2.5
cluster_seed=None

for _ in range(WASTE_COUNT):
    spawned=False
    for _try in range(MAX_SPAWN_TRIES):
        kind=random.choice(WASTE_URDFS)
        if cluster_seed and random.random()<0.30:
            x=cluster_seed[0]+random.uniform(-6.0,6.0)
            y=cluster_seed[1]+random.uniform(-6.0,6.0)
        else:
            x=random.uniform(xmin,xmax); y=random.uniform(ymin,ymax)
        if not (xmin<=x<=xmax and ymin<=y<=ymax): continue
        min_gap=max(MIN_WASTE_GAP,kind["footprint"]*1.6)
        if not far_from_existing(x,y,existing_xy,min_gap): continue
        if not texture_ok_for_spawn(x,y,tex_rgb,aabb_min,aabb_max): continue
        ground_z=terrain_hit_z(x,y,terrain_id)
        if ground_z is None: continue
        mode=random.choice(kind["burial_bias"])
        scale=random.uniform(kind["scale_range"][0],kind["scale_range"][1])
        z,quat,yaw=placement_pose(kind,mode,ground_z)
        try:
            body_id=p.loadURDF(kind["path"],basePosition=[x,y,z],
                               baseOrientation=quat,useFixedBase=True,globalScaling=scale)
            tint_all_links(body_id,[1,1,1,0.0])
        except Exception as e:
            print("URDF load failed:",kind["path"],e); continue
        tx,ty=world_to_tex_uv(x,y,aabb_min,aabb_max,tex_w,tex_h)

        # Assign bottle pose — randomised for variety in training data
        bottle_pose = "on_side"
        if kind["name"] == "bottle":
            # Weight distribution: mostly on_side, some upright, some partially buried forms
            bottle_pose = random.choices(
                ["on_side", "upright", "neck_up", "base_up"],
                weights=[0.40,    0.20,      0.20,     0.20]
            )[0]
            # Snap burial mode to match pose visually
            if bottle_pose == "upright":
                mode = random.choice(["surface", "surface", "half_buried"])
            elif bottle_pose in ("neck_up", "base_up"):
                mode = random.choice(["half_buried", "mostly_buried"])

        waste_items.append({"name":kind["name"],"world_x":x,"world_y":y,
                            "tex_x":tx,"tex_y":ty,"yaw":yaw,
                            "burial_mode":mode,"obj_scale":scale,
                            "bottle_pose":bottle_pose})
        existing_xy.append((x,y))
        if cluster_seed is None or random.random()<0.18: cluster_seed=(x,y)
        spawned=True; break
    if not spawned: print("Could not place one waste item.")

print(f"Waste items placed: {len(waste_items)}")

# ── CAPTURE FRAMES ────────────────────────────────────────────────────────────

capture_dir=os.path.join(SCRIPT_DIR,"captured_frames")
labels_dir =os.path.join(capture_dir,"labels")
os.makedirs(capture_dir,exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

positions=sample_capture_positions(aabb_min,aabb_max,terrain_id,tex_rgb,
                                   FRAME_COUNT,MIN_CAPTURE_SPACING_M)
print(f"Capture positions: {len(positions)}")

for i,(x,y,z) in enumerate(positions):
    cx,cy=world_to_tex_uv(x,y,aabb_min,aabb_max,tex_w,tex_h)
    patch_rgb=crop_texture_patch(tex_rgb,cx,cy,CROP_SIZE_PX,OUTPUT_SIZE)
    patch_bgr=cv2.cvtColor(patch_rgb,cv2.COLOR_RGB2BGR)

    # ── render overlays ──────────────────────────────────────────────────────
    patch_bgr, bboxes = render_waste_on_patch(
        patch_bgr, waste_items,
        center_px=(cx,cy),
        crop_size_px=CROP_SIZE_PX,
        out_size=OUTPUT_SIZE,
    )

    out_path=os.path.join(capture_dir,f"frame_{i:03d}.png")
    cv2.imwrite(out_path,patch_bgr)

    label_path=os.path.join(labels_dir,f"frame_{i:03d}.txt")
    with open(label_path,"w") as f:
        for bb in bboxes:
            cls=CLASS_IDS.get(bb["name"],0)
            bx=(bb["x1"]+bb["x2"])/2.0/OUTPUT_SIZE
            by=(bb["y1"]+bb["y2"])/2.0/OUTPUT_SIZE
            bw=(bb["x2"]-bb["x1"])/OUTPUT_SIZE
            bh=(bb["y2"]-bb["y1"])/OUTPUT_SIZE
            f.write(f"{cls} {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f}\n")

    print(f"frame_{i:03d}.png  {len(bboxes)} objects")

print(f"\nDone — {len(positions)} frames in '{capture_dir}/'")
print(f"YOLO labels in '{labels_dir}/'")

# ── GUI ───────────────────────────────────────────────────────────────────────

cube_col=p.createCollisionShape(p.GEOM_BOX,halfExtents=[0.5,0.5,0.5])
cube_vis=p.createVisualShape(p.GEOM_BOX,halfExtents=[0.5,0.5,0.5],rgbaColor=[1,0,0,1])
p.createMultiBody(baseMass=1,baseCollisionShapeIndex=cube_col,baseVisualShapeIndex=cube_vis,
                  basePosition=[center[0],center[1],center[2]+20])
p.resetDebugVisualizerCamera(cameraDistance=max(15.0,terrain_size_m*0.20),
                             cameraYaw=0,cameraPitch=-89,
                             cameraTargetPosition=[center[0],center[1],0])
while p.isConnected():
    p.stepSimulation(); time.sleep(1/240)

Terrain AABB: (-45.91015625, -45.91015625, -6.0) (45.91015625, 45.91015625, 6.0)
Waste items placed: 55
Capture positions: 50
frame_000.png  6 objects
frame_001.png  8 objects
frame_002.png  2 objects
frame_003.png  9 objects
frame_004.png  6 objects
frame_005.png  11 objects
frame_006.png  14 objects
frame_007.png  4 objects
frame_008.png  10 objects
frame_009.png  7 objects
frame_010.png  2 objects
frame_011.png  15 objects
frame_012.png  6 objects
frame_013.png  4 objects
frame_014.png  7 objects
frame_015.png  3 objects
frame_016.png  9 objects
frame_017.png  6 objects
frame_018.png  6 objects
frame_019.png  3 objects
frame_020.png  5 objects
frame_021.png  1 objects
frame_022.png  6 objects
frame_023.png  6 objects
frame_024.png  16 objects
frame_025.png  7 objects
frame_026.png  4 objects
frame_027.png  1 objects
frame_028.png  1 objects
frame_029.png  11 objects
frame_030.png  7 objects
frame_031.png  12 objects
frame_032.png  14 objects
frame_033.png  19 objects
frame_034.png  

In [32]:
# synthetic_waste_capture.py  — COMPLETE SINGLE FILE (v5, tested)
# Drop this ONE file into your project folder and run it directly.
# No overlay_waste.py needed. Delete any old overlay_waste.py first.
#
# Verified: all 4 object types render clearly on real desert terrain frames.

import pybullet as p
import pybullet_data
import imageio.v2 as imageio
import numpy as np
import time, os, random, math
import cv2

HEIGHTMAP = "terrain_heightmap.png"
TEXTURE   = "terrain_texture_contrast.png"

assert os.path.exists(HEIGHTMAP), f"Missing {HEIGHTMAP}"
assert os.path.exists(TEXTURE),   f"Missing {TEXTURE}"

# ── CONFIG ───────────────────────────────────────────────────────────────────

SCRIPT_DIR  = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
URDF_DIR    = SCRIPT_DIR
FRAME_COUNT           = 400
OUTPUT_SIZE           = 1024
CROP_SIZE_PX          = 320
MIN_CAPTURE_SPACING_M = 8.0
WASTE_COUNT           = 55
NO_DATA_RGB    = np.array([168, 156, 142], dtype=np.float32)
NO_DATA_THRESH = 18.0
MAX_SPAWN_TRIES = 50
MIN_WASTE_GAP   = 3.5    # metres — increased to prevent visual overlaps

WASTE_URDFS = [
    {"name":"bottle",       "path":os.path.join(URDF_DIR,"bottle.urdf"),
     "half_height":0.085,"footprint":0.04,
     "burial_bias":["surface","half_buried","mostly_buried","side_exposed"],
     "scale_range":(0.85,1.15)},
    {"name":"cardboard_bag","path":os.path.join(URDF_DIR,"cardboard_bag.urdf"),
     "half_height":0.10,"footprint":0.22,
     "burial_bias":["surface","surface","half_buried","top_only"],
     "scale_range":(1.35,1.75)},
    {"name":"cardboard_box","path":os.path.join(URDF_DIR,"cardboard_box.urdf"),
     "half_height":0.09,"footprint":0.24,
     "burial_bias":["surface","surface","half_buried","corner_peek","top_only"],
     "scale_range":(1.35,1.75)},
    {"name":"garbage_bag",  "path":os.path.join(URDF_DIR,"garbage_bag.urdf"),
     "half_height":0.18,"footprint":0.24,
     "burial_bias":["surface","surface","half_buried"],
     "scale_range":(1.00,1.30)},
]

CLASS_IDS = {"bottle":0,"cardboard_bag":1,"cardboard_box":2,"garbage_bag":3}

# ── DISPLAY SIZES (pixels at OUTPUT_SIZE=1024) ───────────────────────────────
DISPLAY_PX = {
    "bottle":        {"half_len": 40, "half_r": 16},
    "garbage_bag":   {"half_r":   26},
    "cardboard_box": {"half_w":   30, "half_h": 22},
    "cardboard_bag": {"half_w":   26, "half_h": 26},
}

# (body_alpha, shadow_alpha, size_fraction) — shadow always < body
BURIAL_PARAMS = {
    "surface":       (1.00, 0.45, 1.00),
    "half_buried":   (0.75, 0.30, 0.85),
    "mostly_buried": (0.50, 0.20, 0.65),
    "top_only":      (0.55, 0.22, 0.65),
    "corner_peek":   (0.45, 0.18, 0.55),
    "side_exposed":  (0.80, 0.35, 0.90),
}

# ── OVERLAY DRAWING HELPERS ──────────────────────────────────────────────────

def _rng(seed):
    return np.random.default_rng(int(seed) % (2**31))

def _gauss_mask(H, W, cx, cy, rx, ry, angle_deg=0.0):
    ys, xs = np.mgrid[0:H, 0:W]
    dx = (xs - cx).astype(np.float32)
    dy = (ys - cy).astype(np.float32)
    if angle_deg:
        th = math.radians(angle_deg)
        c, s = math.cos(th), math.sin(th)
        dx, dy = c*dx + s*dy, -s*dx + c*dy
    d2 = (dx/max(rx,1))**2 + (dy/max(ry,1))**2
    return np.exp(-0.5*d2).astype(np.float32)

def _hard_ellipse(H, W, cx, cy, rx, ry, angle_deg=0.0):
    img = np.zeros((H,W), np.uint8)
    cv2.ellipse(img,(int(cx),int(cy)),(max(1,int(rx)),max(1,int(ry))),
                angle_deg,0,360,255,-1)
    return img.astype(np.float32)/255.0

def _hard_rect(H, W, cx, cy, w, h, angle_deg=0.0):
    img = np.zeros((H,W), np.uint8)
    pts = cv2.boxPoints(((float(cx),float(cy)),(float(w),float(h)),float(angle_deg))).astype(np.int32)
    cv2.fillConvexPoly(img, pts, 255)
    return img.astype(np.float32)/255.0

def _noise(H, W, amp, seed):
    return _rng(seed).standard_normal((H,W)).astype(np.float32)*amp

def _blend(canvas, bgr, alpha_map):
    a = np.clip(alpha_map,0,1)[:,:,np.newaxis]
    c = np.array(bgr,dtype=np.float32).reshape(1,1,3)
    canvas[:] = np.clip(canvas.astype(np.float32)*(1-a)+c*a,0,255).astype(np.uint8)

def _blend_img(canvas, img, alpha_map):
    a = np.clip(alpha_map,0,1)[:,:,np.newaxis]
    canvas[:] = np.clip(canvas.astype(np.float32)*(1-a)+img.astype(np.float32)*a,0,255).astype(np.uint8)

def _bbox(cx,cy,hw,hh,W,H):
    return (max(0,int(cx-hw)),max(0,int(cy-hh)),min(W-1,int(cx+hw)),min(H-1,int(cy+hh)))

# ── OBJECT RENDERERS ─────────────────────────────────────────────────────────

def _draw_bottle(canvas, cx, cy, half_len, half_r, angle_deg, body_a, shadow_a, seed,
                 pose="on_side"):
    """
    Bottle with 4 aerial-view poses:
      on_side        — lying flat, full elongated ellipse visible (original)
      upright        — standing up, only circular top/cap visible
      neck_up        — half-buried body, only neck+cap sticking out (small)
      base_up        — half-buried neck, only wide base circle visible
    pose is chosen randomly at spawn time and stored in waste_items.
    """
    H, W = canvas.shape[:2]
    r = _rng(seed)

    # High-contrast colours — clearly visible on desert terrain
    variants = [
        (255, 255, 255),  # clear/white
        (60,  200,  60),  # green PET  (BGR)
        (200,  60,  60),  # blue PET
        (220, 200,  50),  # teal
        (60,  220, 220),  # yellow
    ]
    bgr = variants[int(r.integers(len(variants)))]
    bgr = tuple(int(np.clip(c * float(r.uniform(0.85, 1.0)), 0, 255)) for c in bgr)

    caps = [(30,30,210),(210,40,40),(30,180,30),(20,200,230),(180,20,180)]
    cap_bgr = caps[int(r.integers(len(caps)))]

    sdx = max(2, int(half_r * 0.5))
    sdy = max(1, int(half_r * 0.25))

    if pose == "upright":
        # ── standing upright: circular top view, radius = bottle radius ──────
        # Shadow slightly offset
        s = _gauss_mask(H,W, cx+sdx, cy+sdy, half_r*1.15, half_r*1.15) * shadow_a
        _blend(canvas, (20,20,20), s)
        # Body circle
        body = _hard_ellipse(H,W, cx, cy, half_r, half_r) * body_a
        _blend(canvas, bgr, body)
        # Specular dot offset
        hi = _gauss_mask(H,W, cx-int(half_r*0.30), cy-int(half_r*0.30),
                         max(2,half_r*0.35), max(2,half_r*0.35)) * 0.85 * body_a
        _blend(canvas, (255,255,255), hi)
        # Cap ring in centre (concentric darker circle)
        cap = _hard_ellipse(H,W, cx, cy, max(3,int(half_r*0.55)), max(3,int(half_r*0.55))) * body_a
        _blend(canvas, cap_bgr, cap)
        # Noise
        n = _noise(H,W,8,seed+1)
        cf = canvas.astype(np.float32)
        cf += n[:,:,np.newaxis] * (body*0.4)[:,:,np.newaxis]
        canvas[:] = np.clip(cf,0,255).astype(np.uint8)
        pad = int(half_r * 1.2) + sdx
        return _bbox(cx, cy, pad, pad, W, H)

    elif pose == "neck_up":
        # ── half-buried, only narrow neck+cap poking out ──────────────────────
        # Shows as a small elongated stub — neck is ~1/4 bottle length
        stub_len = max(8, int(half_len * 0.38))
        stub_r   = max(3, int(half_r * 0.48))
        s = _gauss_mask(H,W, cx+sdx, cy+sdy, stub_r*1.1, stub_len*0.7, angle_deg) * shadow_a * 0.6
        _blend(canvas, (20,20,20), s)
        body = _hard_ellipse(H,W, cx, cy, stub_r, stub_len, angle_deg) * body_a
        _blend(canvas, bgr, body)
        hi_ox = int(math.cos(math.radians(angle_deg+90)) * stub_r * 0.38)
        hi_oy = int(math.sin(math.radians(angle_deg+90)) * stub_r * 0.38)
        hi = _gauss_mask(H,W, cx+hi_ox, cy+hi_oy,
                         max(2,stub_r*0.30), stub_len*0.50, angle_deg) * 0.80 * body_a
        _blend(canvas, (255,255,255), hi)
        # Cap at the very tip
        cap_ox = int(math.cos(math.radians(angle_deg+90)) * stub_len * 0.72)
        cap_oy = int(math.sin(math.radians(angle_deg+90)) * stub_len * 0.72)
        cap = _hard_ellipse(H,W, cx+cap_ox, cy+cap_oy,
                            max(3,int(stub_r*0.80)), max(3,int(stub_r*0.80))) * body_a
        _blend(canvas, cap_bgr, cap)
        n = _noise(H,W,8,seed+1)
        cf = canvas.astype(np.float32)
        cf += n[:,:,np.newaxis] * (body*0.4)[:,:,np.newaxis]
        canvas[:] = np.clip(cf,0,255).astype(np.uint8)
        return _bbox(cx, cy, stub_r+sdx+3, stub_len+sdy+3, W, H)

    elif pose == "base_up":
        # ── half-buried neck, only wide base circle sticking out ──────────────
        base_r = int(half_r * 1.05)  # base disc is slightly wider than body
        s = _gauss_mask(H,W, cx+sdx, cy+sdy, base_r*1.2, base_r*1.2) * shadow_a
        _blend(canvas, (20,20,20), s)
        body = _hard_ellipse(H,W, cx, cy, base_r, base_r) * body_a
        _blend(canvas, bgr, body)
        # Rim — slightly darker ring around edge
        rim = _gauss_mask(H,W, cx, cy, base_r*0.72, base_r*0.72) * body_a * 0.5
        _blend(canvas, tuple(int(np.clip(c*0.65,0,255)) for c in bgr), rim)
        # Specular
        hi = _gauss_mask(H,W, cx-int(base_r*0.28), cy-int(base_r*0.28),
                         max(2,base_r*0.28), max(2,base_r*0.28)) * 0.75 * body_a
        _blend(canvas, (255,255,255), hi)
        n = _noise(H,W,8,seed+1)
        cf = canvas.astype(np.float32)
        cf += n[:,:,np.newaxis] * (body*0.35)[:,:,np.newaxis]
        canvas[:] = np.clip(cf,0,255).astype(np.uint8)
        pad = int(base_r * 1.2) + sdx
        return _bbox(cx, cy, pad, pad, W, H)

    else:
        # ── on_side: bottle lying flat — long axis goes LEFT-RIGHT ───────────
        # half_len = long axis (along angle_deg), half_r = short axis (width)
        # The body looks wider and flatter than upright — like a cylinder on its side
        body_w = half_len          # long dimension (length of bottle)
        body_h = int(half_r * 1.6) # short dimension — slightly taller than radius
                                    # to show the cylinder curve from the side

        # Shadow beneath
        s = _gauss_mask(H,W, cx+sdx, cy+sdy, body_w*0.95, body_h*1.1, angle_deg)*shadow_a
        _blend(canvas, (20,20,20), s)

        # Main body — hard ellipse, long and narrow = lying-flat cylinder
        body = _hard_ellipse(H,W, cx, cy, body_w, body_h, angle_deg)*body_a
        _blend(canvas, bgr, body)

        # Top-lit highlight along the top edge (makes it look 3D / rounded)
        hi_ox = int(math.cos(math.radians(angle_deg+90)) * body_h*0.32)
        hi_oy = int(math.sin(math.radians(angle_deg+90)) * body_h*0.32)
        hi = _gauss_mask(H,W, cx+hi_ox, cy+hi_oy,
                         body_w*0.80, max(2, body_h*0.28), angle_deg)*0.80*body_a
        _blend(canvas, (255,255,255), hi)

        # Dark bottom edge (shadow side of cylinder)
        dk = _gauss_mask(H,W, cx-hi_ox, cy-hi_oy,
                         body_w*0.75, max(2, body_h*0.25), angle_deg)*0.38*body_a
        _blend(canvas, tuple(int(np.clip(c*0.40,0,255)) for c in bgr), dk)

        # Label band in the middle (slightly different colour stripe)
        label_col = tuple(int(np.clip(c*0.80,0,255)) for c in bgr)
        lbl = _gauss_mask(H,W, cx, cy,
                          body_w*0.45, max(2,body_h*0.55), angle_deg)*0.45*body_a
        _blend(canvas, label_col, lbl)

        # Cap disc at one end of the long axis
        cap_ox = int(math.cos(math.radians(angle_deg)) * body_w*0.88)
        cap_oy = int(math.sin(math.radians(angle_deg)) * body_w*0.88)
        cap_r  = max(4, int(body_h*0.72))
        cap    = _hard_ellipse(H,W, cx+cap_ox, cy+cap_oy, cap_r, cap_r)*body_a
        _blend(canvas, cap_bgr, cap)

        # Base disc at other end
        base_r = max(4, int(body_h*0.80))
        base   = _hard_ellipse(H,W, cx-cap_ox, cy-cap_oy, base_r, base_r)*body_a
        _blend(canvas, tuple(int(np.clip(c*0.75,0,255)) for c in bgr), base)

        # Noise
        n = _noise(H,W,8,seed+1)
        cf = canvas.astype(np.float32)
        cf += n[:,:,np.newaxis]*(body*0.4)[:,:,np.newaxis]
        canvas[:] = np.clip(cf,0,255).astype(np.uint8)

        return _bbox(cx, cy, body_w+sdx+3, body_h+sdy+3, W, H)


def _draw_garbage_bag(canvas, cx, cy, half_r, angle_deg, body_a, shadow_a, seed):
    H, W = canvas.shape[:2]
    r = _rng(seed)
    colours = [(12,12,12),(18,26,18),(20,20,32),(28,24,24),(38,38,38)]
    bag_bgr = np.array(colours[int(r.integers(len(colours)))],np.float32)
    rx = half_r*float(r.uniform(0.88,1.08))
    ry = half_r*float(r.uniform(0.70,0.86))

    # Shadow CAPPED to body size
    soff = max(2,int(half_r*0.25))
    s = _gauss_mask(H,W,cx+soff,cy+soff, min(rx*1.12,rx+6), min(ry*1.12,ry+6))*shadow_a
    _blend(canvas,(12,12,12),s)

    # Hard ellipse body first
    body = _hard_ellipse(H,W,cx,cy,int(rx),int(ry),angle_deg)*body_a
    _blend(canvas,bag_bgr.tolist(),body)

    # Soft dome on top
    dome = _gauss_mask(H,W,cx,cy,rx*0.72,ry*0.72,angle_deg)*body_a*0.5
    _blend(canvas,np.clip(bag_bgr+10,0,255).tolist(),dome)

    # Wrinkles
    for _ in range(int(r.integers(3,6))):
        wx = cx+int(r.uniform(-rx*0.45,rx*0.45))
        wy = cy+int(r.uniform(-ry*0.45,ry*0.45))
        wr = half_r*float(r.uniform(0.08,0.18))
        wm = _gauss_mask(H,W,wx,wy,wr,wr)*0.40*body_a
        _blend(canvas,np.clip(bag_bgr-float(r.uniform(3,10)),0,255).tolist(),wm)

    # Highlight + knot
    hi = _gauss_mask(H,W,cx-int(rx*0.25),cy-int(ry*0.22),max(2,rx*0.16),max(2,ry*0.16))*0.40*body_a
    _blend(canvas,(68,68,68),hi)
    km = _gauss_mask(H,W,cx+int(r.uniform(-rx*0.12,rx*0.12)),cy-int(ry*0.74),
                     max(2,int(half_r*0.15)),max(2,int(half_r*0.15)))*0.92*body_a
    _blend(canvas,(50,50,50),km)

    return _bbox(cx,cy,rx+soff+2,ry+soff+2,W,H)


def _draw_cardboard_box(canvas, cx, cy, half_w, half_h, angle_deg, body_a, shadow_a, seed):
    H, W = canvas.shape[:2]
    r = _rng(seed)
    palette = [(75,115,165),(65,100,145),(55,85,120),(90,130,178),(60,95,135)]
    base = np.array(palette[int(r.integers(len(palette)))],np.float32)
    base = np.clip(base*float(r.uniform(0.85,1.10)),0,255)
    w,h = half_w*2, half_h*2
    soff = max(2,int(min(w,h)*0.18))

    s = _hard_rect(H,W,cx+soff,cy+soff,w*1.12,h*1.12,angle_deg)*shadow_a
    _blend(canvas,(18,18,18),s)

    face = _hard_rect(H,W,cx,cy,w,h,angle_deg)
    grad = np.linspace(0.82,1.08,W,dtype=np.float32)
    gi = np.outer(np.ones(H,np.float32),grad)
    fi = np.zeros((H,W,3),np.float32)
    n = _noise(H,W,13,seed+2)
    for c in range(3): fi[:,:,c] = base[c]*gi+n
    fi = np.clip(fi,0,255)
    _blend_img(canvas,fi.astype(np.uint8),face*body_a)

    crease = _gauss_mask(H,W,cx,cy,max(1,half_w*0.06),half_h*0.88,angle_deg)*0.45*body_a
    _blend(canvas,np.clip(base*0.50,0,255).tolist(),crease)

    th = math.radians(angle_deg)
    fx = cx+int(math.cos(th)*half_w*0.82); fy = cy+int(math.sin(th)*half_w*0.82)
    flap = _gauss_mask(H,W,fx,fy,max(2,half_w*0.18),max(2,half_h*0.75),angle_deg)*0.30*body_a
    _blend(canvas,np.clip(base*1.30,0,255).tolist(),flap)

    return _bbox(cx,cy,half_w+soff+2,half_h+soff+2,W,H)


def _draw_cardboard_bag(canvas, cx, cy, half_w, half_h, angle_deg, body_a, shadow_a, seed):
    H, W = canvas.shape[:2]
    r = _rng(seed)
    palette = [(80,118,162),(68,102,142),(58,88,122),(95,135,180),(72,110,155)]
    base = np.array(palette[int(r.integers(len(palette)))],np.float32)
    base = np.clip(base*float(r.uniform(0.85,1.10)),0,255)
    w,h = half_w*2, half_h*2
    soff = max(2,int(min(w,h)*0.16))

    s = _hard_rect(H,W,cx+soff,cy+soff,w*1.08,h*1.08,angle_deg)*shadow_a
    _blend(canvas,(18,18,18),s)

    face = _hard_rect(H,W,cx,cy,w,h,angle_deg)
    fi = np.zeros((H,W,3),np.float32)
    n = _noise(H,W,11,seed+5)
    for c in range(3): fi[:,:,c] = base[c]+n
    fi = np.clip(fi,0,255)
    _blend_img(canvas,fi.astype(np.uint8),face*body_a)

    th = math.radians(angle_deg)
    for frac in (-0.28,0.28):
        fx=cx+int(math.cos(th)*half_w*frac*2); fy=cy+int(math.sin(th)*half_w*frac*2)
        fold=_gauss_mask(H,W,fx,fy,max(1,half_w*0.07),half_h*0.82,angle_deg)*0.40*body_a
        _blend(canvas,np.clip(base*0.52,0,255).tolist(),fold)

    hcx=cx+int(math.cos(math.radians(angle_deg+90))*half_h*0.72)
    hcy=cy+int(math.sin(math.radians(angle_deg+90))*half_h*0.72)
    handle=_gauss_mask(H,W,hcx,hcy,max(2,half_w*0.48),max(2,half_h*0.09),angle_deg)*0.75*body_a
    _blend(canvas,np.clip(base*0.42,0,255).tolist(),handle)

    hi_cx=cx-int(math.cos(th)*half_w*0.82); hi_cy=cy-int(math.sin(th)*half_w*0.82)
    hi=_gauss_mask(H,W,hi_cx,hi_cy,max(2,half_w*0.09),max(2,half_h*0.72),angle_deg)*0.32*body_a
    _blend(canvas,np.clip(base*1.32,0,255).tolist(),hi)

    return _bbox(cx,cy,half_w+soff+2,half_h+soff+2,W,H)


# ── MAIN OVERLAY FUNCTION ────────────────────────────────────────────────────

def render_waste_on_patch(patch_bgr, waste_items, center_px, crop_size_px, out_size):
    """
    Renders all waste items onto patch_bgr.
    Returns (annotated_image, bboxes_list).
    bboxes_list = [{name, x1, y1, x2, y2}, ...]
    """
    canvas = patch_bgr.copy()
    H, W   = canvas.shape[:2]
    scale  = out_size / crop_size_px
    half   = crop_size_px / 2.0
    SB     = out_size / 1024.0   # size scale base

    bboxes = []

    for item in waste_items:
        dx = item["tex_x"] - center_px[0]
        dy = item["tex_y"] - center_px[1]
        if abs(dx) > crop_size_px * 0.54 or abs(dy) > crop_size_px * 0.54:
            continue
        px = int((dx + half) * scale)
        py = int((dy + half) * scale)
        if not (0 <= px < W and 0 <= py < H):
            continue

        name      = item["name"]
        angle_deg = -math.degrees(item.get("yaw", 0.0))
        mode      = item.get("burial_mode", "surface")
        oscale    = float(item.get("obj_scale", 1.0))
        body_a, shadow_a, sfrac = BURIAL_PARAMS.get(mode, (1.0,0.45,1.0))
        seed = abs(hash((item["tex_x"], item["tex_y"], name))) % (2**31)
        S    = SB * sfrac * oscale

        if name == "bottle":
            dp   = DISPLAY_PX["bottle"]
            pose = item.get("bottle_pose", "on_side")
            bb   = _draw_bottle(canvas,px,py, dp["half_len"]*S, dp["half_r"]*S,
                                angle_deg,body_a,shadow_a,seed, pose=pose)
        elif name == "garbage_bag":
            dp = DISPLAY_PX["garbage_bag"]
            bb = _draw_garbage_bag(canvas,px,py, dp["half_r"]*S,
                                   angle_deg,body_a,shadow_a,seed)
        elif name == "cardboard_box":
            dp = DISPLAY_PX["cardboard_box"]
            bb = _draw_cardboard_box(canvas,px,py, dp["half_w"]*S, dp["half_h"]*S,
                                     angle_deg,body_a,shadow_a,seed)
        elif name == "cardboard_bag":
            dp = DISPLAY_PX["cardboard_bag"]
            bb = _draw_cardboard_bag(canvas,px,py, dp["half_w"]*S, dp["half_h"]*S,
                                     angle_deg,body_a,shadow_a,seed)
        else:
            continue

        if bb:
            bboxes.append({"name":name,"x1":bb[0],"y1":bb[1],"x2":bb[2],"y2":bb[3]})

    patch_bgr[:] = canvas
    return patch_bgr, bboxes


# ── TERRAIN HELPERS ──────────────────────────────────────────────────────────

def world_to_tex_uv(x, y, aabb_min, aabb_max, tex_w, tex_h):
    u  = (x-aabb_min[0])/(aabb_max[0]-aabb_min[0])
    v  = (y-aabb_min[1])/(aabb_max[1]-aabb_min[1])
    px = int(np.clip(u*(tex_w-1),0,tex_w-1))
    py = int(np.clip((1.0-v)*(tex_h-1),0,tex_h-1))
    return px, py

def texture_ok_for_spawn(x, y, tex_rgb, aabb_min, aabb_max):
    th,tw = tex_rgb.shape[:2]
    px,py = world_to_tex_uv(x,y,aabb_min,aabb_max,tw,th)
    rgb   = tex_rgb[py,px].astype(np.float32)
    if np.linalg.norm(rgb-NO_DATA_RGB) < NO_DATA_THRESH: return False
    x0=max(0,px-8);x1=min(tw,px+9);y0=max(0,py-8);y1=min(th,py+9)
    if tex_rgb[y0:y1,x0:x1].astype(np.float32).std() < 3.0: return False
    return True

def terrain_hit_z(x, y, terrain_id):
    hit = p.rayTest([x,y,200],[x,y,-200])[0]
    return hit[3][2] if hit[0]==terrain_id else None

def far_from_existing(x, y, existing_xy, min_gap):
    for ex,ey in existing_xy:
        if (x-ex)**2+(y-ey)**2 < min_gap**2: return False
    return True

def placement_pose(kind, mode, ground_z):
    yaw=random.uniform(-math.pi,math.pi); roll=0.0; pitch=0.0
    hh=kind["half_height"]
    if mode=="surface":          z=ground_z+hh+random.uniform(0.005,0.02)
    elif mode=="half_buried":
        z=ground_z+hh*random.uniform(0.35,0.60)
        if kind["name"] in ["bottle","cardboard_box"]:
            roll=random.uniform(-0.45,0.45); pitch=random.uniform(-0.45,0.45)
        else:
            roll=random.uniform(-0.20,0.20); pitch=random.uniform(-0.20,0.20)
    elif mode=="mostly_buried":  z=ground_z+hh*random.uniform(0.08,0.25); roll=random.uniform(-0.30,0.30); pitch=random.uniform(-0.30,0.30)
    elif mode=="top_only":       z=ground_z+hh*random.uniform(0.05,0.15); roll=random.uniform(-0.20,0.20); pitch=random.uniform(-0.20,0.20)
    elif mode=="corner_peek":    z=ground_z+hh*random.uniform(0.15,0.30); roll=random.choice([-1,1])*random.uniform(0.6,1.1); pitch=random.choice([-1,1])*random.uniform(0.2,0.7)
    elif mode=="side_exposed":   z=ground_z+hh*random.uniform(0.15,0.35); roll=random.choice([-1,1])*random.uniform(1.15,1.45); pitch=random.uniform(-0.25,0.25)
    else:                        z=ground_z+hh
    return z, p.getQuaternionFromEuler([roll,pitch,yaw]), yaw

def tint_all_links(body_id, rgba):
    p.changeVisualShape(body_id,-1,rgbaColor=rgba)
    for j in range(p.getNumJoints(body_id)):
        p.changeVisualShape(body_id,j,rgbaColor=rgba)

def crop_texture_patch(tex_rgb, cx, cy, crop_size_px, out_size):
    th,tw = tex_rgb.shape[:2]
    half  = crop_size_px//2
    x0,y0 = int(round(cx-half)),int(round(cy-half))
    x1,y1 = x0+crop_size_px, y0+crop_size_px
    pl=max(0,-x0);pt=max(0,-y0);pr=max(0,x1-tw);pb=max(0,y1-th)
    patch = tex_rgb[max(0,y0):min(th,y1), max(0,x0):min(tw,x1)].copy()
    if pl or pt or pr or pb:
        patch = cv2.copyMakeBorder(patch,pt,pb,pl,pr,borderType=cv2.BORDER_REFLECT)
    return cv2.resize(patch,(out_size,out_size),interpolation=cv2.INTER_LINEAR)

def sample_capture_positions(aabb_min, aabb_max, terrain_id, tex_rgb, frame_count, min_spacing=8.0):
    xmin,ymin = aabb_min[0]+5.0, aabb_min[1]+5.0
    xmax,ymax = aabb_max[0]-5.0, aabb_max[1]-5.0
    positions=[]; attempts=0; max_att=frame_count*300
    while len(positions)<frame_count and attempts<max_att:
        attempts+=1
        x=random.uniform(xmin,xmax); y=random.uniform(ymin,ymax)
        if not texture_ok_for_spawn(x,y,tex_rgb,aabb_min,aabb_max): continue
        z=terrain_hit_z(x,y,terrain_id)
        if z is None: continue
        if any((x-px)**2+(y-py)**2<min_spacing**2 for px,py,_ in positions): continue
        positions.append((x,y,z))
    if len(positions)<frame_count:
        print(f"Warning: only {len(positions)} spaced positions, relaxing…")
        while len(positions)<frame_count and attempts<max_att*2:
            attempts+=1
            x=random.uniform(xmin,xmax); y=random.uniform(ymin,ymax)
            if not texture_ok_for_spawn(x,y,tex_rgb,aabb_min,aabb_max): continue
            z=terrain_hit_z(x,y,terrain_id)
            if z is None: continue
            positions.append((x,y,z))
    return positions[:frame_count]

# ── PYBULLET SETUP ────────────────────────────────────────────────────────────

try:
    if p.isConnected(): p.disconnect()
except: pass

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetSimulation()
p.setGravity(0,0,-9.81)
p.configureDebugVisualizer(p.COV_ENABLE_WIREFRAME,0)

# ── LOAD HEIGHTMAP ────────────────────────────────────────────────────────────

hmap = imageio.imread(HEIGHTMAP)
if hmap.ndim==3: hmap=hmap[...,0]
hmap=hmap.astype(np.float32); H,W=hmap.shape
MAX_HW=512
if max(H,W)>MAX_HW:
    from PIL import Image
    hi=Image.fromarray(hmap.astype(np.uint16) if hmap.max()>255 else hmap.astype(np.uint8))
    hi=hi.resize((MAX_HW,MAX_HW),resample=Image.BILINEAR)
    hmap=np.array(hi).astype(np.float32); H,W=hmap.shape
hmin,hmax=float(hmap.min()),float(hmap.max())
if hmax-hmin<1e-6: raise ValueError("Heightmap is flat.")
hmap01=(hmap-hmin)/(hmax-hmin)
heightfieldData=hmap01.flatten().tolist()

# ── LOAD TEXTURE ──────────────────────────────────────────────────────────────

tex=imageio.imread(TEXTURE)
if tex.ndim==2: tex=np.stack([tex,tex,tex],axis=-1)
tex_rgb=tex[:,:,:3]; tex_h,tex_w=tex_rgb.shape[:2]
TARGET_M_PER_PX=0.10
terrain_size_m=max(80.0,min(float(tex_w)*TARGET_M_PER_PX,300.0))
sx=terrain_size_m/W; sy=terrain_size_m/H

# ── CREATE TERRAIN ────────────────────────────────────────────────────────────

terrain_shape=p.createCollisionShape(
    shapeType=p.GEOM_HEIGHTFIELD, meshScale=[sx,sy,12.0],
    heightfieldTextureScaling=1.0, heightfieldData=heightfieldData,
    numHeightfieldRows=H, numHeightfieldColumns=W)
terrain_id=p.createMultiBody(baseMass=0,baseCollisionShapeIndex=terrain_shape)
tex_id=p.loadTexture(TEXTURE)
p.changeVisualShape(terrain_id,-1,textureUniqueId=tex_id)
p.changeDynamics(terrain_id,-1,lateralFriction=0.9,restitution=0.0)
aabb_min,aabb_max=p.getAABB(terrain_id)
center=[(aabb_min[i]+aabb_max[i])/2 for i in range(3)]
print("Terrain AABB:",aabb_min,aabb_max)

# ── SPAWN WASTE ───────────────────────────────────────────────────────────────

waste_items=[]; existing_xy=[]
xmin,ymin=aabb_min[0]+2.5,aabb_min[1]+2.5
xmax,ymax=aabb_max[0]-2.5,aabb_max[1]-2.5
cluster_seed=None

for _ in range(WASTE_COUNT):
    spawned=False
    for _try in range(MAX_SPAWN_TRIES):
        kind=random.choice(WASTE_URDFS)
        if cluster_seed and random.random()<0.30:
            x=cluster_seed[0]+random.uniform(-6.0,6.0)
            y=cluster_seed[1]+random.uniform(-6.0,6.0)
        else:
            x=random.uniform(xmin,xmax); y=random.uniform(ymin,ymax)
        if not (xmin<=x<=xmax and ymin<=y<=ymax): continue
        min_gap=max(MIN_WASTE_GAP,kind["footprint"]*1.6)
        if not far_from_existing(x,y,existing_xy,min_gap): continue
        if not texture_ok_for_spawn(x,y,tex_rgb,aabb_min,aabb_max): continue
        ground_z=terrain_hit_z(x,y,terrain_id)
        if ground_z is None: continue
        mode=random.choice(kind["burial_bias"])
        scale=random.uniform(kind["scale_range"][0],kind["scale_range"][1])
        z,quat,yaw=placement_pose(kind,mode,ground_z)
        try:
            body_id=p.loadURDF(kind["path"],basePosition=[x,y,z],
                               baseOrientation=quat,useFixedBase=True,globalScaling=scale)
            tint_all_links(body_id,[1,1,1,0.0])
        except Exception as e:
            print("URDF load failed:",kind["path"],e); continue
        tx,ty=world_to_tex_uv(x,y,aabb_min,aabb_max,tex_w,tex_h)

        # Assign bottle pose — randomised for variety in training data
        bottle_pose = "on_side"
        if kind["name"] == "bottle":
            # Weight distribution: mostly on_side, some upright, some partially buried forms
            bottle_pose = random.choices(
                ["on_side", "upright", "neck_up", "base_up"],
                weights=[0.40,    0.20,      0.20,     0.20]
            )[0]
            # Snap burial mode to match pose visually
            if bottle_pose == "upright":
                mode = random.choice(["surface", "surface", "half_buried"])
            elif bottle_pose in ("neck_up", "base_up"):
                mode = random.choice(["half_buried", "mostly_buried"])

        waste_items.append({"name":kind["name"],"world_x":x,"world_y":y,
                            "tex_x":tx,"tex_y":ty,"yaw":yaw,
                            "burial_mode":mode,"obj_scale":scale,
                            "bottle_pose":bottle_pose})
        existing_xy.append((x,y))
        if cluster_seed is None or random.random()<0.18: cluster_seed=(x,y)
        spawned=True; break
    if not spawned: print("Could not place one waste item.")

print(f"Waste items placed: {len(waste_items)}")

# ── CAPTURE FRAMES ────────────────────────────────────────────────────────────

capture_dir=os.path.join(SCRIPT_DIR,"captured_frames")
labels_dir =os.path.join(capture_dir,"labels")
os.makedirs(capture_dir,exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

positions=sample_capture_positions(aabb_min,aabb_max,terrain_id,tex_rgb,
                                   FRAME_COUNT,MIN_CAPTURE_SPACING_M)
print(f"Capture positions: {len(positions)}")

for i,(x,y,z) in enumerate(positions):
    cx,cy=world_to_tex_uv(x,y,aabb_min,aabb_max,tex_w,tex_h)
    patch_rgb=crop_texture_patch(tex_rgb,cx,cy,CROP_SIZE_PX,OUTPUT_SIZE)
    patch_bgr=cv2.cvtColor(patch_rgb,cv2.COLOR_RGB2BGR)

    # ── render overlays ──────────────────────────────────────────────────────
    patch_bgr, bboxes = render_waste_on_patch(
        patch_bgr, waste_items,
        center_px=(cx,cy),
        crop_size_px=CROP_SIZE_PX,
        out_size=OUTPUT_SIZE,
    )

    out_path=os.path.join(capture_dir,f"frame_{i:03d}.png")
    cv2.imwrite(out_path,patch_bgr)

    label_path=os.path.join(labels_dir,f"frame_{i:03d}.txt")
    with open(label_path,"w") as f:
        for bb in bboxes:
            cls=CLASS_IDS.get(bb["name"],0)
            bx=(bb["x1"]+bb["x2"])/2.0/OUTPUT_SIZE
            by=(bb["y1"]+bb["y2"])/2.0/OUTPUT_SIZE
            bw=(bb["x2"]-bb["x1"])/OUTPUT_SIZE
            bh=(bb["y2"]-bb["y1"])/OUTPUT_SIZE
            f.write(f"{cls} {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f}\n")

    print(f"frame_{i:03d}.png  {len(bboxes)} objects")

print(f"\nDone — {len(positions)} frames in '{capture_dir}/'")
print(f"YOLO labels in '{labels_dir}/'")

# ── GUI ───────────────────────────────────────────────────────────────────────

cube_col=p.createCollisionShape(p.GEOM_BOX,halfExtents=[0.5,0.5,0.5])
cube_vis=p.createVisualShape(p.GEOM_BOX,halfExtents=[0.5,0.5,0.5],rgbaColor=[1,0,0,1])
p.createMultiBody(baseMass=1,baseCollisionShapeIndex=cube_col,baseVisualShapeIndex=cube_vis,
                  basePosition=[center[0],center[1],center[2]+20])
p.resetDebugVisualizerCamera(cameraDistance=max(15.0,terrain_size_m*0.20),
                             cameraYaw=0,cameraPitch=-89,
                             cameraTargetPosition=[center[0],center[1],0])
while p.isConnected():
    p.stepSimulation(); time.sleep(1/240)

Terrain AABB: (-45.91015625, -45.91015625, -6.0) (45.91015625, 45.91015625, 6.0)
Waste items placed: 55
Capture positions: 400
frame_000.png  2 objects
frame_001.png  2 objects
frame_002.png  8 objects
frame_003.png  4 objects
frame_004.png  10 objects
frame_005.png  7 objects
frame_006.png  9 objects
frame_007.png  2 objects
frame_008.png  8 objects
frame_009.png  8 objects
frame_010.png  8 objects
frame_011.png  14 objects
frame_012.png  3 objects
frame_013.png  8 objects
frame_014.png  5 objects
frame_015.png  5 objects
frame_016.png  12 objects
frame_017.png  5 objects
frame_018.png  7 objects
frame_019.png  7 objects
frame_020.png  6 objects
frame_021.png  8 objects
frame_022.png  4 objects
frame_023.png  5 objects
frame_024.png  9 objects
frame_025.png  8 objects
frame_026.png  6 objects
frame_027.png  3 objects
frame_028.png  10 objects
frame_029.png  10 objects
frame_030.png  6 objects
frame_031.png  7 objects
frame_032.png  11 objects
frame_033.png  10 objects
frame_034.png  1